<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cpp.en/cap03/cap03_aluno.ipynb"><img src="images/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="images/github-badge.png" style="height:20px;vertical-align:middle"></a>

# 3 Spatial Operations: Intensity, Histogram, and Filtering

This chapter delves into image processing in the spatial domain, starting from the direct manipulation of pixels and histograms for contrast enhancement, to the application of local filters via convolution for smoothing, noise reduction, and edge detection. The goal is to develop the mathematical and computational intuition that underpins much of modern Computer Vision algorithms.

## 3.1 Objectives

By the end of this chapter, you will be able to:

* **Manipulate intensity and pixels:** Perform saturated arithmetic operations (`mm::addm`, `mm::subm`) and bitwise logic operations (`mm::band`, `mm::bor`, `mm::bnot`) for combining and selecting regions of interest (ROI), and apply *alpha blending* (`mm::blend`) for weighted image fusion;
* **Process histograms:** Interpret the histogram as a tonal diagnostic and apply global equalization via CDF (`mm::equalize`); in the Python track, also adaptive equalization (CLAHE) and histogram specification for tonal profile transfer between images;
* **Understand spatial fundamentals:** Understand neighborhood, border *padding* (`mm::pad`), and the difference between cross-correlation (`mm::conv`) and convolution — including why asymmetric kernels such as Sobel produce distinct results in the two operations;
* **Apply smoothing filtering:** Use the mean filter (`mm::blur`, or `mm::conv` with a uniform kernel) and the Gaussian filter (`mm::gaussian`) for noise reduction, understanding the advantage of radial weighting and Gaussian separability;
* **Apply enhancement filtering:** Use the Laplacian $w_4$ and $w_8$ (`mm::laplacian`) for isotropic edge enhancement, the Sobel operator (`mm::sobel`) for gradient magnitude — and, in the Python track, the directional decomposition $G_x$, $G_y$, and angle —, and *Unsharp Masking* (`mm::usm`) for high-frequency amplification controlled by the parameter $k$;
* **Use order filters:** Apply the median filter (`mm::median`) for salt-and-pepper noise removal, understanding why its nonlinear nature and robustness to *outliers* make it superior to linear filters in this scenario;
* **Solve practical problems:** Chain techniques into preprocessing *pipelines* (equalization → Gaussian → Canny; with CLAHE replacing equalization in the Python track) and use the `morph` functions (`mm::conv`, `mm::histImg`, `mm::equalize`, `mm::drawImgKernel`) for didactic analysis and visualization of each step.

## 3.2 Intensity Level Operations

The most elementary level of image processing acts directly on pixel values, without considering neighborhood. These operations—called **point operations**—are the fastest computationally and form the basis for more complex techniques.

Formally, a point transformation can be described as:

<a id="eq-03-ponto"></a>
$$
g(x,y) = T[f(x,y)] \tag{3.1}
$$


where $f(x,y)$ is the input image, $g(x,y)$ is the output, and $T$ is a function applied to each pixel individually.

### 3.2.1 Preparing the Practical Environment

The following block loads the `morph` library from the repository (the `morph.py` module and, in the C++ track, also the `morph.hpp` used in the `#include` of compiled cells).

In [1]:
import os, urllib.request

os.makedirs("tmp/state", exist_ok=True)  # C++ track build artifacts (.cpp, binary, PNGs)

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

# The kernel is Python even on the C++ track: `mm` (morph.py) is used by the
# simulators, by the display of the figures the C++ binary generates, and by
# the mm::Image state between cells. cpp=True also downloads the compiled track
# (morph.hpp + stb_image*.h), used in the #include of the %%writefile *.cpp cells.
import config
config.setup(cpp=True)
from morph import mm
import numpy as np

✅ Environment ready. Morph: 1.1.9 | OpenCV: 5.0.0


As the object of study throughout this chapter, we will use the wildlife images presented in [Figure 3.1](#fig-03-mandrill) and [Figure 3.3](#fig-03-leopardo). From them, we will explore spatial operations on intensity, histograms, and filtering, analyzing their effects on enhancement, smoothing, noise reduction, and edge detection, in order to understand the mathematical and computational fundamentals of DIP.

In [2]:
%%writefile tmp/fig_03_mandrill.cpp
#define MM_OUT "tmp/fig_03_mandrill.png"
//| label: fig-03-mandrill
//| fig-cap: "*Mandrill* (*Mandrillus sphinx*) photographed in its natural environment in South Africa. Credit: Carlos Guilherme Rodrigues (CC BY-SA 3.0)."
//| echo: true

#include "morph.hpp"
#include <iostream>
#include <filesystem>

int main() {
    mm::Image img_color = mm::read("https://upload.wikimedia.org/wikipedia/commons/9/9b/Carlos_Guilherme_Rodrigues_%2876515283%29.jpeg");
    mm::Image img_gray = mm::gray(img_color);

    mm::show(img_color, MM_OUT);

    
// [pdi:state-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp/state");
mm::write(img_gray, "tmp/state/img_gray_8.png");
// [pdi:state-io:end]
return 0;
}

Overwriting tmp/fig_03_mandrill.cpp


In [3]:
!g++ -I. -std=c++17 tmp/fig_03_mandrill.cpp -o tmp/fig_03_mandrill \
  && ./tmp/fig_03_mandrill \
  && test -f "tmp/fig_03_mandrill.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_mandrill.png"

In [4]:
try:
    mm.show(mm.read("tmp/fig_03_mandrill.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_mandrill.png (ver a versao Python)")

<Figure size 480x450 with 1 Axes>

**Figure 3.1:** *Mandrill* (*Mandrillus sphinx*) fotografado em ambiente natural na África do Sul. Crédito: Carlos Guilherme Rodrigues (CC BY-SA 3.0).


### 3.2.2 Arithmetic Operations

Arithmetic operations between images are widely used in DIP to combine, compare, or enhance information. **Image subtraction** is especially powerful for detecting differences between two frames — for example, in removing static backgrounds in surveillance cameras:

<a id="eq-03-subtracao"></a>
$$
g(x,y) = f_1(x,y) - f_2(x,y) \tag{3.2}
$$


**Saturated addition** limits the result to the interval $[0, 255]$: values above 255 are clamped to 255, avoiding the silent *overflow* of the `uint8` type (e.g., $200 + 100 = 44$ instead of 300). **Saturated subtraction** applies the same principle on the lower side: negative values are clamped to 0.

> ### ⚠️ Saturation and *overflow*
>
> Arithmetic operations on `uint8` suffer from silent *overflow*: $200 + 100 = 44$ (not 300). `mm::addm` and `mm::subm` perform **automatic saturation**, clamping the result to $[0, 255]$. *Blending* uses fractional weights: `mm::blend` operates internally in floating point and only then rounds and saturates to `uint8`.

[Figure 3.2](#fig-03-aritmetica) demonstrates the addition of a constant (brightening) and the subtraction of a constant (darkening with saturation at 0).

In [5]:
%%writefile tmp/fig_03_aritmetica.cpp
#define MM_OUT "tmp/fig_03_aritmetica.png"
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_8.png");
// [pdi:state-io:end]

    //
    // Operations: log transformation, gamma correction, and addition.
    //

    int fundo = 60;

    mm::Image img_add = mm::addm(img_gray, fundo);
    mm::Image img_sub = mm::subm(img_gray, fundo);

    mm::show(
        std::vector<mm::Image>{img_gray, img_add, img_sub},
        MM_OUT,
        std::vector<std::string>{"Original", "addm (+60)", "subm (−60)"},
        3
    );

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray, "tmp/fig_03_aritmetica_0.png");
mm::write(img_add, "tmp/fig_03_aritmetica_1.png");
mm::write(img_sub, "tmp/fig_03_aritmetica_2.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_03_aritmetica.cpp


In [6]:
!g++ -I. -std=c++17 tmp/fig_03_aritmetica.cpp -o tmp/fig_03_aritmetica \
  && ./tmp/fig_03_aritmetica \
  && test -f "tmp/fig_03_aritmetica.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_aritmetica.png"

[1] Original
[2] addm (+60)
[3] subm (−60)


In [7]:
try:
    mm.show(
        [
            mm.read("tmp/fig_03_aritmetica_0.png"),
            mm.read("tmp/fig_03_aritmetica_1.png"),
            mm.read("tmp/fig_03_aritmetica_2.png"),
        ],
        titles=[
            'Original',
            'addm (+60)',
            'subm (−60)',
        ],
        cols=3,
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_aritmetica_0.png (ver a versao Python)")

<Figure size 2250x750 with 3 Axes>

**Figure 3.2:** Operações aritméticas saturadas: adição de constante (clareamento) e subtração de constante (escurecimento com saturação em 0).


### 3.2.3 Weighted Mixture (*Alpha Blending*)

The **weighted mixture** (*alpha blending*) combines two images using complementary weights $\alpha$ and $(1-\alpha)$:

<a id="eq-03-blend"></a>
$$
g(x,y) = \alpha\,f_1(x,y) + (1-\alpha)\,f_2(x,y), \quad \alpha \in [0,1] \tag{3.3}
$$


When $\alpha = 1$, only image $f_1$ is obtained; when $\alpha = 0$, only $f_2$. Intermediate values produce a smooth transition between the two, being widely used in image composition, layer overlaying, watermarks, and visual blending effects.

For the combination to produce a coherent result, it is necessary to align the regions of interest beforehand. In [Figure 3.4](#fig-03-blend), the leopard's face is cropped with `mm::crop(img_leop_gray, 250, H-300, 100, W-200)` and the mandrill's facial region with `mm::crop(img_gray, 100, 400, 380, 530)`, so that the eyes and facial structure are approximately aligned. The leopard crop is then resized (`mm::resize`) to the mandrill's dimensions before the blending.

`mm::blend` performs the operation in floating point — avoiding *overflow* in the calculations with fractional weights — and only then rounds and saturates the result to `uint8`.

In [8]:
%%writefile tmp/fig_03_leopardo.cpp
#define MM_OUT "tmp/fig_03_leopardo.png"
//| label: fig-03-leopardo
//| fig-cap: "Retrato de um leopardo (*Panthera pardus*) em ambiente natural. Crédito: C. Brück (CC BY-SA 4.0)."
//| echo: true

#include "morph.hpp"
#include <filesystem>

int main() {
    mm::Image img_leop = mm::read("https://upload.wikimedia.org/wikipedia/commons/9/92/Leopard_%28Panthera_pardus%29_portrait.jpg");
    mm::Image img_leop_gray = mm::gray(img_leop);

    mm::show(img_leop, MM_OUT);

    
// [pdi:state-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp/state");
mm::write(img_leop, "tmp/state/img_leop_12.png");
mm::write(img_leop_gray, "tmp/state/img_leop_gray_12.png");
// [pdi:state-io:end]
return 0;
}

Overwriting tmp/fig_03_leopardo.cpp


In [9]:
!g++ -I. -std=c++17 tmp/fig_03_leopardo.cpp -o tmp/fig_03_leopardo \
  && ./tmp/fig_03_leopardo \
  && test -f "tmp/fig_03_leopardo.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_leopardo.png"

In [10]:
try:
    mm.show(mm.read("tmp/fig_03_leopardo.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_leopardo.png (ver a versao Python)")

<Figure size 621x662 with 1 Axes>

**Figure 3.3:** Retrato de um leopardo (*Panthera pardus*) em ambiente natural. Crédito: C. Brück (CC BY-SA 4.0).


In [11]:
%%writefile tmp/fig_03_blend.cpp
#define MM_OUT "tmp/fig_03_blend.png"
//| label: fig-03-blend
//| fig-cap: "*Alpha blending* entre recortes alinhados de mandrill e do leopardo (@fig-03-leopardo) para diferentes valores de α. Em α=1 vê-se apenas mandrill; em α=0, apenas o leopardo; valores intermediários fundem os olhares das duas imagens proporcionalmente."
//| echo: true
//| output: true

#include "morph.hpp"
#include <vector>
#include <string>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_8.png");
mm::Image img_leop_gray = mm::_read_state("tmp/state/img_leop_gray_12.png");
// [pdi:state-io:end]

    // Recortes alinhados: rosto do leopardo e região facial do mandril
    mm::Image leo = mm::crop(img_leop_gray, 250, img_leop_gray.h - 300, 100, img_leop_gray.w - 200);
    mm::Image mandrill = mm::crop(img_gray, 100, 400, 380, 530);
    mm::Image leo_r = mm::resize(leo, mandrill.w, mandrill.h, "bilinear");

    mm::show(
        std::vector<mm::Image>{mm::blend(mandrill, leo_r, 1.0), mm::blend(mandrill, leo_r, 0.8),
                               mm::blend(mandrill, leo_r, 0.6), mm::blend(mandrill, leo_r, 0.4),
                               mm::blend(mandrill, leo_r, 0.2), mm::blend(mandrill, leo_r, 0.0)},
        MM_OUT,
        std::vector<std::string>{"α=1.0", "α=0.8", "α=0.6", "α=0.4", "α=0.2", "α=0.0"},
        6
    );

    return 0;
}

Overwriting tmp/fig_03_blend.cpp


In [12]:
!g++ -I. -std=c++17 tmp/fig_03_blend.cpp -o tmp/fig_03_blend \
  && ./tmp/fig_03_blend \
  && test -f "tmp/fig_03_blend.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_blend.png"

[1] α=1.0
[2] α=0.8
[3] α=0.6
[4] α=0.4
[5] α=0.2
[6] α=0.0


In [13]:
try:
    mm.show(mm.read("tmp/fig_03_blend.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_blend.png (ver a versao Python)")

<Figure size 1564x450 with 1 Axes>

**Figure 3.4:** *Alpha blending* entre recortes alinhados de mandrill e do leopardo (@fig-03-leopardo) para diferentes valores de α. Em α=1 vê-se apenas mandrill; em α=0, apenas o leopardo; valores intermediários fundem os olhares das duas imagens proporcionalmente.


### 3.2.4 Logical Operations and Bitwise Masks

Bitwise logical operations (AND, OR, and NOT) act directly on the bits of each pixel and are the foundation for creating and applying **masks** (*masks*) — binary images with only 0 (black) and 255 (white) used to isolate **Regions of Interest** (**ROI**).

The behavior of each operation stems from the binary representation of 255 (`11111111`) and 0 (`00000000`):

- **AND** with the mask: where $m = 255$, the original bits are preserved; where $m = 0$, the pixel is zeroed. Result: ROI cropping.
<a id="eq-03-mascara"></a>
$$
g(x,y) = f(x,y) \;\text{AND}\; m(x,y) \tag{3.4}
$$

- **OR** with the mask: where $m = 255$, the pixel is forced to white; where $m = 0$, the original value is retained. Result: ROI highlighting.
- **NOT** (without mask): inverts all bits ($g = 255 - f$), producing the photographic negative of the image.

[Figure 3.5](#fig-03-logica) illustrates the three operations applied to the mandrill image with a circular mask.

In [14]:
%%writefile tmp/fig_03_logica.cpp
#define MM_OUT "tmp/fig_03_logica.png"
#include "morph.hpp"
#include <iostream>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_8.png");
// [pdi:state-io:end]

    //| label: fig-03-logica
    //| fig-cap: "Binary bitwise operations with circular mask: AND (ROI isolation), OR (ROI illumination) and NOT (negative)."
    //| echo: true
    //| output: true

    int h = img_gray.h;
    int w = img_gray.w;

    // Filled circular mask, centered on the image (mm.circle draws the
    // disk; the didactic version, pixel-by-pixel radius test, is mm.circle0).
    mm::Image mask_circ(h, w);
    mask_circ = mm::circle(mask_circ, w / 2, h / 2, std::min(h, w) / 3 - 10, 255, -1);

    // Operations via morph
    mm::Image img_not = mm::bnot(img_gray);            // NOT: photographic negative
    mm::Image img_and = mm::band(img_gray, mask_circ); // preserves only the circular ROI
    mm::Image img_or  = mm::bor(img_gray, mask_circ);  // illuminates the mask region

    mm::show(
        std::vector<mm::Image>{img_gray, img_and, img_or, img_not},
        MM_OUT,
        std::vector<std::string>{"Original", "AND (circular ROI)", "OR (illuminates ROI)", "NOT (negative)"},
        4
    );

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray, "tmp/fig_03_logica_0.png");
mm::write(img_and, "tmp/fig_03_logica_1.png");
mm::write(img_or, "tmp/fig_03_logica_2.png");
mm::write(img_not, "tmp/fig_03_logica_3.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_03_logica.cpp


In [15]:
!g++ -I. -std=c++17 tmp/fig_03_logica.cpp -o tmp/fig_03_logica \
  && ./tmp/fig_03_logica \
  && test -f "tmp/fig_03_logica.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_logica.png"

[1] Original
[2] AND (circular ROI)
[3] OR (illuminates ROI)
[4] NOT (negative)


In [16]:
try:
    mm.show(
        [
            mm.read("tmp/fig_03_logica_0.png"),
            mm.read("tmp/fig_03_logica_1.png"),
            mm.read("tmp/fig_03_logica_2.png"),
            mm.read("tmp/fig_03_logica_3.png"),
        ],
        titles=[
            'Original',
            'AND (ROI circular)',
            'OR (ilumina ROI)',
            'NOT (negativo)',
        ],
        cols=4,
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_logica_0.png (ver a versao Python)")

<Figure size 3000x750 with 4 Axes>

**Figure 3.5:** Operações lógicas bit a bit com máscara circular: AND (isolamento da ROI), OR (iluminação da ROI) e NOT (negativo).


## 3.3 Image Histogram

The **histogram** of a grayscale image is a discrete function that describes the frequency distribution of intensities:

<a id="eq-03-histograma"></a>
$$
h(r_k) = n_k, \quad k = 0, 1, \ldots, L-1 \tag{3.5}
$$


where $r_k$ is the $k$-th intensity level, $n_k$ is the number of pixels with that intensity, and $L$ is the total number of levels (typically 256 for 8 bits). The normalized histogram estimates the probability of each level:

<a id="eq-03-hist-norm"></a>
$$
p(r_k) = \frac{n_k}{MN} \tag{3.6}
$$


where $MN$ is the total number of pixels. As a **global statistic**, the histogram does not carry positional information, but it reveals essential characteristics such as average brightness, contrast, and tonal distribution. In practice, `mm::hist(img)` returns the count vector $h(r_k)$, which serves both for visualization (via `mm::histImg`) and for calculations such as the **cumulative distribution function (CDF)** and equalization.

> ### 📝 Histogram Interpretation
>
> - **Narrow on the left:** underexposed image (dark).
> - **Narrow on the right:** overexposed image (bright).
> - **Concentrated in the center:** low contrast.
> - **Spread across the entire range:** high contrast, good utilization of the available tones.

[Figure 3.6](#fig-03-histograma) presents the histogram of the mandrill image, as well as darkened (`mm::subm`) and brightened (`mm::addm`) versions. The shift of the intensity distribution to the left and to the right is observed, respectively. Note that the range represented on the $x$-axis does not necessarily correspond to the entire 0 to 255 range.

In [17]:
%%writefile tmp/fig_03_histograma.cpp
#define MM_OUT "tmp/fig_03_histograma.png"
//| label: fig-03-histograma
//| fig-cap: "Histogramas da imagem original, de uma versão escurecida (−80) e de uma clareada (+80). A subtração/adição satura em 0 e 255."
//| echo: true
//| output: true

#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_8.png");
// [pdi:state-io:end]

    mm::Image img_dark = mm::subm(img_gray, 80);
    mm::Image img_high = mm::addm(img_gray, 80);

    mm::show(
        std::vector<mm::Image>{img_gray, img_dark, img_high,
                               mm::histImg(img_gray), mm::histImg(img_dark), mm::histImg(img_high)},
        MM_OUT,
        std::vector<std::string>{"Original", "Escurecida (-80)", "Clareada (+80)",
                                 "Histograma - original", "Histograma - escurecida", "Histograma - clareada"},
        3
    );

    return 0;
}

Overwriting tmp/fig_03_histograma.cpp


In [18]:
!g++ -I. -std=c++17 tmp/fig_03_histograma.cpp -o tmp/fig_03_histograma \
  && ./tmp/fig_03_histograma \
  && test -f "tmp/fig_03_histograma.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_histograma.png"

[1] Original
[2] Escurecida (-80)
[3] Clareada (+80)
[4] Histograma - original
[5] Histograma - escurecida
[6] Histograma - clareada


In [19]:
try:
    mm.show(mm.read("tmp/fig_03_histograma.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_histograma.png (ver a versao Python)")

<Figure size 784x524 with 1 Axes>

**Figure 3.6:** Histogramas da imagem original, de uma versão escurecida (−80) e de uma clareada (+80). A subtração/adição satura em 0 e 255.


### 3.3.1 Histogram Equalization

**Histogram equalization** redistributes intensities so that the resulting histogram is as uniform as possible. The mapping is given by the **cumulative distribution function (CDF)**:

<a id="eq-03-equalizacao"></a>
$$
s_k = T(r_k) = (L-1)\sum_{j=0}^{k} p(r_j) = \frac{L-1}{MN}\sum_{j=0}^{k} n_j \tag{3.7}
$$


The transformation is monotonic: frequent levels receive larger intervals in the output domain (greater separation → more contrast), while rare levels are compressed.

The complete algorithm, in five steps, is presented in [Table 3.1](#tbl-03-equalizacao).

<a id="tbl-03-equalizacao"></a>

**Tabela 3.1:** Histogram equalization algorithm.

| Step | Operation | Formula |
|:----:|:----------|:--------|
| 1 | **Histogram** | $h[k] \leftarrow$ number of pixels with intensity $k$, $k=0\ldots L-1$ |
| 2 | **Probability** | $p[k] \leftarrow h[k] / MN$ |
| 3 | **CDF** | $\text{cdf}[k] \leftarrow \sum_{j=0}^{k} p[j]$ (cumulative sum) |
| 4 | ***Look-Up Table* (mapping)** | $\text{lut}[k] \leftarrow \text{round}(\text{cdf}[k] \times (L-1))$ |
| 5 | **Application** | $g[i,j] \leftarrow \text{lut}[f[i,j]]$ (for every pixel) |


Note in [Figure 3.7](#fig-03-equalizacao-didatica) that equalization **redistributes** the existing tones to more spaced positions in the range $[0, L-1]$, but it does not create new tones — the equalized image still has exactly 3 distinct tones, now at $\{1, 5, 7\}$ instead of $\{2, 3, 4\}$.

In [20]:
%%writefile tmp/fig_03_equalizacao_didatica.cpp
#define MM_OUT "tmp/fig_03_equalizacao_didatica.png"
#include "morph.hpp"
#include <iostream>
#include <vector>

//| label: fig-03-equalizacao-didatica
//| fig-cap: "Equalização de histograma numa imagem 5×5 de 3 bits (L=8): tons concentrados em {2,3,4} são redistribuídos pela CDF. *mm::equalize(img, 3)* faz o mapeamento."
//| echo: true
//| output: true

int main() {
    mm::Image img5(5, 5);
    int vals[5][5] = {{3, 4, 2, 3, 4},
                      {4, 3, 3, 4, 3},
                      {2, 3, 4, 3, 2},
                      {3, 4, 3, 2, 3},
                      {4, 3, 2, 3, 4}};
    for (int y = 0; y < 5; y++)
        for (int x = 0; x < 5; x++)
            img5.at(y, x) = vals[y][x];

    mm::Image img5_eq = mm::equalize(img5, 3);   // L = 2^3 = 8

    std::cout << "Imagem original 5x5 (3 bits):\n";
    std::cout << mm::drawImg(img5);
    std::cout << "Imagem equalizada 5x5:\n";
    std::cout << mm::drawImg(img5_eq);

    mm::show(std::vector<mm::Image>{img5, img5_eq, mm::histImg(img5), mm::histImg(img5_eq)},
             MM_OUT,
             std::vector<std::string>{"Original", "Equalizada", "Histograma - original", "Histograma - equalizada"},
             2);

    return 0;
}

Overwriting tmp/fig_03_equalizacao_didatica.cpp


In [21]:
!g++ -I. -std=c++17 tmp/fig_03_equalizacao_didatica.cpp -o tmp/fig_03_equalizacao_didatica \
  && ./tmp/fig_03_equalizacao_didatica \
  && test -f "tmp/fig_03_equalizacao_didatica.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_equalizacao_didatica.png"

Imagem original 5x5 (3 bits):
3 4 2 3 4 
4 3 3 4 3 
2 3 4 3 2 
3 4 3 2 3 
4 3 2 3 4 
Imagem equalizada 5x5:
5 7 1 5 7 
7 5 5 7 5 
1 5 7 5 1 
5 7 5 1 5 
7 5 1 5 7 
[1] Original
[2] Equalizada
[3] Histograma - original
[4] Histograma - equalizada


In [22]:
try:
    mm.show(mm.read("tmp/fig_03_equalizacao_didatica.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_equalizacao_didatica.png (ver a versao Python)")

<Figure size 524x524 with 1 Axes>

**Figure 3.7:** Equalização de histograma numa imagem 5×5 de 3 bits (L=8): tons concentrados em {2,3,4} são redistribuídos pela CDF. *mm::equalize(img, 3)* faz o mapeamento.


**Limitation:** global equalization can over-enhance noise and produce excessive contrast in homogeneous regions. **CLAHE** (*Contrast Limited Adaptive Histogram Equalization*) reduces this problem by applying equalization in local blocks (*tiles*) and limiting the height of histogram peaks before equalization.

[Figure 3.8](#fig-03-equalizacao) compares the original image, the global equalization via `mm::equalize`, and OpenCV's CLAHE, also displaying the resulting histograms. Unlike global equalization, which uses a single transformation based on the CDF of the entire image, CLAHE adapts contrast to each region, being particularly useful in images with non-uniform illumination.

In the example, `clipLimit=2.0` and `tileGridSize=(32,32)` were used. The `clipLimit` parameter defines how much the local histogram peaks can grow before being clipped. In OpenCV, this value is a relative factor: the actual limit is roughly calculated as `clipLimit × (number of pixels in the block / number of gray levels)`. For example, in a block with 4096 pixels and an 8-bit image (256 gray levels), the average frequency per level is $4096/256=16$. Thus, `clipLimit=2.0` allows peaks of approximately $2\times16=32$ occurrences before clipping. The excess occurrences are not discarded: they are redistributed among the remaining gray levels of the histogram, reducing excessive concentration in few levels and avoiding exaggerated amplification of local contrast. Smaller values limit contrast more and reduce noise amplification, while larger values allow more intense enhancement but may introduce artifacts.

- `clipLimit=1.0`: smooth and conservative enhancement;
- `clipLimit=2.0`: good balance between contrast and naturalness;
- `clipLimit=4.0`: greater emphasis on local details;
- `clipLimit=8.0`: aggressive contrast, with possible noise amplification.

Thus, CLAHE usually produces more natural results than global equalization, especially in images with shadows, reflections, or uneven illumination.

In [23]:
%%writefile tmp/fig_03_equalizacao.cpp
#define MM_OUT "tmp/fig_03_equalizacao.png"
//| label: fig-03-equalizacao
//| fig-cap: "Equalização de histograma global (mm::equalize, via CDF) e os histogramas antes/depois. CLAHE (adaptativa) fica só na trilha Python — não tem equivalente em morph.hpp."
//| echo: true
//| output: true

#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_8.png");
// [pdi:state-io:end]

    // img_gray is assumed to be already initialized

    mm::Image img_eq = mm::equalize(img_gray);

    mm::show(
        std::vector<mm::Image>{img_gray, img_eq, mm::histImg(img_gray), mm::histImg(img_eq)},
        MM_OUT,
        std::vector<std::string>{"Original", "mm.equalize (CDF)", "Histograma - original", "Histograma - equalizado"},
        2
    );

    return 0;
}

Overwriting tmp/fig_03_equalizacao.cpp


In [24]:
!g++ -I. -std=c++17 tmp/fig_03_equalizacao.cpp -o tmp/fig_03_equalizacao \
  && ./tmp/fig_03_equalizacao \
  && test -f "tmp/fig_03_equalizacao.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_equalizacao.png"

[1] Original
[2] mm.equalize (CDF)
[3] Histograma - original
[4] Histograma - equalizado


In [25]:
try:
    mm.show(mm.read("tmp/fig_03_equalizacao.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_equalizacao.png (ver a versao Python)")

<Figure size 524x524 with 1 Axes>

**Figure 3.8:** Equalização de histograma global (mm::equalize, via CDF) e os histogramas antes/depois. CLAHE (adaptativa) fica só na trilha Python — não tem equivalente em morph.hpp.


### 3.3.2 Histogram Specification

While equalization imposes a uniform distribution, **histogram specification** (*histogram matching*) allows the output image's histogram to follow an **arbitrary** distribution — for example, the histogram of another reference image.

The procedure involves three steps:

1. Compute the CDF of the input image: $P_r(r_k)$.
2. Compute the CDF of the reference image: $P_z(z_k)$.
3. For each level $r_k$, find the level $z$ that minimizes $|P_z(z) - P_r(r_k)|$.

<a id="eq-03-especificacao"></a>
$$
T(r_k) = \arg\min_{z}\,|P_z(z) - P_r(r_k)| \tag{3.8}
$$


In [Figure 3.9](#fig-03-especificacao), we transfer the tonal profile of the leopard ([Figure 3.3](#fig-03-leopardo)) to the mandrill image — a direct application of the concept seen in *blending*: instead of blending pixels, here we blend tonal distributions.

In [26]:
%%writefile tmp/fig_03_especificacao.cpp
#define MM_OUT "tmp/fig_03_especificacao.png"
//| label: fig-03-especificacao
//| fig-cap: "Especificação de histograma (mapear o mandril para o perfil tonal do leopardo) precisa da CDF inversa da referência — fica só na trilha Python. Aqui, a equalização global do mandril, como comparação."
//| echo: true
//| output: true

#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_8.png");
mm::Image img_leop_gray = mm::_read_state("tmp/state/img_leop_gray_12.png");
// [pdi:state-io:end]

    // img_gray and img_leop_gray are pre-initialized

    mm::Image img_eq = mm::equalize(img_gray);

    mm::show({img_gray, img_leop_gray, img_eq}, MM_OUT,
             {"Mandril (original)", "Leopardo (referencia)", "Mandril equalizado"}, 3);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray, "tmp/fig_03_especificacao_0.png");
mm::write(img_leop_gray, "tmp/fig_03_especificacao_1.png");
mm::write(img_eq, "tmp/fig_03_especificacao_2.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_03_especificacao.cpp


In [27]:
!g++ -I. -std=c++17 tmp/fig_03_especificacao.cpp -o tmp/fig_03_especificacao \
  && ./tmp/fig_03_especificacao \
  && test -f "tmp/fig_03_especificacao.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_especificacao.png"

[1] Mandril (original)
[2] Leopardo (referencia)
[3] Mandril equalizado


In [28]:
try:
    mm.show(
        [
            mm.read("tmp/fig_03_especificacao_0.png"),
            mm.read("tmp/fig_03_especificacao_1.png"),
            mm.read("tmp/fig_03_especificacao_2.png"),
        ],
        titles=[
            'Mandril (original)',
            'Leopardo (referencia)',
            'Mandril equalizado',
        ],
        cols=3,
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_especificacao_0.png (ver a versao Python)")

<Figure size 2250x750 with 3 Axes>

**Figure 3.9:** Especificação de histograma (mapear o mandril para o perfil tonal do leopardo) precisa da CDF inversa da referência — fica só na trilha Python. Aqui, a equalização global do mandril, como comparação.


## 3.4 Spatial Fundamentals: Neighborhood, Convolution, and *Kernels*

Spatial filtering operations do not act on a single isolated pixel, but on a **neighborhood** around it. For this purpose, a small matrix of coefficients called a **kernel** (or mask) is used, which traverses the entire image by means of a **sliding window**.

The most common windows are 3×3, 5×5, and 7×7. In a 3×3 window, for example, the central pixel is processed together with its eight immediate neighbors. At each window position, the pixel values are combined with the kernel coefficients, producing a new value for the central pixel.

### 3.4.1 Neighborhood

Consider a 3×3 window centered on the pixel $(x,y)$:

<a id="eq-03-box3x3"></a>
$$
\begin{bmatrix}
(x-1,y-1) & (x,y-1) & (x+1,y-1) \\
(x-1,y)   & (x,y)   & (x+1,y)   \\
(x-1,y+1) & (x,y+1) & (x+1,y+1)
\end{bmatrix} \tag{3.9}
$$


In general, a window of size $(2a+1)\times(2b+1)$ encompasses all pixels located up to $a$ positions horizontally and up to $b$ positions vertically relative to the central pixel. Thus, a 3×3 window corresponds to $a=b=1$, a 5×5 window to $a=b=2$, and so on.

Mathematically, the neighborhood is defined by

<a id="eq-03-vizinhanca"></a>
$$
\mathcal{V}(x,y)=
\{(x+s,\,y+t): -a\le s\le a,\,-b\le t\le b\} \tag{3.10}
$$


### 3.4.2 Edge Handling

Pixels near the edges have part of their neighborhood outside the image. To apply filters in these regions, it is necessary to define how the external values will be obtained. The three most common strategies (with the equivalent OpenCV constant in parentheses) are:

- **Zero-padding** (`BORDER_CONSTANT`): fills the external region with zeros.
- **Replication** (`BORDER_REPLICATE`): repeats the value of the border pixel.
- **Reflection** (`BORDER_REFLECT_101`): mirrors the neighboring pixels, without repeating the border pixel.

`mm::conv` uses reflection by default, as it better preserves the continuity of gray levels and reduces artifacts in edge handling.

The following example compares the three strategies with `mm::pad` on a 3×3 matrix. Note how each one fills the external pixels needed to apply a 3×3 filter also at the corners.

In [29]:
%%writefile tmp/mm_out_1.cpp
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>

int main() {
    mm::Image img(3, 3);
    img.at(0, 0) = 1; img.at(0, 1) = 2; img.at(0, 2) = 3;
    img.at(1, 0) = 4; img.at(1, 1) = 5; img.at(1, 2) = 6;
    img.at(2, 0) = 7; img.at(2, 1) = 8; img.at(2, 2) = 9;

    std::vector<std::string> bordas = {"constant", "replicate", "reflect101"};

    std::cout << "Imagem original:\n";
    std::cout << mm::drawImg(img) << "\n";

    for (int k : {3, 5}) {
        int b = k / 2;
        std::cout << "=== Kernel " << k << "x" << k << " (padding b=" << b << ") ===\n";
        for (const std::string& nome : bordas) {
            std::cout << nome << "\n";

            mm::Border border;
            if (nome == "constant") border = mm::Border::CONSTANT;
            else if (nome == "replicate") border = mm::Border::REPLICATE;
            else border = mm::Border::REFLECT101;

            std::cout << mm::drawImg(mm::pad(img, b, border)) << "\n";
        }
    }

    return 0;
}

Overwriting tmp/mm_out_1.cpp


In [30]:
!g++ -I. -std=c++17 tmp/mm_out_1.cpp -o tmp/mm_out_1 \
  && ./tmp/mm_out_1

Imagem original:
1 2 3 
4 5 6 
7 8 9 

=== Kernel 3x3 (padding b=1) ===
constant
0 0 0 0 0 
0 1 2 3 0 
0 4 5 6 0 
0 7 8 9 0 
0 0 0 0 0 

replicate
1 1 2 3 3 
1 1 2 3 3 
4 4 5 6 6 
7 7 8 9 9 
7 7 8 9 9 

reflect101
5 4 5 6 5 
2 1 2 3 2 
5 4 5 6 5 
8 7 8 9 8 
5 4 5 6 5 

=== Kernel 5x5 (padding b=2) ===
constant
0 0 0 0 0 0 0 
0 0 0 0 0 0 0 
0 0 1 2 3 0 0 
0 0 4 5 6 0 0 
0 0 7 8 9 0 0 
0 0 0 0 0 0 0 
0 0 0 0 0 0 0 

replicate
1 1 1 2 3 3 3 
1 1 1 2 3 3 3 
1 1 1 2 3 3 3 
4 4 4 5 6 6 6 
7 7 7 8 9 9 9 
7 7 7 8 9 9 9 
7 7 7 8 9 9 9 

reflect101
9 8 7 8 9 8 7 
6 5 4 5 6 5 4 
3 2 1 2 3 2 1 
6 5 4 5 6 5 4 
9 8 7 8 9 8 7 
6 5 4 5 6 5 4 
3 2 1 2 3 2 1 



Note that the result of a filter can vary significantly depending on the treatment adopted for the image borders.

In `morph.hpp`, the filtering functions (`mm::conv`, `mm::blur`, `mm::gaussian`, `mm::laplacian`, `mm::usm`) apply *padding* by **reflection** (`mm::Border::REFLECT101`) by default — the same behavior as `cv2.filter2D`. The didactic variant `mm::conv0` uses `mm::Border::KEEP`: the border pixels retain their original value, without the filter. Meanwhile, `mm::sobel` and `mm::prewitt` leave the border at zero (they compute only the interior).

### 3.4.3 Correlation vs. Convolution

There are two mathematically related mechanisms.

**Cross-correlation** — the *kernel* is applied directly:

<a id="eq-03-correlacao"></a>
$$
g(x,y) = \sum_{s=-a}^{a}\sum_{t=-b}^{b} w(s,t)\,f(x+s,\,y+t) \tag{3.11}
$$


**Two-dimensional convolution** — the *kernel* is rotated 180° before application:

<a id="eq-03-convolucao"></a>
$$
g(x,y) = \sum_{s=-a}^{a}\sum_{t=-b}^{b} w(s,t)\,f(x-s,\,y-t) \tag{3.12}
$$


For symmetric *kernels* (Gaussian, Laplacian, mean), the two operations yield identical results. For asymmetric *kernels* (Sobel, Prewitt), the difference is significant, as shown in the following examples.

#### 3.4.3.1 Correlation (`mm::conv`)

In [31]:
%%writefile tmp/mm_out_2.cpp
#include "morph.hpp"
#include <iostream>

int main() {
    // imagem 4x4 (uint8) e kernel assimétrico 3x3
    mm::Image img(4, 4);
    // fill with values 1..16
    {
        int vals[4][4] = {{ 1,  2,  3,  4},
                          { 5,  6,  7,  8},
                          { 9, 10, 11, 12},
                          {13, 14, 15, 16}};
        for (int y = 0; y < 4; ++y)
            for (int x = 0; x < 4; ++x)
                img.at(y, x) = vals[y][x];
    }

    mm::Kernel w{{0, 1, 2},
                 {0, 0, 0},
                 {0, 0, 0}};

    mm::Image corr = mm::conv(img, w, mm::Border::CONSTANT);  // zero fora da imagem

    std::cout << "Imagem original:\n";         std::cout << mm::drawImg(img) << "\n";
    std::cout << "Kernel:\n";                  std::cout << mm::drawImg(w) << "\n";
    std::cout << "Resultado da correlação:\n"; std::cout << mm::drawImg(corr) << "\n";

    return 0;
}

Overwriting tmp/mm_out_2.cpp


In [32]:
!g++ -I. -std=c++17 tmp/mm_out_2.cpp -o tmp/mm_out_2 \
  && ./tmp/mm_out_2

Imagem original:
 1  2  3  4 
 5  6  7  8 
 9 10 11 12 
13 14 15 16 

Kernel:
   0    1    2 
   0    0    0 
   0    0    0 

Resultado da correlação:
 0  0  0  0 
 5  8 11  4 
17 20 23  8 
29 32 35 12 



`mm::conv` performs correlation, that is, it applies the *kernel* exactly in the provided orientation.

#### 3.4.3.2 Convolution

In [33]:
%%writefile tmp/mm_out_3.cpp
// Compile with: g++ -std=c++17 -I. -o program program.cpp -lprotobuf -pthread
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>

int main() {
    // repeated here so the cell is self-contained
    mm::Image img(4, 4);
    // Initialize img with values 1..16
    for (int y = 0; y < img.h; y++) {
        for (int x = 0; x < img.w; x++) {
            img.at(y, x) = static_cast<unsigned char>(y * 4 + x + 1);
        }
    }

    mm::Kernel w{{0,1,2},{0,0,0},{0,0,0}};

    // kernel rotated 180° (equivalent to np.rot90(w, 2))
    mm::Kernel w_conv{{0,0,0},{0,0,0},{2,1,0}};

    mm::Image conv_result = mm::conv(img, w_conv, mm::Border::CONSTANT);

    std::cout << "Imagem original:";            std::cout << mm::drawImg(img) << "\n";
    std::cout << "Kernel original:";            std::cout << mm::drawImg(w) << "\n";
    std::cout << "Kernel rotacionado 180°:";    std::cout << mm::drawImg(w_conv) << "\n";
    std::cout << "Resultado da convolução:";    std::cout << mm::drawImg(conv_result) << "\n";

    return 0;
}

Overwriting tmp/mm_out_3.cpp


In [34]:
!g++ -I. -std=c++17 tmp/mm_out_3.cpp -o tmp/mm_out_3 \
  && ./tmp/mm_out_3

Imagem original: 1  2  3  4 
 5  6  7  8 
 9 10 11 12 
13 14 15 16 

Kernel original:   0    1    2 
   0    0    0 
   0    0    0 

Kernel rotacionado 180°:   0    0    0 
   0    0    0 
   2    1    0 

Resultado da convolução: 5 16 19 22 
 9 28 31 34 
13 40 43 46 
 0  0  0  0 



The convolution uses the *kernel* rotated by 180°. To reproduce the mathematical definition of convolution, the *kernel* is rotated (here, `[[0,1,2],[0,0,0],[0,0,0]]` → `[[0,0,0],[0,0,0],[2,1,0]]`) before applying `mm::conv`.

### 3.4.4 The Role of the *Kernel*

The *kernel* coefficients completely determine the effect produced by the filter, as summarized in [Table 3.2](#tbl-03-kernels).

<a id="tbl-03-kernels"></a>

**Tabela 3.2:** Typical interpretation of *kernel* coefficients.

| Feature | Typical effect |
|:---|:---|
| Positive coefficients summing to 1 | Smoothing (*low-pass*) |
| Sum equal to 0, with positive and negative values | Edge detection (*high-pass*) |
| Dominant central positive coefficient and negative neighbors | Sharpening |
| Asymmetric coefficients | Directional gradient |


**Examples:**

Smoothing:
$$
\frac{1}{9}
\begin{bmatrix}
1&1&1\\
1&1&1\\
1&1&1
\end{bmatrix}
$$

Edge detection:
$$
\begin{bmatrix}
-1&-1&-1\\
-1&8&-1\\
-1&-1&-1
\end{bmatrix}
$$

Sharpening:
$$
\begin{bmatrix}
0&-1&0\\
-1&5&-1\\
0&-1&0
\end{bmatrix}
$$

Directional gradient (Sobel):
$$
\begin{bmatrix}
-1&0&1\\
-2&0&2\\
-1&0&1
\end{bmatrix}
$$

[Figure 3.10](#fig-03-convolucao-passo) demonstrates the step-by-step mechanism: for each position of the window, each coefficient of the *kernel* is multiplied by the corresponding pixel in the neighborhood, and the resulting products are summed. The result is exactly the value defined by [Equation 3.11](#eq-03-correlacao) for that position in the image. Although the images produced by `mm::conv0` and `cv2.filter2D` (or `mm::conv`) are visually very similar, the OpenCV-based implementation is thousands of times faster, as shown below.

> ### ⚠️ Performance: Python loops vs. vectorized operations
>
> The `mm::conv0` function implements correlation directly in Python through nested loops. Although this approach is suitable for didactic purposes, it executes a large number of operations and becomes slow for larger images.
>
> In contrast, `mm::conv` uses `cv2.filter2D`, implemented in C++ and optimized for matrix operations. In the example presented, the vectorized version was more than **3000 times faster** than the didactic implementation, producing a visually equivalent result.
>
> The numerical differences observed are mainly concentrated at the image borders. In `mm::conv0`, border pixels remain unchanged, while `mm::conv` uses a border reflection strategy (`cv2.BORDER_REFLECT_101`, the default for `cv2.filter2D`).
>
> Therefore, `mm::conv0` should be used to understand the algorithm, while `mm::conv` is the recommended option for practical applications.

In [35]:
%%writefile tmp/fig_03_convolucao_passo.cpp
#define MM_OUT "tmp/fig_03_convolucao_passo.png"
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_leop_gray = mm::_read_state("tmp/state/img_leop_gray_12.png");
// [pdi:state-io:end]

    //#| label: fig-03-convolucao-passo
    //#| fig-cap: "Correlação com *kernel* de média 3×3: versão didática mm::conv0 (bordas preservadas) vs. mm::conv (borda refletida). A diferença se concentra nas bordas."
    //#| echo: true
    //#| output: true

    // w_mean = np.ones((3, 3), dtype=np.float32) / 9.0
    mm::Kernel w_mean = mm::Kernel::mean(3);

    // img_gray = img_leop_gray
    mm::Image img_gray = img_leop_gray;

    // img_conv0 = mm.conv0(img_gray, w_mean)   # loops, borders preserved
    mm::Image img_conv0 = mm::conv0(img_gray, w_mean);

    // img_conv = mm.conv(img_gray, w_mean)    # reflected border
    mm::Image img_conv = mm::conv(img_gray, w_mean);

    std::cout << "Correlacao no pixel central [251,251]:" << std::endl;
    std::cout << "  original = " << (int)img_gray.at(251, 251) << std::endl;
    std::cout << "  conv0    = " << (int)img_conv0.at(251, 251) << std::endl;
    std::cout << "  conv     = " << (int)img_conv.at(251, 251) << std::endl;

    mm::show(std::vector<mm::Image>{img_gray, img_conv0, img_conv},
             MM_OUT,
             std::vector<std::string>{"Original", "conv0 (laços)", "conv (vetorizado)"}, 3);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray, "tmp/fig_03_convolucao_passo_0.png");
mm::write(img_conv0, "tmp/fig_03_convolucao_passo_1.png");
mm::write(img_conv, "tmp/fig_03_convolucao_passo_2.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_03_convolucao_passo.cpp


**Figure 3.10:** Correlação com *kernel* de média 3×3: versão didática mm::conv0 (bordas preservadas) vs. mm::conv (borda refletida). A diferença se concentra nas bordas.


In [36]:
!g++ -I. -std=c++17 tmp/fig_03_convolucao_passo.cpp -o tmp/fig_03_convolucao_passo \
  && ./tmp/fig_03_convolucao_passo \
  && test -f "tmp/fig_03_convolucao_passo.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_convolucao_passo.png"

Correlacao no pixel central [251,251]:
  original = 104
  conv0    = 102
  conv     = 102
[1] Original
[2] conv0 (laços)
[3] conv (vetorizado)


In [37]:
try:
    mm.show(
        [
            mm.read("tmp/fig_03_convolucao_passo_0.png"),
            mm.read("tmp/fig_03_convolucao_passo_1.png"),
            mm.read("tmp/fig_03_convolucao_passo_2.png"),
        ],
        titles=[
            'Original',
            'conv0 (laços)',
            'conv (vetorizado)',
        ],
        cols=3,
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_convolucao_passo_0.png (ver a versao Python)")

<Figure size 2250x750 with 3 Axes>

**Figure 3.10:** Correlação com *kernel* de média 3×3: versão didática mm::conv0 (bordas preservadas) vs. mm::conv (borda refletida). A diferença se concentra nas bordas.


In [38]:
%%writefile tmp/mm_out_4.cpp
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <filesystem>

//| echo: false
// From here on the "subject" of the filtering examples becomes the
// leopard (more texture and edges than the mandrill).
int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_leop = mm::_read_state("tmp/state/img_leop_12.png");
// [pdi:state-io:end]

    mm::Image img_gray = mm::gray(img_leop);
    
// [pdi:state-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp/state");
mm::write(img_gray, "tmp/state/img_gray_45.png");
// [pdi:state-io:end]
return 0;
}

Overwriting tmp/mm_out_4.cpp


In [39]:
!g++ -I. -std=c++17 tmp/mm_out_4.cpp -o tmp/mm_out_4 \
  && ./tmp/mm_out_4

### 3.4.5 Numerical Example: Step-by-Step Correlation

To make the mechanism of [Equation 3.11](#eq-03-correlacao) concrete, consider the 3×3 averaging *kernel* ($a=b=1$, all coefficients $= 1/9 \approx 0.111$) applied to the 5×5 *patch* extracted from the leopard image. [Figure 3.11](#fig-03-patch) displays the *patch* with a grid and highlights in yellow the 3×3 window centered at pixel $[1,1]$:

In [40]:
%%writefile tmp/fig_03_patch.cpp
#define MM_OUT "tmp/fig_03_patch.png"
#include "morph.hpp"
#include <iostream>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_45.png");
// [pdi:state-io:end]

    //| label: fig-03-patch
    //| fig-cap: "*Patch* 5×5 extraído da imagem do leopardo (posição [250:255, 250:255]). A janela amarela destaca a vizinhança 3×3 centrada no pixel [1,1] onde a correlação será calculada."
    //| echo: true
    //| output: true

    mm::Image patch = mm::crop(img_gray, 250, 255, 250, 255);
    mm::Kernel B{{1,1,1},{1,1,1},{1,1,1}};

    std::cout << "Patch 5×5 (intensidades):\n";
    std::cout << mm::drawImg(patch);

    mm::drawImgKernel(patch, B, 1, 1, MM_OUT, 40);

    return 0;
}

Overwriting tmp/fig_03_patch.cpp


In [41]:
!g++ -I. -std=c++17 tmp/fig_03_patch.cpp -o tmp/fig_03_patch \
  && ./tmp/fig_03_patch \
  && test -f "tmp/fig_03_patch.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_patch.png"

Patch 5×5 (intensidades):
 94  96 103 113 115 
107 104 103 107 114 
 98 102 112 117 116 
 81  91 112 122 118 
 85  91 110 119 121 
Processando pixel (x,y)=(1,1)  |  janela do kernel 3x3
 94  96 103 113 115 
107 104 103 107 114 
 98 102 112 117 116 
 81  91 112 122 118 
 85  91 110 119 121 


In [42]:
try:
    mm.show(mm.read("tmp/fig_03_patch.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_patch.png (ver a versao Python)")

<Figure size 450x450 with 1 Axes>

**Figure 3.11:** *Patch* 5×5 extraído da imagem do leopardo (posição [250:255, 250:255]). A janela amarela destaca a vizinhança 3×3 centrada no pixel [1,1] onde a correlação será calculada.


To illustrate the calculation of correlation, consider the pixel at position $[1,1]$ in the 5×5 *patch* shown in [Figure 3.11](#fig-03-patch). This position was chosen solely for didactic convenience, as it has a complete 3×3 neighborhood around it.

The values in this neighborhood correspond to the upper-left submatrix of the *patch*:

$$
\text{neighborhood} = \begin{bmatrix}
 91 &  95 & 108 \\
106 & 107 & 108 \\
102 & 103 & 107
\end{bmatrix}
$$

Applying [Equation 3.11](#eq-03-correlacao) with the mean kernel:

$$
g[1,1] = \frac{ 91+95+108+106+107+108+102+103+107}{9} = \frac{927}{9} = 103
$$

The result (103) is slightly lower than the original value of the central pixel (107), because the mean incorporates neighbors of lower intensity, producing a smoothing effect. In practice, the algorithm starts processing at $[0,0]$ and repeats this same calculation for each position of the image, shifting the window until it covers the entire domain.

## 3.5 Smoothing Spatial Filtering

Smoothing filters attenuate abrupt intensity variations, reducing noise and high-frequency details. They are **low-pass** filters — they preserve low-frequency components (large structures) and attenuate high-frequency ones (noise, edges).

### 3.5.1 Mean Filter (*Box Filter*)

The mean filter uses a uniform kernel of size $n \times n$, where all coefficients equal $1/n^2$:

<a id="eq-03-media"></a>
$$
w_{\text{mean}} = \frac{1}{n^2}
\begin{bmatrix}
1 & \cdots & 1 \\
\vdots & \ddots & \vdots \\
1 & \cdots & 1
\end{bmatrix}_{n \times n} \tag{3.13}
$$


Each output pixel is the arithmetic mean of the $n^2$ pixels in its neighborhood. Note that the sum of the coefficients is always 1 — the average brightness of the image is preserved. Larger kernels produce more aggressive smoothing but progressively blur the edges.

[Figure 3.12](#fig-03-media) shows the effect of the mean filter with $3\times3$, $7\times7$, and $15\times15$ kernels on a detail of the leopard image. The results were obtained with `mm::blur`, which implements the mean filter using the `cv2.blur` function, equivalent to convolving the image with a uniform kernel whose coefficients are $h(x,y)=1/N^2$; equivalently, the same result can be obtained with `mm::conv`, computing ($g=f*h$). As the kernel size increases, more pixels contribute to each output value, intensifying smoothing, reducing noise, and making fine details and edges progressively more blurred.

In [43]:
%%writefile tmp/fig_03_media.cpp
#define MM_OUT "tmp/fig_03_media.png"
//| label: fig-03-media
//| fig-cap: "Filtro de média com *kernels* de tamanho crescente (3×3, 7×7, 15×15). O borramento das bordas aumenta com o tamanho do *kernel*."
//| echo: true
//| output: true

#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_45.png");
// [pdi:state-io:end]

    // Detalhe da região do olho
    int y0 = 580, y1 = 740, x0 = 680, x1 = 900;
    mm::Image img_gray_crop = mm::crop(img_gray, y0, y1, x0, x1);

    std::vector<int> sizes = {3, 7, 15};
    std::vector<mm::Image> imgs = {img_gray_crop};
    for (int k : sizes) {
        imgs.push_back(mm::blur(img_gray_crop, k));
    }
    std::vector<std::string> titles = {"Original"};
    for (int k : sizes) {
        titles.push_back("Média " + std::to_string(k) + "×" + std::to_string(k));
    }

    mm::show(imgs, MM_OUT, titles, 4);

    return 0;
}

Overwriting tmp/fig_03_media.cpp


In [44]:
!g++ -I. -std=c++17 tmp/fig_03_media.cpp -o tmp/fig_03_media \
  && ./tmp/fig_03_media \
  && test -f "tmp/fig_03_media.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_media.png"

[1] Original
[2] Média 3×3
[3] Média 7×7
[4] Média 15×15


In [45]:
try:
    mm.show(mm.read("tmp/fig_03_media.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_media.png (ver a versao Python)")

<Figure size 1044x450 with 1 Axes>

**Figure 3.12:** Filtro de média com *kernels* de tamanho crescente (3×3, 7×7, 15×15). O borramento das bordas aumenta com o tamanho do *kernel*.


### 3.5.2 Gaussian Filter

The Gaussian filter weights the pixels in the neighborhood according to a two-dimensional Gaussian function:

<a id="eq-03-gaussiana"></a>
$$
G(s,t) = \frac{1}{2\pi\sigma^2}\,e^{-\frac{s^2+t^2}{2\sigma^2}} \tag{3.14}
$$


where $\sigma$ is the standard deviation and controls the radius of influence. Pixels closer to the center have greater weight; distant pixels are progressively ignored.

[Figure 3.13](#fig-03-gauss-kernel) presents the Gaussian kernel $5\times5$ generated for $\sigma=1$. The kernel was constructed from the outer product of two one-dimensional Gaussian vectors and subsequently normalized so that the sum of its coefficients equals $1$. It is observed that the largest weights are concentrated at the center of the matrix, decreasing radially toward the edges. This distribution causes the central pixels to have greater influence on the filtering result, contributing to a more natural smoothing with better edge preservation than the mean filter.

In [46]:
%%writefile tmp/fig_03_gauss_kernel.cpp
#define MM_OUT "tmp/fig_03_gauss_kernel.png"
// Compile: g++ -std=c++17 -o program program.cpp -I. -L. -lmorph -lopencv_core -lopencv_imgproc -lopencv_imgcodecs
#include "morph.hpp"
#include <iostream>
#include <iomanip>

int main() {
    //| label: fig-03-gauss-kernel
    //| fig-cap: "*Kernel* Gaussiano 5×5 (σ=1): resposta ao impulso do filtro — pesos maiores no centro, decrescendo radialmente."
    //| echo: true
    //| output: true

    mm::Kernel w = mm::Kernel::gaussian(5, 1.0);   // mm::Kernel::gaussian(5, 1.0) na morph.hpp

    std::cout << "Kernel Gaussiano 5x5 (s=1), normalizado:\n";
    for (int y = 0; y < 5; y++) {
        std::cout << "  ";
        for (int x = 0; x < 5; x++) {
            std::cout << std::fixed << std::setprecision(4) << w.at(y, x) << "  ";
        }
        std::cout << "\n";
    }
    std::cout << "Peso central [2,2] = " << std::fixed << std::setprecision(4) << w.at(2, 2) 
              << "   |   canto [0,0] = " << std::fixed << std::setprecision(4) << w.at(0, 0) << "\n";

    // Visualização: resposta do filtro Gaussiano a um impulso central
    mm::Image impulso(5, 5);
    impulso.at(2, 2) = 255;
    mm::drawImgPlt(mm::gaussian(impulso, 5, 1.0), MM_OUT);

    return 0;
}

Overwriting tmp/fig_03_gauss_kernel.cpp


In [47]:
!g++ -I. -std=c++17 tmp/fig_03_gauss_kernel.cpp -o tmp/fig_03_gauss_kernel \
  && ./tmp/fig_03_gauss_kernel \
  && test -f "tmp/fig_03_gauss_kernel.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_gauss_kernel.png"

Kernel Gaussiano 5x5 (s=1), normalizado:
  0.0030  0.0133  0.0219  0.0133  0.0030  
  0.0133  0.0596  0.0983  0.0596  0.0133  
  0.0219  0.0983  0.1621  0.0983  0.0219  
  0.0133  0.0596  0.0983  0.0596  0.0133  
  0.0030  0.0133  0.0219  0.0133  0.0030  
Peso central [2,2] = 0.1621   |   canto [0,0] = 0.0030
 3  7 11  7  3 
 7 15 25 15  7 
11 25 41 25 11 
 7 15 25 15  7 
 3  7 11  7  3 


In [48]:
try:
    mm.show(mm.read("tmp/fig_03_gauss_kernel.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_gauss_kernel.png (ver a versao Python)")

<Figure size 450x450 with 1 Axes>

**Figure 3.13:** *Kernel* Gaussiano 5×5 (σ=1): resposta ao impulso do filtro — pesos maiores no centro, decrescendo radialmente.


> ### 📝 Computational advantage of separability
>
> Consider a square *kernel* of size $n \times n$. If this filter is separable (such as the Gaussian), the 2D convolution can be decomposed into two 1D convolutions: one horizontal and one vertical.
>
> In this case, the cost per pixel decreases from approximately $O(n^2)$ operations (direct 2D convolution) to $O(2n)$ operations (two 1D convolutions). Thus, the complexity is significantly reduced, making processing more efficient

Compared to the averaging filter, the Gaussian:

- **Better preserves edges** — the radial weighting smooths without creating abrupt transitions;
- **Does not introduce ringing** in the frequency domain, since the Gaussian is its own Fourier transform (Chapter 5);
- **Is controlled by $\sigma$** — increasing $\sigma$ is equivalent to increasing the smoothing radius in a continuous and predictable manner.

[Figure 3.14](#fig-03-gauss) compares the mean and Gaussian filters applied to the leopard image using a $9\times9$ window. The mean filter was implemented by convolution with a uniform *kernel*, where all $81$ pixels in the neighborhood have the same weight ($1/81$), while the Gaussian filter was obtained with `cv2.GaussianBlur`, using weights defined by a Gaussian distribution. Both reduce noise and smooth the image, but the Gaussian filter better preserves edges and local details, as can be observed in the enlarged region of the eye.

[Figure 3.14](#fig-03-gauss) compares the mean and Gaussian filters applied to a detail of the leopard image with $9\times 9$ *kernels*. The mean filter was obtained with `mm::blur`, equivalent to convolution with a uniform *kernel* whose coefficients equal $1/81$, while the Gaussian filter was obtained with `mm::gaussian`, equivalent to convolution with a *kernel* generated from a Gaussian distribution. Both promote smoothing and noise reduction, but the Gaussian filter assigns greater weight to the central pixels of the neighborhood, better preserving edges and local details, as can be observed in the enlarged region of the eye.

In [49]:
%%writefile tmp/fig_03_gauss.cpp
#define MM_OUT "tmp/fig_03_gauss.png"
// C++ translation
// Compile: g++ -std=c++17 -o program program.cpp -I<path-to-morph.hpp>

#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_45.png");
// [pdi:state-io:end]

    // #| label: fig-03-gauss
    // #| fig-cap: "Comparação entre filtro de média e Gaussiano (*kernel* 9×9, σ=0). O Gaussiano preserva melhor as bordas, visível no detalhe do rosto."
    // #| echo: true
    // #| output: true

    mm::Image img_media9 = mm::blur(img_gray, 9);
    mm::Image img_gauss9 = mm::gaussian(img_gray, 9, 0);

    // Detalhe da região do olho
    mm::Image img_gray_crop = mm::crop(img_gray, 580, 740, 680, 900);
    mm::Image img_media9_crop = mm::crop(img_media9, 580, 740, 680, 900);
    mm::Image img_gauss9_crop = mm::crop(img_gauss9, 580, 740, 680, 900);

    mm::show(
        std::vector<mm::Image>{img_gray_crop, img_media9_crop, img_gauss9_crop},
        MM_OUT,
        std::vector<std::string>{"Detalhe: Original", "Média 9×9", "Gaussiano 9×9"},
        3
    );

    
// [pdi:state-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp/state");
mm::write(img_gray_crop, "tmp/state/img_gray_crop_58.png");
// [pdi:state-io:end]

// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray_crop, "tmp/fig_03_gauss_0.png");
mm::write(img_media9_crop, "tmp/fig_03_gauss_1.png");
mm::write(img_gauss9_crop, "tmp/fig_03_gauss_2.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_03_gauss.cpp


**Figure 3.14:** Comparação entre filtro de média e Gaussiano (*kernel* 9×9, σ=0). O Gaussiano preserva melhor as bordas, visível no detalhe do rosto.


In [50]:
!g++ -I. -std=c++17 tmp/fig_03_gauss.cpp -o tmp/fig_03_gauss \
  && ./tmp/fig_03_gauss \
  && test -f "tmp/fig_03_gauss.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_gauss.png"

[1] Detalhe: Original
[2] Média 9×9
[3] Gaussiano 9×9


In [51]:
try:
    mm.show(
        [
            mm.read("tmp/fig_03_gauss_0.png"),
            mm.read("tmp/fig_03_gauss_1.png"),
            mm.read("tmp/fig_03_gauss_2.png"),
        ],
        titles=[
            'Detalhe: Original',
            'Média 9×9',
            'Gaussiano 9×9',
        ],
        cols=3,
        figsize=(12, 8),
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_gauss_0.png (ver a versao Python)")

<Figure size 1800x1200 with 3 Axes>

**Figure 3.14:** Comparação entre filtro de média e Gaussiano (*kernel* 9×9, σ=0). O Gaussiano preserva melhor as bordas, visível no detalhe do rosto.


## 3.6 Enhancement Spatial Filtering

Enhancement filters emphasize abrupt intensity transitions, increasing sharpness and edge visibility. They are **high-pass** filters — they amplify high-frequency components (edges, texture) and suppress low-frequency ones (uniform regions).

The intuition is simple: if we subtract from an image its smoothed version (which contains only low frequencies), what remains are the high frequencies — edges and details. Adding this residual back to the original image increases local contrast:

<a id="eq-03-realce-intuitivo"></a>
$$
g = f + k\,(f - f_{\text{smooth}}), \quad k > 0 \tag{3.15}
$$


[Figure 3.15](#fig-03-sim-03-filtragem1d) illustrates this process on a synthetic 1D signal with three distinct structures: a wide step, a narrow peak, and a smooth ramp. Panel ① shows the original signal $f(x)$; panel ② shows the smoothed version $f_{\text{smooth}}(x)$ obtained by moving average — note how the narrow peak is attenuated. Panel ③ displays the residual $f - f_{\text{smooth}}$, which retains only the abrupt transitions. Finally, panel ④ shows $g(x)$: the previously attenuated peak is restored and amplified relative to the original. Adjust $k$ and the window size to observe the trade-off between sharpness and noise amplification.

Enhancement filters formalize this idea directly in the kernel, without requiring two separate steps.

In [52]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-03-filtragem1d" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🎮 Simulator: 1D Spatial Unsharp Filtering</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = f + k·(f − f_smooth)</span>
  </div>

  <div style="padding:16px;background:#ffffff;">

    <!-- Controles -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;display:flex;flex-wrap:wrap;gap:16px;align-items:flex-end;">
      
      <div style="display:flex;flex-direction:column;gap:4px;flex:1;min-width:140px;">
        <label style="font-size:11px;font-weight:700;color:#b9770e;">Parameter k (Enhancement)</label>
        <input id="sim_cap03_k_slider" type="range" min="0" max="5" step="0.1" value="1.5" style="width:100%;cursor:pointer;">
        <span style="font-size:11px;font-family:monospace;font-weight:700;color:#26241d;">k = <span id="sim_cap03_k_val">1.5</span></span>
      </div>

      <div style="display:flex;flex-direction:column;gap:4px;flex:1;min-width:140px;">
        <label style="font-size:11px;font-weight:700;color:#2980b9;">Smoothing Window (pts)</label>
        <input id="sim_cap03_win_slider" type="range" min="3" max="31" step="2" value="9" style="width:100%;cursor:pointer;">
        <span style="font-size:11px;font-family:monospace;font-weight:700;color:#26241d;">window = <span id="sim_cap03_win_val">9</span></span>
      </div>

      <div style="display:flex;flex-direction:column;gap:4px;flex:1;min-width:140px;">
        <label style="font-size:11px;font-weight:700;color:#8e44ad;">Noise Level σ</label>
        <input id="sim_cap03_noise_slider" type="range" min="0" max="0.3" step="0.01" value="0.04" style="width:100%;cursor:pointer;">
        <span style="font-size:11px;font-family:monospace;font-weight:700;color:#26241d;">σ = <span id="sim_cap03_noise_val">0.04</span></span>
      </div>

    </div>

    <!-- Gráficos com Fundos Pastéis -->
    <div style="display:flex;flex-direction:column;gap:6px;">

      <!-- ① Azul Pastéis -->
      <div style="background:#ebf4fd;border:1px solid #d4e5f7;border-radius:8px;padding:8px 10px;">
        <div style="font-size:11px;font-weight:700;color:#1a5fa8;margin-bottom:3px;font-family:monospace;">① f(x) — Original Signal</div>
        <canvas id="sim_cap03_c1" height="80" style="width:100%;display:block;"></canvas>
      </div>

      <div style="text-align:center;font-size:14px;color:#8a8371;line-height:1;">↓ low-pass filter (moving average)</div>

      <!-- ② Verde Pastéis -->
      <div style="background:#eaf7f2;border:1px solid #d1efe3;border-radius:8px;padding:8px 10px;">
        <div style="font-size:11px;font-weight:700;color:#0d6b4f;margin-bottom:3px;font-family:monospace;">② f_smooth(x) — Peak Attenuated by Filter</div>
        <canvas id="sim_cap03_c2" height="80" style="width:100%;display:block;"></canvas>
      </div>

      <div style="text-align:center;font-size:14px;color:#8a8371;line-height:1;">↓ subtraction: f − f_smooth</div>

      <!-- ③ Salmão/Rosa Pastéis -->
      <div style="background:#fef0eb;border:1px solid #fcdad0;border-radius:8px;padding:8px 10px;">
        <div style="font-size:11px;font-weight:700;color:#a03010;margin-bottom:3px;font-family:monospace;">③ Residual (f − f_smooth) — High Frequencies / Edges</div>
        <canvas id="sim_cap03_c3" height="80" style="width:100%;display:block;"></canvas>
      </div>

      <div style="text-align:center;font-size:14px;color:#8a8371;line-height:1;">↓ addition: f + k · residual</div>

      <!-- ④ Roxo Pastéis -->
      <div style="background:#f2f0fd;border:1px solid #e1dcf9;border-radius:8px;padding:8px 10px;">
        <div style="font-size:11px;font-weight:700;color:#4a3faa;margin-bottom:3px;font-family:monospace;">④ g(x) — Signal with Enhanced Peak</div>
        <canvas id="sim_cap03_c4" height="80" style="width:100%;display:block;"></canvas>
      </div>

    </div>

  </div>
</div>

<script>
(function(){
  function initSimCap03(root){
    if (!root || root.dataset.simCap03Init) return;
    root.dataset.simCap03Init = "1";

    var N = 250;
    var peak_i = 115;

    var slK     = root.querySelector('#sim_cap03_k_slider');
    var slWin   = root.querySelector('#sim_cap03_win_slider');
    var slNoise = root.querySelector('#sim_cap03_noise_slider');

    var valK     = root.querySelector('#sim_cap03_k_val');
    var valWin   = root.querySelector('#sim_cap03_win_val');
    var valNoise = root.querySelector('#sim_cap03_noise_val');

    function make_signal(noise) {
      var seed = 42;
      function rand() { seed = (seed * 9301 + 49297) % 233280; return seed / 233280; }
      function randn() { return Math.sqrt(-2 * Math.log(rand() + 1e-9)) * Math.cos(2 * Math.PI * rand()); }
      var f = [];
      for (var i = 0; i < N; i++) {
        var v = 0.2;
        if (i >= 30  && i <= 80)  v += 0.7;
        if (i >= 110 && i <= 120) v += 1.0;
        if (i >= 150 && i <= 190) v += 0.5 * (i - 150) / 40;
        v += noise * randn();
        f.push(v);
      }
      return f;
    }

    function moving_avg(f, win) {
      var half = Math.floor(win / 2);
      return f.map(function(_, i){
        var s = 0, c = 0;
        for (var j = Math.max(0, i - half); j <= Math.min(f.length - 1, i + half); j++) {
          s += f[j];
          c++;
        }
        return s / c;
      });
    }

    function drawCanvas(canvasId, datasets, annotations) {
      var canvas = root.querySelector('#' + canvasId);
      if (!canvas) return;
      canvas.width = canvas.offsetWidth || 800;
      var ctx = canvas.getContext('2d');
      var W = canvas.width, H = canvas.height;
      ctx.clearRect(0, 0, W, H);

      var padL = 8, padR = 8, padT = 6, padB = 6;
      var allVals = [].concat.apply([], datasets.map(function(d){ return d.data; }));
      var mn = Math.min.apply(null, allVals);
      var mx = Math.max.apply(null, allVals);
      var rng = mx - mn || 1;
      var drawH = H - padT - padB, drawW = W - padL - padR;

      function toY(v){ return H - padB - (v - mn) / rng * drawH; }
      function toX(i){ return padL + i / (N - 1) * drawW; }

      // Faixa de destaque do pico
      var xA = toX(108), xB = toX(122);
      ctx.fillStyle = 'rgba(255, 200, 50, 0.15)';
      ctx.fillRect(xA, padT, xB - xA, drawH);

      // Grade
      ctx.strokeStyle = 'rgba(38, 36, 29, 0.08)';
      ctx.lineWidth = 0.5;
      for (var g = 0; g <= 3; g++) {
        var gy = padT + g / 3 * drawH;
        ctx.beginPath(); ctx.moveTo(padL, gy); ctx.lineTo(W - padR, gy); ctx.stroke();
      }

      // Linha do zero
      if (mn < 0 && mx > 0) {
        ctx.strokeStyle = 'rgba(38, 36, 29, 0.2)';
        ctx.lineWidth = 0.8;
        ctx.setLineDash([4, 3]);
        var y0 = toY(0);
        ctx.beginPath(); ctx.moveTo(padL, y0); ctx.lineTo(W - padR, y0); ctx.stroke();
        ctx.setLineDash([]);
      }

      // Desenho das séries
      datasets.forEach(function(ds){
        ctx.strokeStyle = ds.color;
        ctx.lineWidth   = ds.width || 1.8;
        ctx.globalAlpha = ds.alpha || 1;
        if (ds.dash) ctx.setLineDash(ds.dash); else ctx.setLineDash([]);
        ctx.beginPath();
        ds.data.forEach(function(v, i){
          if (i === 0) ctx.moveTo(toX(i), toY(v)); else ctx.lineTo(toX(i), toY(v));
        });
        ctx.stroke();
        ctx.setLineDash([]);
        ctx.globalAlpha = 1;
      });

      // Anotações
      if (annotations) {
        annotations.forEach(function(an){
          var xi = toX(an.i), yi = toY(an.v);
          ctx.strokeStyle = an.color || '#5e5a4a';
          ctx.lineWidth = 1;
          ctx.setLineDash([3, 3]);
          ctx.beginPath(); ctx.moveTo(xi, padT); ctx.lineTo(xi, H - padB); ctx.stroke();
          ctx.setLineDash([]);

          ctx.fillStyle = an.color || '#5e5a4a';
          ctx.beginPath(); ctx.arc(xi, yi, 4, 0, 2 * Math.PI); ctx.fill();

          ctx.fillStyle = an.color || '#26241d';
          ctx.font = 'bold 10px monospace';
          ctx.fillText(an.label, xi + 6, Math.max(padT + 12, Math.min(H - padB - 4, yi - 6)));
        });
      }
    }

    function update() {
      var k     = parseFloat(slK.value) || 0;
      var win   = parseInt(slWin.value) || 3;
      var noise = parseFloat(slNoise.value) || 0;

      valK.textContent     = k.toFixed(1);
      valWin.textContent   = win;
      valNoise.textContent = noise.toFixed(2);

      var f       = make_signal(noise);
      var f_suave = moving_avg(f, win);
      var residuo = f.map(function(v, i){ return v - f_suave[i]; });
      var g       = f.map(function(v, i){ return v + k * residuo[i]; });

      var pk = peak_i;
      var peakF = f[pk], peakS = f_suave[pk], peakR = residuo[pk], peakG = g[pk];

      // ① Original (Azul)
      drawCanvas('sim_cap03_c1',
        [{ data: f, color: '#1a5fa8' }],
        [{ i: pk, v: peakF, color: '#1a5fa8', label: 'pico' }]
      );

      // ② Suavizado (Verde)
      drawCanvas('sim_cap03_c2',
        [{ data: f, color: '#1a5fa8', width: 1, alpha: 0.35, dash: [4, 3] },
         { data: f_suave, color: '#0d6b4f', width: 2 }],
        [{ i: pk, v: peakS, color: '#0d6b4f', label: 'atenuado' }]
      );

      // ③ Resíduo (Salmão / Laranja)
      drawCanvas('sim_cap03_c3',
        [{ data: residuo, color: '#a03010' }],
        [{ i: pk, v: peakR, color: '#a03010', label: 'borda detectada' }]
      );

      // ④ Realçado (Roxo)
      drawCanvas('sim_cap03_c4',
        [{ data: f, color: '#1a5fa8', width: 1, alpha: 0.35, dash: [4, 3] },
         { data: g, color: '#4a3faa', width: 2.2 }],
        [{ i: pk, v: peakG, color: '#4a3faa', label: 'pico realçado' }]
      );
    }

    [slK, slWin, slNoise].forEach(function(sl){
      sl.addEventListener('input', update);
    });

    update();
  }

  function tryInitSimCap03(){
    var root = document.getElementById('sim-03-filtragem1d');
    if (root) initSimCap03(root); else setTimeout(tryInitSimCap03, 200);
  }
  tryInitSimCap03();
})();
</script>
</div>
""")

**Figure 3.15:** Simulator: 1D Spatial Enhancement Filtering (Unsharp Masking and High-Boost)


<figure id="fig-03-sim-03-filtragem1d">
  <img src="imagens/fig-03-sim-03-filtragem1d.png" alt=" Simulator: 1D Spatial Enhancement Filtering (Unsharp Masking and High-Boost) " style="max-width:80%" />
  <figcaption><strong>Figure 3.15:</strong>  Simulator: 1D Spatial Enhancement Filtering (Unsharp Masking and High-Boost) </figcaption>
</figure>

### 3.6.1 Laplacian

The Laplacian is an isotropic **second-derivative** operator, meaning it responds equally to variations in all directions, unlike first-derivative operators such as Sobel and Prewitt, which are directional:

<a id="eq-03-laplaciano"></a>
$$
\nabla^2 f = \frac{\partial^2 f}{\partial x^2} + \frac{\partial^2 f}{\partial y^2} \tag{3.16}
$$


An important property of the second derivative is that its value is **close to zero in uniform regions** and high at intensity transitions. Thus, by subtracting the Laplacian from the original image, edges and details are enhanced, increasing local contrast:

<a id="eq-03-realce-lap"></a>
$$
g(x,y) = f(x,y) - \nabla^2 f(x,y) \tag{3.17}
$$


In the discrete form, the second derivative in $x$ is approximated by $f(x+1,y) - 2f(x,y) + f(x-1,y)$, and similarly in $y$. By summing the two directions, we obtain the *kernel* $w_4$ (4-neighbor) or $w_8$ (8-neighbor, including diagonals):

<a id="eq-03-laplaciano-kernel"></a>
$$
w_4 = \begin{bmatrix} 0 & 1 & 0 \\ 1 & -4 & 1 \\ 0 & 1 & 0 \end{bmatrix}, \qquad
w_8 = \begin{bmatrix} 1 & 1 & 1 \\ 1 & -8 & 1 \\ 1 & 1 & 1 \end{bmatrix} \tag{3.18}
$$


> ### 📝 Zero sum and negative center
>
> Both *kernels* have **coefficient sum equal to zero**: in uniform regions, the output is 0 — the Laplacian does not alter the average brightness, it only detects variations. The **negative center** indicates that the pixel is compared with its neighbors: the more it stands out (upward or downward), the greater the absolute value of the Laplacian at that point.

In the following example, the central pixel $[1,1]=107$ has neighbors $\{95, 106, 108, 103\}$. Since these values are close to one another, the region is nearly uniform, and the Laplacian returns a low value, producing little enhancement. In edge regions, where there are greater differences between the central pixel and its neighbors, the Laplacian assumes higher values (positive or negative), and the operation in [Equation 3.17](#eq-03-realce-lap) intensifies these transitions.

[Figure 3.16](#fig-03-laplaciano-patch) illustrates the computation of the Laplacian using the *kernel* $w_4$ in a $3\times3$ neighborhood highlighted within a $5\times5$ patch. The example shows the value obtained by the operator and the corresponding enhanced pixel in the output image, which changes from 107 to 123.

In [53]:
%%writefile tmp/fig_03_laplaciano_patch.cpp
#define MM_OUT "tmp/fig_03_laplaciano_patch.png"
#include "morph.hpp"
#include <iostream>
#include <vector>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray = mm::_read_state("tmp/state/img_gray_45.png");
// [pdi:state-io:end]

    //| label: fig-03-laplaciano-patch
    //| fig-cap: "*Kernel* Laplaciano w4 sobre o *patch* 5×5: a janela amarela destaca a vizinhança 3×3 onde o operador de segunda derivada é calculado."
    //| echo: true
    //| output: true

    mm::Image patch = mm::crop(img_gray, 250, 255, 250, 255);

    std::cout << "Patch 5x5 (intensidades):" << std::endl;
    std::cout << mm::drawImg(patch) << std::endl;

    mm::Kernel B = {{1,1,1},{1,1,1},{1,1,1}};
    mm::drawImgKernel(patch, B, 1, 1, MM_OUT, 40);

    return 0;
}

Overwriting tmp/fig_03_laplaciano_patch.cpp


In [54]:
!g++ -I. -std=c++17 tmp/fig_03_laplaciano_patch.cpp -o tmp/fig_03_laplaciano_patch \
  && ./tmp/fig_03_laplaciano_patch \
  && test -f "tmp/fig_03_laplaciano_patch.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_laplaciano_patch.png"

Patch 5x5 (intensidades):
 94  96 103 113 115 
107 104 103 107 114 
 98 102 112 117 116 
 81  91 112 122 118 
 85  91 110 119 121 

Processando pixel (x,y)=(1,1)  |  janela do kernel 3x3
 94  96 103 113 115 
107 104 103 107 114 
 98 102 112 117 116 
 81  91 112 122 118 
 85  91 110 119 121 


In [55]:
try:
    mm.show(mm.read("tmp/fig_03_laplaciano_patch.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_laplaciano_patch.png (ver a versao Python)")

<Figure size 450x450 with 1 Axes>

**Figure 3.16:** *Kernel* Laplaciano w4 sobre o *patch* 5×5: a janela amarela destaca a vizinhança 3×3 onde o operador de segunda derivada é calculado.


[Figure 3.17](#fig-03-laplaciano) compares the application of the Laplacian *kernels* $w_4$ and $w_8$ on a larger crop of the leopard image. For each case, the raw operator response, which highlights edges and intensity transitions, and the image obtained after enhancement by Laplacian subtraction are shown. It is observed that the *kernel* $w_8$, by also considering diagonal neighbors, produces a stronger response and detects variations in more directions, resulting in a slightly more pronounced enhancement.

In [56]:
%%writefile tmp/fig_03_laplaciano.cpp
#define MM_OUT "tmp/fig_03_laplaciano.png"
//| label: fig-03-laplaciano
//| fig-cap: "Laplaciano applied to the leopard image: raw response (edges) with w4 and w8, and images enhanced by subtracting the Laplacian. w8 is more sensitive to diagonals."
//| echo: true
//| output: true

#include "morph.hpp"
#include <vector>
#include <string>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray_crop = mm::_read_state("tmp/state/img_gray_crop_58.png");
// [pdi:state-io:end]

    mm::Kernel w4{{0,1,0},{1,-4,1},{0,1,0}};
    mm::Kernel w8{{1,1,1},{1,-8,1},{1,1,1}};

    mm::show(
        std::vector<mm::Image>{img_gray_crop, mm::laplacian_viz(img_gray_crop, w4), mm::laplacian(img_gray_crop, w4),
         img_gray_crop, mm::laplacian_viz(img_gray_crop, w8), mm::laplacian(img_gray_crop, w8)},
        MM_OUT,
        std::vector<std::string>{"Original", "Laplaciano w4", "Realce w4",
                "Original", "Laplaciano w8", "Realce w8"},
        3
    );

    return 0;
}

Overwriting tmp/fig_03_laplaciano.cpp


In [57]:
!g++ -I. -std=c++17 tmp/fig_03_laplaciano.cpp -o tmp/fig_03_laplaciano \
  && ./tmp/fig_03_laplaciano \
  && test -f "tmp/fig_03_laplaciano.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_laplaciano.png"

[1] Original
[2] Laplaciano w4
[3] Realce w4
[4] Original
[5] Laplaciano w8
[6] Realce w8


In [58]:
try:
    mm.show(mm.read("tmp/fig_03_laplaciano.png"), figsize=(14, 8))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_laplaciano.png (ver a versao Python)")

<Figure size 2100x1200 with 1 Axes>

**Figure 3.17:** Laplaciano aplicado à imagem do leopardo: resposta bruta (bordas) com w4 e w8, e imagens realçadas pela subtração do Laplaciano. w8 é mais sensível às diagonais.


### 3.6.2 Sobel Operator

The Sobel operator estimates the **first-order partial derivatives** in the horizontal and vertical directions. Unlike the Laplacian (second derivative), Sobel is directional and more robust to noise, as each *kernel* combines a derivative with a perpendicular Gaussian smoothing:

<a id="eq-03-sobel"></a>
$$
G_x = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix} * f, \qquad
G_y = \begin{bmatrix} -1 & -2 & -1 \\ 0 & 0 & 0 \\ 1 & 2 & 1 \end{bmatrix} * f \tag{3.19}
$$


$G_x$ detects **vertical** edges (variation in the $x$ direction); $G_y$ detects **horizontal** edges (variation in the $y$ direction). The weights $\{1,2,1\}$ in the perpendicular direction correspond to 1D Gaussian smoothing, which reduces sensitivity to noise.

> ### 📝 Sobel is correlation, not convolution
>
> The Sobel *kernels* are **asymmetric** — a 180° rotation alters the result. `cv2.Sobel` implements cross-correlation (like `cv2.filter2D`). To obtain the correct directional derivative, the signs are already defined for correlation: $G_x$ returns positive values where intensity increases from left to right.

The **gradient** magnitude combines the two components, representing edge strength independent of direction:

<a id="eq-03-gradiente"></a>
$$
|\nabla f| = \sqrt{G_x^2 + G_y^2} \tag{3.20}
$$


And the **direction** of the gradient (perpendicular to the edge) is:

<a id="eq-03-direcao"></a>
$$
\theta = \arctan\left(\frac{G_y}{G_x}\right) \tag{3.21}
$$


To illustrate numerically, $G_x$ and $G_y$ are computed manually at the central pixel $[1,1]$ of the 5×5 *patch*:

The reduced value of $|{\nabla f}|$ in this *patch* confirms that the region is nearly uniform, since the gradient assumes high values only where there are significant intensity changes. [Figure 3.18](#fig-03-sobel) applies the Sobel operator to a larger crop of the leopard image. The horizontal ($G_x$) and vertical ($G_y$) responses are shown, obtained by convolution with the respective Sobel *kernels*, as well as the magnitude $|{\nabla f}|$, calculated from the combination of both. While $G_x$ highlights vertical edges and $G_y$ horizontal edges, the magnitude evidences edges in any direction.

In [59]:
%%writefile tmp/fig_03_sobel.cpp
#define MM_OUT "tmp/fig_03_sobel.png"
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray_crop = mm::_read_state("tmp/state/img_gray_crop_58.png");
// [pdi:state-io:end]

    //| label: fig-03-sobel
    //| fig-cap: "Operador de Sobel na imagem do leopardo: a magnitude |∇f| combina os gradientes horizontal e vertical, revelando todas as bordas. A decomposição Gx/Gy com sinal fica na trilha Python — mm::sobel devolve a magnitude já com clip."
    //| echo: true
    //| output: true

    mm::Image mag = mm::sobel(img_gray_crop);

    mm::show(std::vector<mm::Image>{img_gray_crop, mag},
             MM_OUT,
             std::vector<std::string>{"Original", "Magnitude |grad f| (mm.sobel)"}, 2);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray_crop, "tmp/fig_03_sobel_0.png");
mm::write(mag, "tmp/fig_03_sobel_1.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_03_sobel.cpp


In [60]:
!g++ -I. -std=c++17 tmp/fig_03_sobel.cpp -o tmp/fig_03_sobel \
  && ./tmp/fig_03_sobel \
  && test -f "tmp/fig_03_sobel.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_sobel.png"

[1] Original
[2] Magnitude |grad f| (mm.sobel)


In [61]:
try:
    mm.show(
        [
            mm.read("tmp/fig_03_sobel_0.png"),
            mm.read("tmp/fig_03_sobel_1.png"),
        ],
        titles=[
            'Original',
            'Magnitude |grad f| (mm.sobel)',
        ],
        cols=2,
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_sobel_0.png (ver a versao Python)")

<Figure size 1500x750 with 2 Axes>

**Figure 3.18:** Operador de Sobel na imagem do leopardo: a magnitude |∇f| combina os gradientes horizontal e vertical, revelando todas as bordas. A decomposição Gx/Gy com sinal fica na trilha Python — mm::sobel devolve a magnitude já com clip.


### 3.6.3 Prewitt Operator

The Prewitt operator is structurally identical to Sobel, but replaces the Gaussian weighting $\{1,2,1\}$ with uniform weights $\{1,1,1\}$:

<a id="eq-03-prewitt"></a>
$$
G_x = \begin{bmatrix} -1 & 0 & 1 \\ -1 & 0 & 1 \\ -1 & 0 & 1 \end{bmatrix} * f, \qquad
G_y = \begin{bmatrix} -1 & -1 & -1 \\ 0 & 0 & 0 \\ 1 & 1 & 1 \end{bmatrix} * f \tag{3.22}
$$


The gradient magnitude and direction follow the same equations as Sobel ([Equation 3.20](#eq-03-gradiente) and [Equation 3.21](#eq-03-direcao)). The practical difference is that Prewitt is slightly more sensitive to noise—the uniform perpendicular smoothing weights the central pixel of the line less—but it is computationally simpler. In low-noise images, the results are equivalent.

In [62]:
%%writefile tmp/fig_03_prewitt.cpp
#define MM_OUT "tmp/fig_03_prewitt.png"
// Compile: g++ -std=c++17 -o prewitt prewitt.cpp $(pkg-config --cflags --libs morph)
#include "morph.hpp"
#include <iostream>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray_crop = mm::_read_state("tmp/state/img_gray_crop_58.png");
// [pdi:state-io:end]

    mm::show(std::vector<mm::Image>{img_gray_crop, mm::prewitt(img_gray_crop)},
             MM_OUT,
             std::vector<std::string>{"Original", "Magnitude |grad f| (mm.prewitt)"},
             2);
    return 0;
}

Overwriting tmp/fig_03_prewitt.cpp


In [63]:
!g++ -I. -std=c++17 tmp/fig_03_prewitt.cpp -o tmp/fig_03_prewitt \
  && ./tmp/fig_03_prewitt \
  && test -f "tmp/fig_03_prewitt.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_prewitt.png"

[1] Original
[2] Magnitude |grad f| (mm.prewitt)


In [64]:
try:
    mm.show(mm.read("tmp/fig_03_prewitt.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_prewitt.png (ver a versao Python)")

<Figure size 524x450 with 1 Axes>

**Figure 3.19:** Operador de Prewitt: magnitude do gradiente, comparável ao Sobel mas sem a ponderação central. mm::prewitt devolve |∇f| com clip.


### 3.6.4 *Unsharp Masking* (USM)

*Unsharp Masking* is a classic sharpening technique originating from analog photography, now widely used in image editing software. The central idea is to extract the **high-frequency components** of the image (edges and details) and add them back to the original with a weight $k$:

<a id="tbl-03-usm"></a>

**Tabela 3.3:** Steps of *Unsharp Masking*.

| Step | Operation | Description |
|:-----:|:---------|:----------|
| 1 | $\bar{f} = f * G_\sigma$ | Smooths with a Gaussian — retains low frequencies |
| 2 | $m = f - \bar{f}$ | Mask: difference = high frequencies (edges) |
| 3 | $g = f + k \cdot m$ | Weighted sum of the mask added to the original |


Substituting step 2 into step 3 yields the compact expression:

<a id="eq-03-usm"></a>
$$
g = f + k\,(f - f*G_\sigma) = (1+k)\,f - k\,(f*G_\sigma) \tag{3.23}
$$


The parameter $k$ controls the intensity of the enhancement:

- $k = 0$: no enhancement ($g = f$);
- $k = 1$: classic USM — doubles the contribution of high frequencies;
- $k > 1$: *High Boost Filtering* — amplification beyond double, useful for highly blurred images.

> ### ⚠️ Noise Amplification
>
> USM does not distinguish edges from noise — both are high-frequency components. For high $k$, the noise present in the image is amplified along with the edges. Therefore, it is advisable to apply slight smoothing before USM on noisy images, or to use a small $\sigma$ in the Gaussian.

To illustrate the steps of USM, [Figure 3.20](#fig-03-usm2) applies the method to a $30\times30$ *patch* of the leopard image, using $\sigma=1$ and $k=1$. Initially, the image is smoothed by a Gaussian filter. Then, the high-frequency mask is obtained by the difference between the original and the smoothed image. Finally, this mask is added to the original image, enhancing edges and details. The figure presents the three steps of the process and the final enhancement result.

In [65]:
%%writefile tmp/fig_03_usm2.cpp
#define MM_OUT "tmp/fig_03_usm2.png"
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray_crop = mm::_read_state("tmp/state/img_gray_crop_58.png");
// [pdi:state-io:end]

    //| label: fig-03-usm2
    //| fig-cap: "Realce por *Unsharp Masking* num *patch* do leopardo: mm::usm faz suavização Gaussiana, subtrai da original (máscara de alta frequência) e reintroduz a máscara realçada."
    //| echo: true
    //| output: true

    mm::Image patch = mm::crop(img_gray_crop, 35, 65, 45, 75);

    mm::Image p_suave = mm::gaussian(patch, 7, 1.0);   // suavização Gaussiana
    mm::Image p_usm   = mm::usm(patch, 1.0);           // realce completo (k = 1.0)

    mm::show(std::vector<mm::Image>{patch, p_suave, p_usm},
            MM_OUT,
            std::vector<std::string>{"Patch original", "Suavizado (σ=1)", "Realçado USM (k=1)"}, 3);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(patch, "tmp/fig_03_usm2_0.png");
mm::write(p_suave, "tmp/fig_03_usm2_1.png");
mm::write(p_usm, "tmp/fig_03_usm2_2.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_03_usm2.cpp


In [66]:
!g++ -I. -std=c++17 tmp/fig_03_usm2.cpp -o tmp/fig_03_usm2 \
  && ./tmp/fig_03_usm2 \
  && test -f "tmp/fig_03_usm2.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_usm2.png"

[1] Patch original
[2] Suavizado (σ=1)
[3] Realçado USM (k=1)


In [67]:
try:
    mm.show(
        [
            mm.read("tmp/fig_03_usm2_0.png"),
            mm.read("tmp/fig_03_usm2_1.png"),
            mm.read("tmp/fig_03_usm2_2.png"),
        ],
        titles=[
            'Patch original',
            'Suavizado (σ=1)',
            'Realçado USM (k=1)',
        ],
        cols=3,
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_usm2_0.png (ver a versao Python)")

<Figure size 2250x750 with 3 Axes>

**Figure 3.20:** Realce por *Unsharp Masking* num *patch* do leopardo: mm::usm faz suavização Gaussiana, subtrai da original (máscara de alta frequência) e reintroduz a máscara realçada.


[Figure 3.21](#fig-03-usm) applies the USM method to a larger crop of the leopard image using $\sigma=1$ and different values of the gain factor $k$. In all cases, the high-frequency mask is obtained by the difference between the original image and its version smoothed by a Gaussian filter. The parameter $k$ controls the intensity of the enhancement: smaller values produce a subtle increase in sharpness, while larger values progressively reinforce edges and details. It is observed that, for high values of $k$, halos appear around edges and the noise present in the image becomes amplified.

In [68]:
%%writefile tmp/fig_03_usm.cpp
#define MM_OUT "tmp/fig_03_usm.png"
//| label: fig-03-usm
//| fig-cap: "*Unsharp Masking* on the leopard image with σ=1 and k from 0.5 to 8.0. For k>2 halos appear at edges and background noise appears."
//| echo: true
//| output: true

#include "morph.hpp"
#include <vector>
#include <string>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray_crop = mm::_read_state("tmp/state/img_gray_crop_58.png");
// [pdi:state-io:end]

    mm::show(
        std::vector<mm::Image>{img_gray_crop,
            mm::usm(img_gray_crop, 0.5), mm::usm(img_gray_crop, 1.0),
            mm::usm(img_gray_crop, 3.0), mm::usm(img_gray_crop, 5.0),
            mm::usm(img_gray_crop, 8.0)},
        MM_OUT,
        std::vector<std::string>{"Original", "USM k=0.5", "USM k=1.0", "USM k=3.0", "USM k=5.0", "USM k=8.0"},
        3
    );
    return 0;
}

Overwriting tmp/fig_03_usm.cpp


In [69]:
!g++ -I. -std=c++17 tmp/fig_03_usm.cpp -o tmp/fig_03_usm \
  && ./tmp/fig_03_usm \
  && test -f "tmp/fig_03_usm.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_usm.png"

[1] Original
[2] USM k=0.5
[3] USM k=1.0
[4] USM k=3.0
[5] USM k=5.0
[6] USM k=8.0


In [70]:
try:
    mm.show(mm.read("tmp/fig_03_usm.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_usm.png (ver a versao Python)")

<Figure size 784x524 with 1 Axes>

**Figure 3.21:** *Unsharp Masking* na imagem do leopardo com σ=1 e k de 0.5 a 8.0. Para k>2 surgem halos nas bordas e o ruído de fundo aparece.


### 3.6.5 Canny Detector

Canny combines four steps in sequence — Gaussian smoothing, Sobel gradient,
non-maximum suppression, and dual-threshold hysteresis — to produce **thin,
binary, and connected** edges. Unlike Sobel and Prewitt, the result is not a
continuous gradient map, but a mask where each pixel is either an edge or not.

The central parameter is the threshold pair $(T_{low}, T_{high})$. Pixels with gradient above
$T_{high}$ are certain edges; below $T_{low}$, they are discarded. The ambiguous pixels —
between the two thresholds — are decided by **hysteresis**: they become edges if they are
connected to a certain edge, and are discarded otherwise. This avoids both the loss of
weak segments of real edges and the inclusion of isolated noise. A common heuristic
is $T_{high} = 3 \times T_{low}$.

In [71]:
%%writefile tmp/fig_03_canny.cpp
#define MM_OUT "tmp/fig_03_canny.png"
//| label: fig-03-canny
//| fig-cap: "Detector de Canny com diferentes pares de limiar: limiares baixos capturam mais bordas (inclusive ruído); limiares altos retêm apenas as bordas mais fortes."
//| echo: true
//| output: true

#include "morph.hpp"
#include <iostream>
#include <vector>
#include <string>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray_crop = mm::_read_state("tmp/state/img_gray_crop_58.png");
// [pdi:state-io:end]

    mm::show(
        std::vector<mm::Image>{
            img_gray_crop,
            mm::canny(img_gray_crop, 30, 90),
            mm::canny(img_gray_crop, 60, 180),
            mm::canny(img_gray_crop, 120, 240)
        },
        MM_OUT,
        std::vector<std::string>{"Original", "Canny (30/90)", "Canny (60/180)", "Canny (120/240)"},
        4
    );
    return 0;
}

Overwriting tmp/fig_03_canny.cpp


In [72]:
!g++ -I. -std=c++17 tmp/fig_03_canny.cpp -o tmp/fig_03_canny \
  && ./tmp/fig_03_canny \
  && test -f "tmp/fig_03_canny.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_canny.png"

[1] Original
[2] Canny (30/90)
[3] Canny (60/180)
[4] Canny (120/240)


In [73]:
try:
    mm.show(mm.read("tmp/fig_03_canny.png"))
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_canny.png (ver a versao Python)")

<Figure size 1044x450 with 1 Axes>

**Figure 3.22:** Detector de Canny com diferentes pares de limiar: limiares baixos capturam mais bordas (inclusive ruído); limiares altos retêm apenas as bordas mais fortes.


> ### 📝 Threshold Selection
>
> A common heuristic is $T_{high} = 3 \times T_{low}$. Typical values depend on the image's gradient range — `cv2.Canny` accepts absolute values in $[0, 255]$. For images with variable contrast, computing the thresholds from the percentiles of the Sobel magnitude is more robust than using fixed values.

## 3.7 Order Filters: Median Filter

Order-statistic filters replace the central pixel with the value of a **percentile** of the neighborhood intensity distribution—unlike linear filters, which compute weighted combinations. The most important one is the **median filter**.

### 3.7.1 Impulsive Noise: Salt and Pepper

**Salt-and-pepper noise** replaces random pixels with extreme values: 0 (pepper, black) or 255 (salt, white). It is common in image transmission with bit errors and in cameras with defective sensors.

To understand why linear filters fail, consider a 3×3 neighborhood where a single pixel has been corrupted to 255:

$$
\text{neighborhood} = \begin{bmatrix} 102 & 98 & 105 \\ 100 & \mathbf{255} & 97 \\ 103 & 99 & 101 \end{bmatrix}
$$

<a id="tbl-03-mediana"></a>

**Tabela 3.4:** Mean vs. median with one corrupted pixel. The median ignores the outlier; the mean is shifted by ~40 levels.

| Method | Calculation | Result |
|:-------|:--------|----------:|
| Mean | (102+98+...+255+...+101)/9 | ≈ 140 |
| Median | {97,98,99,100,**101**,102,103,105,255} | 101 |


> ### ⚠️ Why do mean filters fail with impulsive noise?
>
> The mean is sensitive to ***outliers***—a single pixel with a value of 255 in a neighborhood with values ≈ 100 raises the output to ≈ 140, spreading the noise throughout the image. The median, being a **robust estimator**, selects the central value of the sorted distribution, naturally discarding the extremes without any special adjustment.

The following example illustrates the behavior of the mean and the median in the presence of a pixel corrupted by impulsive noise. It is observed that the mean is strongly influenced by the extreme value (255), producing an estimate far from the predominant values of the neighborhood. Meanwhile, the median remains close to the original value of the region, evidencing its greater robustness to *outliers* and justifying its use in salt-and-pepper noise removal.

[Figure 3.23](#fig-03-ruido) shows the effect of salt-and-pepper noise at different densities. The noise was generated by randomly replacing a fraction of pixels with minimum values (0, pepper) and maximum values (255, salt). As the density increases from 2% to 10%, the number of corrupted pixels grows, making the visual degradation more evident and hindering the perception of image details.

In [74]:
%%writefile tmp/fig_03_ruido.cpp
#define MM_OUT "tmp/fig_03_ruido.png"
//| label: fig-03-ruido
//| fig-cap: "Ruído sal e pimenta com densidades crescentes (2%, 5%, 10%): metade dos pixels corrompidos vira sal (255), metade pimenta (0)."
//| echo: true
//| output: true

#include "morph.hpp"
#include <random>
#include <vector>
#include <string>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray_crop = mm::_read_state("tmp/state/img_gray_crop_58.png");
// [pdi:state-io:end]

    auto salt_pepper = [](const mm::Image& img, double prob) {
        mm::Image out = img;
        int h = out.h, w = out.w;
        std::mt19937 rng(42);
        std::uniform_int_distribution<int> dist_y(0, h - 1);
        std::uniform_int_distribution<int> dist_x(0, w - 1);
        std::uniform_real_distribution<double> dist_r(0.0, 1.0);
        int n = static_cast<int>(prob * h * w);
        for (int i = 0; i < n; ++i) {
            int y = dist_y(rng);
            int x = dist_x(rng);
            out.at(y, x) = (dist_r(rng) < 0.5) ? 0 : 255;
        }
        return out;
    };

    mm::Image n2  = salt_pepper(img_gray_crop, 0.02);
    mm::Image n5  = salt_pepper(img_gray_crop, 0.05);
    mm::Image n10 = salt_pepper(img_gray_crop, 0.10);

    mm::show({img_gray_crop, n2, n5, n10}, MM_OUT,
             {"Original", "Ruido 2%", "Ruido 5%", "Ruido 10%"}, 4);

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray_crop, "tmp/fig_03_ruido_0.png");
mm::write(n2, "tmp/fig_03_ruido_1.png");
mm::write(n5, "tmp/fig_03_ruido_2.png");
mm::write(n10, "tmp/fig_03_ruido_3.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_03_ruido.cpp


In [75]:
!g++ -I. -std=c++17 tmp/fig_03_ruido.cpp -o tmp/fig_03_ruido \
  && ./tmp/fig_03_ruido \
  && test -f "tmp/fig_03_ruido.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_ruido.png"

[1] Original
[2] Ruido 2%
[3] Ruido 5%
[4] Ruido 10%


In [76]:
try:
    mm.show(
        [
            mm.read("tmp/fig_03_ruido_0.png"),
            mm.read("tmp/fig_03_ruido_1.png"),
            mm.read("tmp/fig_03_ruido_2.png"),
            mm.read("tmp/fig_03_ruido_3.png"),
        ],
        titles=[
            'Original',
            'Ruido 2%',
            'Ruido 5%',
            'Ruido 10%',
        ],
        cols=4,
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_ruido_0.png (ver a versao Python)")

<Figure size 3000x750 with 4 Axes>

**Figure 3.23:** Ruído sal e pimenta com densidades crescentes (2%, 5%, 10%): metade dos pixels corrompidos vira sal (255), metade pimenta (0).


### 3.7.2 Median Filter

The median filter replaces each pixel with the **median value** of the pixels in its $n \times n$ neighborhood:

<a id="eq-03-mediana"></a>
$$
g(x,y) = \text{med}_{(s,t) \in \mathcal{V}_{n}} \{f(x+s, y+t)\} \tag{3.24}
$$


The median value is the one occupying the central position when the $n^2$ values of the neighborhood are sorted. For a $3\times3$ window ($n^2=9$ pixels), the median is the 5th value in the sorted sequence.

To illustrate, consider the same 5×5 *patch* with a pixel artificially corrupted at $[1,1]$:

The example confirms: even with the pixel corrupted to 255, the median returns the correct central value — the *outlier* occupies the last position in the ordering and is naturally discarded.

Because it is based on ordering rather than summation, the median has three fundamental properties that distinguish it from linear filters:

- **Robust** to impulsive noise — outliers go to the ends of the ordered sequence and do not affect the central value;
- **Edge-preserving** — abrupt intensity transitions are maintained, since the median selects a value that already exists in the neighborhood, without creating new intermediate levels;
- **Nonlinear** — it cannot be expressed as a convolution, therefore `mm::conv` does not apply; use `cv2.medianBlur`.

[Figure 3.24](#fig-03-ruido-filtros) compares different salt-and-pepper noise removal techniques applied to an image with 10% corrupted pixels. The Gaussian, Mean, Median, Bilateral, and Morphological filters (next chapter) were evaluated, allowing observation of the trade-off between noise removal and detail preservation. In general, the mean and Gaussian filters reduce noise but tend to blur edges, while the median filter performs better for impulsive noise. The bilateral filter preserves edges better, and the morphological filter removes a good portion of corrupted pixels without excessively degrading the image structure.

In [77]:
%%writefile tmp/fig_03_ruido_filtros.cpp
#define MM_OUT "tmp/fig_03_ruido_filtros.png"
// Compile with: g++ -std=c++17 -O2 -o program program.cpp -lmorph
#include <random>
#include "morph.hpp"
#include <filesystem>

// salt and pepper noise generator
//| label: fig-03-ruido-filtros
//| fig-cap: "Filtros para ruído sal e pimenta (10%): Gaussiano, Média e Mediana. Bilateral e morfológico (open+close) ficam só na trilha Python — bilateral não tem equivalente em morph.hpp e morfologia é do próximo capítulo."
//| echo: true
//| output: true

mm::Image salt_pepper(const mm::Image& img, double prob) {
    mm::Image out = img;
    int h = out.h, w = out.w;
    std::mt19937 rng(42);
    std::uniform_int_distribution<int> dist_y(0, h - 1);
    std::uniform_int_distribution<int> dist_x(0, w - 1);
    std::uniform_real_distribution<double> dist_p(0.0, 1.0);
    for (int i = 0; i < static_cast<int>(prob * h * w); ++i) {
        int y = dist_y(rng);
        int x = dist_x(rng);
        out.at(y, x) = (dist_p(rng) < 0.5) ? 0 : 255;
    }
    return out;
}

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray_crop = mm::_read_state("tmp/state/img_gray_crop_58.png");
// [pdi:state-io:end]

    mm::Image noisy = salt_pepper(img_gray_crop, 0.10);

    mm::Image f_gauss   = mm::gaussian(noisy, 5, 1.0);
    mm::Image f_media   = mm::blur(noisy, 5);
    mm::Image f_median3 = mm::median(noisy, 3);
    mm::Image f_median5 = mm::median(noisy, 5);

    mm::show(
        std::vector<mm::Image>{img_gray_crop, noisy, f_gauss, f_media, f_median3, f_median5},
        MM_OUT,
        std::vector<std::string>{"Original", "Ruido 10%", "Gaussiano 5x5", "Media 5x5",
            "Mediana 3x3", "Mediana 5x5"},
        3
    );

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray_crop, "tmp/fig_03_ruido_filtros_0.png");
mm::write(noisy, "tmp/fig_03_ruido_filtros_1.png");
mm::write(f_gauss, "tmp/fig_03_ruido_filtros_2.png");
mm::write(f_media, "tmp/fig_03_ruido_filtros_3.png");
mm::write(f_median3, "tmp/fig_03_ruido_filtros_4.png");
mm::write(f_median5, "tmp/fig_03_ruido_filtros_5.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_03_ruido_filtros.cpp


In [78]:
!g++ -I. -std=c++17 tmp/fig_03_ruido_filtros.cpp -o tmp/fig_03_ruido_filtros \
  && ./tmp/fig_03_ruido_filtros \
  && test -f "tmp/fig_03_ruido_filtros.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_ruido_filtros.png"

[1] Original
[2] Ruido 10%
[3] Gaussiano 5x5
[4] Media 5x5
[5] Mediana 3x3
[6] Mediana 5x5


In [79]:
try:
    mm.show(
        [
            mm.read("tmp/fig_03_ruido_filtros_0.png"),
            mm.read("tmp/fig_03_ruido_filtros_1.png"),
            mm.read("tmp/fig_03_ruido_filtros_2.png"),
            mm.read("tmp/fig_03_ruido_filtros_3.png"),
            mm.read("tmp/fig_03_ruido_filtros_4.png"),
            mm.read("tmp/fig_03_ruido_filtros_5.png"),
        ],
        titles=[
            'Original',
            'Ruido 10%',
            'Gaussiano 5x5',
            'Media 5x5',
            'Mediana 3x3',
            'Mediana 5x5',
        ],
        cols=3,
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_ruido_filtros_0.png (ver a versao Python)")

<Figure size 2250x1500 with 6 Axes>

**Figure 3.24:** Filtros para ruído sal e pimenta (10%): Gaussiano, Média e Mediana. Bilateral e morfológico (open+close) ficam só na trilha Python — bilateral não tem equivalente em morph.hpp e morfologia é do próximo capítulo.


## 3.8 Practical Application: Preprocessing for Segmentation

In practice, the techniques in this chapter are rarely used in isolation. A typical **preprocessing pipeline** combines multiple steps in sequence, tailored to the image type and application. [Figure 3.25](#fig-03-pipeline) illustrates a complete pipeline:

1. **Histogram equalization (CLAHE):** normalizes contrast regardless of lighting conditions;
2. **Gaussian filter:** smooths acquisition noise without destroying edges;
3. **Edge detection (Sobel/Canny):** extracts relevant structures for segmentation.

> ### 📝 Order matters
>
> The order of operations affects the final result. In general: **(1) intensity normalization → (2) noise reduction → (3) enhancement/segmentation**. Reversing the order can amplify noise or cause edges to be lost before detection.

In [80]:
%%writefile tmp/fig_03_pipeline.cpp
#define MM_OUT "tmp/fig_03_pipeline.png"
#include "morph.hpp"
#include <iostream>
#include <vector>
#include <filesystem>

int main() {
// [pdi:state-io] auto-generated — do not edit by hand
mm::Image img_gray_crop = mm::_read_state("tmp/state/img_gray_crop_58.png");
// [pdi:state-io:end]

    // img_gray_crop is provided as an already-valid mm::Image variable

    //| label: fig-03-pipeline
    //| fig-cap: "*Pipeline* de pre-processing: equalization → Gaussian → Canny. The Python track uses CLAHE instead of global equalization; CLAHE has no equivalent in morph.hpp."
    //| echo: true
    //| output: true

    mm::Image img_eq    = mm::equalize(img_gray_crop);     // Step 1: global equalization
    mm::Image img_gauss = mm::gaussian(img_eq, 5, 0);      // Step 2: Gaussian
    mm::Image edges     = mm::canny(img_gauss, 50, 150);   // Step 3: Canny

    mm::Image edges_direct = mm::canny(img_gray_crop, 50, 150);   // Direct Canny, no pre-processing

    mm::show(
        std::vector<mm::Image>{img_gray_crop, img_eq, img_gauss, edges, edges_direct},
        MM_OUT,
        std::vector<std::string>{"Original", "1. Equalized", "2. Gaussian", "3. Canny (pipeline)", "Canny (direct)"},
        5
    );

    
// [pdi:panel-io] auto-generated — do not edit by hand
std::filesystem::create_directories("tmp");
mm::write(img_gray_crop, "tmp/fig_03_pipeline_0.png");
mm::write(img_eq, "tmp/fig_03_pipeline_1.png");
mm::write(img_gauss, "tmp/fig_03_pipeline_2.png");
mm::write(edges, "tmp/fig_03_pipeline_3.png");
mm::write(edges_direct, "tmp/fig_03_pipeline_4.png");
// [pdi:panel-io:end]
return 0;
}

Overwriting tmp/fig_03_pipeline.cpp


In [81]:
!g++ -I. -std=c++17 tmp/fig_03_pipeline.cpp -o tmp/fig_03_pipeline \
  && ./tmp/fig_03_pipeline \
  && test -f "tmp/fig_03_pipeline.png" \
  || echo "⚠ mm::show não gravou tmp/fig_03_pipeline.png"

[1] Original
[2] 1. Equalized
[3] 2. Gaussian
[4] 3. Canny (pipeline)
[5] Canny (direct)


In [82]:
try:
    mm.show(
        [
            mm.read("tmp/fig_03_pipeline_0.png"),
            mm.read("tmp/fig_03_pipeline_1.png"),
            mm.read("tmp/fig_03_pipeline_2.png"),
            mm.read("tmp/fig_03_pipeline_3.png"),
            mm.read("tmp/fig_03_pipeline_4.png"),
        ],
        titles=[
            'Original',
            '1. Equalizado',
            '2. Gaussiano',
            '3. Canny (pipeline)',
            'Canny (direto)',
        ],
        cols=5,
    )
except Exception as _e:
    print("figura indisponivel nesta trilha (C++): " + repr(_e) + " tmp/fig_03_pipeline_0.png (ver a versao Python)")

<Figure size 3750x750 with 5 Axes>

**Figure 3.25:** *Pipeline* de pré-processamento: equalização → Gaussiano → Canny. A trilha Python usa CLAHE no lugar da equalização global; CLAHE não tem equivalente em morph.hpp.


## 3.9 Summary

This chapter presented the main spatial domain processing techniques, from direct pixel manipulation to neighborhood filtering:

- **Point operations:** saturated arithmetic (`mm::addm`, `mm::subm`) and bitwise logic (`mm::band`, `mm::bor`, `mm::bnot`) for ROI cropping and image combination; *alpha blending* (`mm::blend`) for weighted fusion with weight $\alpha \in [0,1]$.
- **Histogram:** discrete function of intensity distribution; visualized with `mm::histImg` and computed with `mm::hist`; the basis for tonal diagnosis and for equalization and specification techniques.
- **Equalization:** automatic redistribution of intensities via the CDF (`mm::equalize`), with the adaptive CLAHE variant for local contrast control.
- **Histogram specification:** transfer of the tonal profile from a reference image via inverse CDF mapping — a generalization of equalization to arbitrary distributions.
- **Correlation and convolution:** sliding window mechanism implemented in `mm::conv` (`cv2.filter2D`); distinguished by the 180° rotation of the *kernel* — relevant only for asymmetric kernels.
- **Smoothing filters:** mean (uniform *kernel*, blurs edges proportionally to size) and Gaussian (radial weighting, separable, no *ringing*, preserves edges better).
- **Sharpening filters:** Laplacian ($w_4$/$w_8$, isotropic second derivative), Sobel (first-order directional gradient, with magnitude $|\nabla f|$ and direction $\theta$) and *Unsharp Masking* (high-frequency amplification with parameter $k$).
- **Median filter:** nonlinear, robust to outliers, edge-preserving — superior to linear filters for salt-and-pepper noise.
- **Practical pipeline:** CLAHE → Gaussian → Canny chaining as a preprocessing strategy; `mm::drawImgKernel` for didactic visualization of the sliding window.

Chapter 4 will address **mathematical morphology** (erosion, dilation, opening, and closing), exploring in depth the `mm::ero` and `mm::dil` functions from the `morph.py` library. Next, Chapter 5 will present **frequency domain processing**, focusing on the Fourier Transform and spectral filtering techniques.

## 3.10 🤖 Using Gemini Notebook as a Complementary Tutor

In this edition, we encourage the use of **Gemini Notebook** as a complementary learning tool. This AI tool uses exclusively the documents provided by the author as its knowledge base, ensuring responses consistent with the book's content—including the functions of the `morph.py` library and the experiments conducted in this chapter.

For each chapter, we have prepared a specific project on the platform containing the chapter's PDF, the *notebooks*, and auxiliary materials. We suggest exploring in particular:

- **Study Guide:** a structured summary of the concepts, ideal for review before exams;
- **Chat:** ask questions about equalization, convolution, filters, and pipelines directly with the tutor;
- **Frequently Asked Questions:** typical questions about the difference between mean and median, USM, Laplacian vs. Sobel.

> ### ❗ 🎓 Study with the Intelligent Tutor
>
> To interact with the content of this chapter, access the following *link*. The environment contains teaching materials in different formats, generated from the chapter's **PDF**. On the platform, explore especially the **Study Guide** and **Chat** options to deepen your understanding.
>
> [🚀 ACCESS GEMINI NOTEBOOK: CHAPTER 03](https://notebooklm.google.com/notebook/d6593e26-a008-4d3b-8073-5c9b7d00eacc)
>
> #### 🌐 Language and Programming Language
>
> The project for this chapter in Gemini Notebook was built using only the text in **Portuguese** and the code examples in **Python**. If you are studying from the English or French edition, or following the C++ track, the tutor's answers may not correspond exactly to the version you are reading.
>
> #### ⚠️ Notice on AI-Generated Content
>
> AI is a powerful ally in your studies, but the generated content may contain **errors or inaccuracies**. Always consult **books, scientific articles, and other reliable academic sources** to validate the information. Whenever possible, run the practical examples provided in this chapter to verify the results.

## 3.11 Exercise List

1. **(10%)** Explain the difference between **convolution** and **cross-correlation**. For which types of *kernels* are the results identical? Give an example of an asymmetric *kernel* (such as Sobel $G_x$) and show numerically that the results differ by applying it to the 5×5 patch from the chapter in both ways.

2. **(15%)** Consider a 5×5 image with intensities concentrated between levels 3 and 5 (low contrast, 3 bits). Manually apply the equalization algorithm from [Table 3.1](#tbl-03-equalizacao), filling in all the columns of the table ($k$, $h[k]$, $p[k]$, $\text{cdf}[k]$, $\text{lut}[k]$). Verify the result with `mm::equalize`.

3. **(15%)** Using `mm::conv`, apply the mean filter with kernels of size 3×3, 9×9, and 21×21 to the mandrill image. For each version, compute the **PSNR** (*Peak Signal-to-Noise Ratio*) relative to the original:
$$\text{PSNR} = 10\log_{10}\!\left(\frac{255^2}{\text{MSE}}\right), \quad \text{MSE} = \frac{1}{MN}\sum_{i,j}(f-g)^2$$
Plot the PSNR as a function of kernel size and explain what the progressive decline indicates about the relationship between smoothing and information loss.

4. **(15%)** Using `add_salt_pepper` with a density of 5%, apply and compare: (a) `mm::conv` with a 3×3 mean filter, (b) `cv2.GaussianBlur` with $\sigma=1$, (c) `cv2.medianBlur` with a 3×3 window, and (d) `cv2.medianBlur` with a 5×5 window. Display the images with `mm::show` in a 2×4 grid (row 1: images, row 2: histograms via `mm::histImg`). Explain why the median outperforms linear filters using the argument from [Table 3.4](#tbl-03-mediana).

5. **(15%)** Implement `mm::conv0` using only vectorized NumPy operations — without Python loops and without `cv2.filter2D` — using the *stride tricks* operator (`np.lib.stride_tricks.sliding_window_view`). Compare the result and execution time with `mm::conv0` (loops) and `mm::conv` (cv2) for 3×3 and 15×15 kernels on the mandrill image.

6. **(15%)** Apply *Unsharp Masking* with $\sigma=1$ and $k \in \{0.5, 1.0, 2.0, 4.0\}$ using the `usm` function from the chapter. For each value of $k$: (a) compute the absolute difference $|g - f|$, (b) display the images and the differences with `mm::show`, and (c) plot the histogram of the differences with `mm::histImg`. Identify from which $k$ onward the artifacts (halos and noise amplification) become visually unacceptable.

7. **(15%)** Choose a publicly available X-ray or tomography image (e.g., via `mm::read` from a URL) and design a preprocessing pipeline with at least 4 sequential steps, justifying each choice based on the concepts from the chapter. Display with `mm::show` in a grid: the original image, each intermediate stage, and the final result along with their histograms (`mm::histImg`).

## Chapter References

The theoretical foundation of this chapter is based on the following works:

* Gonzalez (2018) for the concepts of intensity operations, histogram, convolution, and spatial filtering.
* Szeliski (2022) for computer vision and practical applications of filtering.
* Bradski (2008) for the practical implementation with OpenCV and `morph.py`.

------------------------------------------------------------------------


<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cpp.en/cap03/cap03.EPs_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

Figure 3.18 presents a small notebook suggestion with the possible structure to be adopted by the class. The suggestion in this section may help you in completing your digital notebook, which is part of the continuous assessment (CA) of this course — see Section 3.9.

The assessment activities proposed here may be carried out in groups of up to 3 (three) students, with the respective completion deadline.

To work on the **Google Colab** environment, you must have a Google account (Gmail). The suggestions in this notebook do not follow any pre-established standard, but are just coding examples that can assist in testing the proposed digital image processing (DIP) and computer vision (CV) tasks.

**Note:** The codes below are just examples for testing the main functions of the task.

The proposed exercise list (EPs, in Portuguese) contains the following experiments:

- **EP01** – Calculate the negative of the image `mario.png`.
- **EP02** – Swap the color channels of the image `mario.png`, transforming it from RGB to GRB.
- **EP03** – Represent the image `mario.png` in the HSV color space.
- **EP04** – Threshold the image `mario.png` in a binary fashion, using the HSV color space.
- **EP05** – Swap the color channels of the image `mario.png`, transforming it from RGB to BGR.
- **EP06** – Using the `img1.pgm` and `img2.pgm` images, present: (a) the result of `img1` AND `img2`; and (b) the result of `img1` OR `img2`.
- **EP07** – Provide the histogram and the negative of the image `pout.tif`, making the necessary adjustments so that the image appears with good visual quality.

## 3.12 💻 **Practical Part with Programming Exercises**

### 🎯 Objective of this Notebook

The notebook allows you to develop, validate, organize, and test solutions for **Programming Exercises (EPs)** in interactive environments, such as Colab, using the same test cases as Moodle, and only copy them there when recording the official grade.

#### *Download*

Download `morph.py` and `testsuite.py` by running the cell below:

In [83]:
import os, urllib.request

os.makedirs("tmp/state", exist_ok=True)  # C++ track build artifacts (.cpp, binary, PNGs)

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

# The kernel is Python even in the C++ track: `mm` (morph.py) is used by the
# simulators, by the display of the figures the C++ binary generates and by the
# mm::Image state between cells. cpp=True also downloads the compiled track
# (morph.hpp + stb_image*.h), used in the #include of the %%writefile *.cpp cells.
import config
config.setup(testsuite=True, cpp=True)
from morph import mm
from testsuite import TestSuite

✅ Environment ready. Morph: 1.1.9 | OpenCV: 5.0.0 | TestSuite: 1.1.2


#### Running the Tests
To evaluate the tests, run `TestSuite("EP03_01.extension").run()` in a new cell, replacing the extension with the one corresponding to the language used (`.py`, `.java`, `.c`, `.cpp`, `.js`, or `.r`). The system downloads the test cases from GitHub, runs the program, and calculates the grade automatically.

To test Python code directly, without saving a file, use `run_code(code)` passing the code as a *string* in a variable `code`:

```python
code = """
from morph import mm
# 3 ... your code here ...
"""
TestSuite("EP03_01").run_code(code)
```

### 3.0.1 EP03_01 ➕ Saturated Addition of a Constant

In video surveillance systems, cameras in environments with variable lighting produce underexposed images. Adjusting brightness by **saturated addition of a constant** is the simplest operation for immediate correction, being applied in real time on embedded camera *chips* and in preprocessing *pipelines* of mobile robots.

See [Figure 3.26](#fig-03-sim-ep0301-adicao) for a simulation of this EP.


#### 3.0.1.1 📋 Implementation Guidelines

1. **Dimensions:** Read the integers $L$ (rows) and $C$ (columns).
2. **Constant:** Read the integer $k$ (value to be added).
3. **Data:** Read the integer values of the original matrix row by row.
4. **Mapping:** For each pixel $p$, compute the new value using the equation:

$$p' = \text{clip}(p + k)$$

5. **Output:** Display the resulting matrix with dimensions $L \times C$.

#### 3.0.1.2 📌 Computational Constraints

* **Saturation (*Clipping*):** Values must be confined to the interval $[0, 255]$:
$$\text{clip}(x) = \max(0, \min(255, x))$$
* **Type:** The final result must be an integer (no decimal places).
* **$k$ can be negative:** negative values darken the image; positive values lighten it.

#### 3.0.1.3 🧠 Theoretical Background

| Parameter | Type | Visual Impact |
|-----------|------|----------------|
| **$k > 0$** | Integer | Lightens the image; pixels near 255 saturate to white |
| **$k < 0$** | Integer | Darkens the image; pixels near 0 saturate to black |
| **$k = 0$** | Integer | Image unchanged |

#### 3.0.1.4 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $L$.
* Line 2: Integer $C$.
* Line 3: Integer $k$.
* Following lines: Integer elements of the original matrix.

**Output:**

* Transformed matrix with $L$ rows and $C$ columns, integer values separated by spaces.

#### 3.0.1.5 📌 Examples

| Input | Output | Observation |
|---------|-------|------------|
| 2<br>3<br>50<br>0 100 200<br>210 240 255 | 50 150 250<br>255 255 255 | Saturation at 255 for high pixels |
| 1<br>4<br>-30<br>0 20 200 255 | 0 0 170 225 | Saturation at 0 for low pixels |

In [84]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0301-adicao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">➕ Simulator EP03_01: Constant Saturated Addition</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">p' = clip(p + k)</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Adjust the constant value k to observe the image brightness shift and saturation truncation in the range [0, 255].</p>

    <!-- Controle da Constante k -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;">
      <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:6px;">
        <label style="font-size:11px;font-weight:700;color:#27ae60;">Constant (k)</label>
        <span id="sim_ep0301_vl_k" style="font-family:monospace;font-size:12px;font-weight:700;color:#27ae60;">0</span>
      </div>
      <input type="range" id="sim_ep0301_sl_k" min="-128" max="128" step="1" value="0" style="width:100%;cursor:pointer;">
    </div>

    <!-- Comparativo Lado a Lado: Entrada vs Resultado -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Entrada Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Original Input (p)</span>
        <div id="sim_ep0301_grid_orig" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0301_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 New Image</button>
      </div>

      <!-- Resultado Transformado -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Transformed Result (p')</span>
        <div id="sim_ep0301_grid_new" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0301_btnReset" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">↩ Reset (k = 0)</button>
      </div>

    </div>

    <!-- Mensagem Explicativa Dinâmica -->
    <div id="sim_ep0301_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      Formula applied: <b>clip(p + (0))</b>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0301(root){
    if (!root || root.dataset.simEp0301Init) return;
    root.dataset.simEp0301Init = "1";

    var slK      = root.querySelector('#sim_ep0301_sl_k');
    var vlK      = root.querySelector('#sim_ep0301_vl_k');
    var gridOrig = root.querySelector('#sim_ep0301_grid_orig');
    var gridNew  = root.querySelector('#sim_ep0301_grid_new');
    var debugDiv = root.querySelector('#sim_ep0301_debug');

    var btnNew   = root.querySelector('#sim_ep0301_btnNew');
    var btnReset = root.querySelector('#sim_ep0301_btnReset');

    var pixels = Array(16).fill(0).map(function(){ return Math.floor(Math.random() * 256); });

    function render() {
      var k = parseInt(slK.value) || 0;
      vlK.textContent = k;
      debugDiv.innerHTML = 'Fórmula aplicada: <b>clip(p + (' + k + '))</b>';

      gridOrig.innerHTML = '';
      gridNew.innerHTML  = '';

      pixels.forEach(function(p) {
        // Célula Original
        var cellO = document.createElement('div');
        var fgColorO = p > 128 ? '#000000' : '#ffffff';
        cellO.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgColorO + ';box-sizing:border-box;';
        cellO.textContent = p;
        gridOrig.appendChild(cellO);

        // Célula Resultado
        var res = Math.max(0, Math.min(255, p + k));
        var fgColorN = res > 128 ? '#000000' : '#ffffff';
        var cellN = document.createElement('div');
        cellN.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;background:rgb(' + res + ',' + res + ',' + res + ');color:' + fgColorN + ';box-sizing:border-box;';
        cellN.textContent = res;
        gridNew.appendChild(cellN);
      });
    }

    slK.addEventListener('input', render);

    btnNew.addEventListener('click', function() {
      pixels = Array(16).fill(0).map(function(){ return Math.floor(Math.random() * 256); });
      render();
    });

    btnReset.addEventListener('click', function() {
      slK.value = 0;
      render();
    });

    render();
  }

  function tryInitSimEP0301(){
    var root = document.getElementById('sim-ep0301-adicao');
    if (root) initSimEP0301(root); else setTimeout(tryInitSimEP0301, 200);
  }
  tryInitSimEP0301();
})();
</script>
</div>
""")

**Figure 3.26:** Simulador EP03_01: Saturated Addition of Constant (p


<figure id="fig-03-sim-ep0301-adicao">
  <img src="imagens/fig-03-sim-ep0301-adicao.png" alt=" Simulador EP03_01: Saturated Addition of Constant (p' = clip(p + k)) " style="max-width:80%" />
  <figcaption><strong>Figure 3.26:</strong>  Simulador EP03_01: Saturated Addition of Constant (p' = clip(p + k)) </figcaption>
</figure>

In [85]:
%%writefile EP03_01.cpp
// your solution

Overwriting EP03_01.cpp


In [86]:
TestSuite("EP03_01.cpp").run()

### 3.0.2 EP03_02 🔀 Alpha Blending of Two Images

In nuclear medicine, images from different modalities (computed tomography and magnetic resonance imaging) are fused to aid in diagnosis. **Weighted blending** (*alpha blending*) is the fundamental operation of this process, allowing the radiologist to interactively control the weight of each modality in the displayed image.

See [Figure 3.27](#fig-03-sim-ep0302-blending) for a simulation of this EP.


#### 3.0.2.1 📋 Implementation Guidelines

1. **Dimensions:** Read the integers $L$ (rows) and $C$ (columns).
2. **Parameter:** Read the real value $\alpha \in [0, 1]$.
3. **Data:** Read the integer values of matrix $f_1$ (image 1) followed by those of matrix $f_2$ (image 2).
4. **Mapping:** For each position $(i, j)$, compute:

$$g(i,j) = \text{clip}\left(\text{round}\left(\alpha \cdot f_1(i,j) + (1-\alpha) \cdot f_2(i,j)\right)\right)$$

5. **Output:** Display the resulting $L \times C$ matrix.

#### 3.0.2.2 📌 Computational Constraints

* **Rounding:** Apply `round` before converting to integer.
* **Saturation:** Constrain to the interval $[0, 255]$ with $\text{clip}(x) = \max(0, \min(255, x))$.
* **Float operation:** Perform the operation in floating point before rounding.

#### 3.0.2.3 🧠 Theoretical Background

| Value of $\alpha$ | Result |
|:-----------------:|:----------|
| $\alpha = 1.0$ | Only $f_1$ |
| $\alpha = 0.5$ | Arithmetic mean of $f_1$ and $f_2$ |
| $\alpha = 0.0$ | Only $f_2$ |

#### 3.0.2.4 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $L$.
* Line 2: Integer $C$.
* Line 3: Real $\alpha$.
* Following lines: Elements of $f_1$ ($L$ lines with $C$ values each).
* Following lines: Elements of $f_2$ ($L$ lines with $C$ values each).

**Output:**

* Resulting $L \times C$ matrix.

#### 3.0.2.5 📌 Examples

| Input | Output | Observation |
|---------|-------|------------|
| 1<br>3<br>0.5<br>0 100 200<br>100 200 50 | 50 150 125 | Mean between the two images |
| 1<br>3<br>1.0<br>10 20 30<br>90 80 70 | 10 20 30 | Only $f_1$ (alpha=1) |

In [87]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0302-blending" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🔀 Simulator EP03_02: Alpha Blending of Two Images</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = α·f1 + (1−α)·f2</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Adjust the transparency parameter α to observe the pixel-by-pixel weighted linear combination between images f1 and f2.</p>

    <!-- Controle do Parâmetro Alpha -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;">
      <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:6px;">
        <label style="font-size:11px;font-weight:700;color:#8e44ad;">α (Alpha — Weight of f1)</label>
        <span id="sim_ep0302_vl_a" style="font-family:monospace;font-size:12px;font-weight:700;color:#8e44ad;">0.50</span>
      </div>
      <input type="range" id="sim_ep0302_sl_a" min="0" max="1" step="0.05" value="0.5" style="width:100%;cursor:pointer;">
      <div style="margin-top:6px;font-size:10px;color:#8a8371;text-align:center;font-family:monospace;">
        α = 0.00 → Only f2 &nbsp;|&nbsp; α = 0.50 → Equal Weighted Average &nbsp;|&nbsp; α = 1.00 → Only f1
      </div>
    </div>

    <!-- Comparativo em 3 Colunas: f1 vs f2 vs Resultado g -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(160px, 1fr));gap:12px;align-items:start;margin-bottom:14px;">
      
      <!-- Imagem f1 -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Image f1</span>
        <div id="sim_ep0302_grid_f1" style="display:grid;grid-template-columns:repeat(4, 38px);gap:3px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0302_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 New Images</button>
      </div>

      <!-- Imagem f2 -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#b9770e;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Image f2</span>
        <div id="sim_ep0302_grid_f2" style="display:grid;grid-template-columns:repeat(4, 38px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Resultado g -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#8e44ad;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Result g</span>
        <div id="sim_ep0302_grid_g" style="display:grid;grid-template-columns:repeat(4, 38px);gap:3px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0302_btnReset" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">↩ Reset (α = 0.5)</button>
      </div>

    </div>

    <!-- Mensagem Explicativa Dinâmica -->
    <div id="sim_ep0302_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      Formula: <b>clip(round(0.50 · f1 + 0.50 · f2))</b>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0302(root){
    if (!root || root.dataset.simEp0302Init) return;
    root.dataset.simEp0302Init = "1";

    var slA      = root.querySelector('#sim_ep0302_sl_a');
    var vlA      = root.querySelector('#sim_ep0302_vl_a');
    var gF1      = root.querySelector('#sim_ep0302_grid_f1');
    var gF2      = root.querySelector('#sim_ep0302_grid_f2');
    var gG       = root.querySelector('#sim_ep0302_grid_g');
    var debugDiv = root.querySelector('#sim_ep0302_debug');

    var btnNew   = root.querySelector('#sim_ep0302_btnNew');
    var btnReset = root.querySelector('#sim_ep0302_btnReset');

    var px1 = [], px2 = [];

    function generate() {
      px1 = Array.from({length: 16}, function(){ return Math.floor(Math.random() * 256); });
      px2 = Array.from({length: 16}, function(){ return Math.floor(Math.random() * 256); });
    }

    function createCell(val) {
      var c = document.createElement('div');
      var fgColor = val > 128 ? '#000000' : '#ffffff';
      c.style.cssText = 'width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;background:rgb(' + val + ',' + val + ',' + val + ');color:' + fgColor + ';box-sizing:border-box;';
      c.textContent = val;
      return c;
    }

    function render() {
      var a = parseFloat(slA.value) || 0;
      var a1 = a.toFixed(2);
      var a2 = (1 - a).toFixed(2);

      vlA.textContent = a1;
      debugDiv.innerHTML = 'Fórmula: <b>clip(round(' + a1 + ' · f1 + ' + a2 + ' · f2))</b>';

      gF1.innerHTML = '';
      gF2.innerHTML = '';
      gG.innerHTML  = '';

      for (var i = 0; i < 16; i++) {
        gF1.appendChild(createCell(px1[i]));
        gF2.appendChild(createCell(px2[i]));

        var res = Math.max(0, Math.min(255, Math.round(a * px1[i] + (1 - a) * px2[i])));
        gG.appendChild(createCell(res));
      }
    }

    slA.addEventListener('input', render);

    btnNew.addEventListener('click', function() {
      generate();
      render();
    });

    btnReset.addEventListener('click', function() {
      slA.value = '0.5';
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0302(){
    var root = document.getElementById('sim-ep0302-blending');
    if (root) initSimEP0302(root); else setTimeout(tryInitSimEP0302, 200);
  }
  tryInitSimEP0302();
})();
</script>
</div>
""")

**Figure 3.27:** Simulator EP03_02: Alpha Blending of Two Images (g = α·f1 + (1−α)·f2)


<figure id="fig-03-sim-ep0302-blending">
  <img src="imagens/fig-03-sim-ep0302-blending.png" alt=" Simulator EP03_02: Alpha Blending of Two Images (g = α·f1 + (1−α)·f2) " style="max-width:80%" />
  <figcaption><strong>Figure 3.27:</strong>  Simulator EP03_02: Alpha Blending of Two Images (g = α·f1 + (1−α)·f2) </figcaption>
</figure>

In [88]:
%%writefile EP03_02.cpp
// your solution

Overwriting EP03_02.cpp


In [89]:
TestSuite("EP03_02.cpp").run()

### 3.0.3 EP03_03 🎭 Image Inversion (Photographic Negative)

In radiology, X-ray images are traditionally viewed as negatives: bones appear in black on a white background. The **photographic negative** operation is routinely applied in PACS (*Picture Archiving and Communication Systems*) to facilitate the detection of fractures and bone densities.

See [Figure 3.28](#fig-03-sim-ep0303-inversao) for a simulation of this EP.

#### 3.0.3.1 📋 Implementation Guidelines

1. **Dimensions:** Read the integers $L$ (rows) and $C$ (columns).
2. **Data:** Read the integer values of the original matrix.
3. **Mapping:** For each pixel $p$, compute the negative:

$$p' = 255 - p$$

4. **Output:** Display the resulting matrix $L \times C$.

#### 3.0.3.2 📌 Computational Constraints

* **No clipping required:** The result of $255 - p$ with $p \in [0, 255]$ is always $\in [0, 255]$.
* **Integer type:** The output must consist of integer values.
* **Logical equivalence:** The operation is identical to the bitwise `NOT` (`mm::bnot`) on 8-bit images.

#### 3.0.3.3 🧠 Theoretical Foundation

| Original Pixel $p$ | Negative Pixel $p'$ | Observation |
|:------------------:|:-------------------:|:-----------:|
| 0 (black) | 255 (white) | Total inversion |
| 128 (medium gray) | 127 (medium gray) | Central value |
| 255 (white) | 0 (black) | Total inversion |

#### 3.0.3.4 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $L$.
* Line 2: Integer $C$.
* Following lines: Integer elements of the original matrix.

**Output:**

* Negative matrix with $L$ rows and $C$ columns.

#### 3.0.3.5 📌 Examples

| Input | Output | Observation |
|-------|--------|-------------|
| 1<br>4<br>0 128 200 255 | 255 127 55 0 | Inversion of each pixel |
| 2<br>2<br>10 20<br>30 40 | 245 235<br>225 215 | Inverted 2x2 matrix |

In [90]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0303-inversao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🎭 Simulator EP03_03: Photographic Negative (Inversion)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">p' = 255 − p</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Observe the complementary intensity inversion: dark tones become light and light tones become dark by subtracting each pixel from the maximum value of 255.</p>

    <!-- Comparativo Lado a Lado: Entrada vs Negativo -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Entrada Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Original Input (p)</span>
        <div id="sim_ep0303_grid_orig" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0303_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 New Image</button>
      </div>

      <!-- Negativo (p' = 255 - p) -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Negative (p' = 255 − p)</span>
        <div id="sim_ep0303_grid_neg" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <div style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid transparent;background:transparent;color:transparent;user-select:none;">&nbsp;</div>
      </div>

    </div>

    <!-- Painel de Informação Explicativo -->
    <div id="sim_ep0303_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      Formula applied: <b>p' = 255 − p</b>
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0303(root){
    if (!root || root.dataset.simEp0303Init) return;
    root.dataset.simEp0303Init = "1";

    var gO = root.querySelector('#sim_ep0303_grid_orig');
    var gN = root.querySelector('#sim_ep0303_grid_neg');
    var btnNew = root.querySelector('#sim_ep0303_btnNew');

    var pixels = [];

    function generate() {
      pixels = Array.from({length: 16}, function(){ return Math.floor(Math.random() * 256); });
    }

    function render() {
      gO.innerHTML = '';
      gN.innerHTML = '';

      pixels.forEach(function(p) {
        // Célula Original
        var cellO = document.createElement('div');
        var fgColorO = p > 128 ? '#000000' : '#ffffff';
        cellO.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgColorO + ';box-sizing:border-box;';
        cellO.textContent = p;
        gO.appendChild(cellO);

        // Célula Negativo
        var r = 255 - p;
        var fgColorN = r > 128 ? '#000000' : '#ffffff';
        var cellN = document.createElement('div');
        cellN.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;background:rgb(' + r + ',' + r + ',' + r + ');color:' + fgColorN + ';box-sizing:border-box;';
        cellN.textContent = r;
        gN.appendChild(cellN);
      });
    }

    btnNew.addEventListener('click', function() {
      generate();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0303(){
    var root = document.getElementById('sim-ep0303-inversao');
    if (root) initSimEP0303(root); else setTimeout(tryInitSimEP0303, 200);
  }
  tryInitSimEP0303();
})();
</script>
""")

**Figure 3.28:** EP03_03 Simulator: Image Inversion — Photographic Negative (p


<figure id="fig-03-sim-ep0303-inversao">
  <img src="imagens/fig-03-sim-ep0303-inversao.png" alt=" EP03_03 Simulator: Image Inversion — Photographic Negative (p' = 255 − p) " style="max-width:80%" />
  <figcaption><strong>Figure 3.28:</strong>  EP03_03 Simulator: Image Inversion — Photographic Negative (p' = 255 − p) </figcaption>
</figure>

In [91]:
%%writefile EP03_03.cpp
// your solution

Overwriting EP03_03.cpp


In [92]:
TestSuite("EP03_03.cpp").run()

### 3.0.4 EP03_04 📊 Histogram Equalization (L bits)

In remote sensing satellite images, variation in illumination throughout the day produces low-contrast images. **Histogram equalization** is automatically applied in satellites such as Landsat to redistribute tones, revealing details of vegetation, relief, and urban areas that are invisible in the original image.

See [Figure 3.29](#fig-03-sim-ep0304-equalizacao) for a simulation of this EP.

#### 3.0.4.1 📋 Implementation Guidelines

1. **Dimensions:** Read the integers $L$ (rows), $C$ (columns), and $B$ (number of bits, with $L_{\max} = 2^B$).
2. **Data:** Read the pixel matrix $f$ with values in $[0, 2^B - 1]$.
3. **Histogram:** Compute $h[k]$ = number of pixels with intensity $k$, for $k = 0 \ldots 2^B-1$.
4. **Probability:** $p[k] = h[k] / (L \cdot C)$.
5. **CDF:** $\text{cdf}[k] = \sum_{j=0}^{k} p[j]$; cumulative distribution function.
6. **LUT:** $\text{lut}[k] = \text{round}\left(\text{cdf}[k] \cdot (2^B - 1)\right)$; *Look-Up Table*.
7. **Application:** $g[i,j] = \text{lut}[f[i,j]]$.
8. **Output:** Display the equalized matrix of size $L \times C$.

#### 3.0.4.2 📌 Computational Constraints

* **Rounding:** Use mathematical rounding (`round`) in the LUT.
* **Bits:** The number of levels is $2^B$ (e.g., $B=3 \Rightarrow 8$ levels, $B=8 \Rightarrow 256$ levels).
* **Cumulative CDF:** $\text{cdf}[k] = \sum_{j=0}^{k} p[j]$, with $\text{cdf}[2^B-1] = 1.0$.

#### 3.0.4.3 🧠 Theoretical Foundation

| Step | Operation | Formula |
|:-----:|:---------|:--------|
| 1 | Histogram | $h[k] \leftarrow$ number of pixels with intensity $k$ |
| 2 | Probability | $p[k] = h[k] / (L \cdot C)$ |
| 3 | CDF | $\text{cdf}[k] = \sum_{j=0}^{k} p[j]$ |
| 4 | LUT | $\text{lut}[k] = \text{round}(\text{cdf}[k] \cdot (2^B-1))$ |
| 5 | Application | $g[i,j] = \text{lut}[f[i,j]]$ |

#### 3.0.4.4 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $L$.
* Line 2: Integer $C$.
* Line 3: Integer $B$ (number of bits).
* Following lines: Integer elements of the matrix.

**Output:**

* Equalized matrix with $L$ rows and $C$ columns.

#### 3.0.4.5 📌 Examples

| Input | Output | Observation |
|---------|-------|------------|
| 5<br>5<br>3<br>3 4 2 3 4<br>4 3 3 4 3<br>2 3 4 3 2<br>3 4 3 2 3<br>4 3 2 3 4 | 5 7 1 5 7<br>7 5 5 7 5<br>1 5 7 5 1<br>5 7 5 1 5<br>7 5 1 5 7 | Example with 3 bits from the chapter |
| 1<br>4<br>3<br>0 0 7 7 | 0 0 7 7 | Extreme bimodal histogram |

In [93]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0304-equalizacao" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">📊 Simulator EP03_04: Histogram Equalization</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">lut[k] = round(cdf[k] · (L − 1))</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Choose the bit depth (B) and generate images to analyze the dynamic spreading of the histogram and the remapping table (LUT) in real time.</p>

    <!-- Controle de Bits B -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;">
      <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:6px;">
        <label style="font-size:11px;font-weight:700;color:#16a085;">Bit Depth (B)</label>
        <span id="sim_ep0304_vl_bits" style="font-family:monospace;font-size:12px;font-weight:700;color:#16a085;">3 bits → 8 levels</span>
      </div>
      <input type="range" id="sim_ep0304_sl_bits" min="1" max="8" step="1" value="3" style="width:100%;cursor:pointer;">
      <div style="display:flex;justify-content:space-between;margin-top:6px;font-size:10px;color:#8a8371;font-family:monospace;">
        <span>1 bit (2 levels)</span>
        <span>4 bits (16 levels)</span>
        <span>8 bits (256 levels)</span>
      </div>
    </div>

    <!-- Comparativo Lado a Lado: Entrada vs Equalizada -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:16px;">
      
      <!-- Entrada Original -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#7f8c8d;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Original Input</span>
        <div id="sim_ep0304_grid_orig" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0304_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 New Image</button>
      </div>

      <!-- Resultado Equalizado -->
      <div style="background:#fafaf7;border:1px solid #16a085;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#16a085;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Equalized Result</span>
        <div id="sim_ep0304_grid_eq" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0304_btnReset" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">↩ New Sampling</button>
      </div>

    </div>

    <!-- Histogramas Lado a Lado -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(240px, 1fr));gap:16px;margin-bottom:16px;">
      
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
        <span style="font-size:10px;font-weight:700;color:#7f8c8d;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;text-align:center;">Original Histogram</span>
        <canvas id="sim_ep0304_hist_orig" style="width:100%;height:80px;display:block;" width="340" height="80"></canvas>
      </div>

      <div style="background:#fafaf7;border:1px solid #16a085;border-radius:12px;padding:12px;">
        <span style="font-size:10px;font-weight:700;color:#16a085;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:8px;text-align:center;">Equalized Histogram</span>
        <canvas id="sim_ep0304_hist_eq" style="width:100%;height:80px;display:block;" width="340" height="80"></canvas>
      </div>

    </div>

    <!-- Tabela LUT -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:10px 14px;margin-bottom:14px;overflow-x:auto;">
      <span style="font-size:10px;font-weight:700;color:#5e5a4a;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:6px;">LUT (Remapping Table k → v)</span>
      <div id="sim_ep0304_lut_table" style="font-family:monospace;font-size:11px;color:#26241d;white-space:nowrap;"></div>
    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0304_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      lut[k] = round(cdf[k] · 7) | B=3, levels=8
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0304(root){
    if (!root || root.dataset.simEp0304Init) return;
    root.dataset.simEp0304Init = "1";

    var slBits     = root.querySelector('#sim_ep0304_sl_bits');
    var vlBits     = root.querySelector('#sim_ep0304_vl_bits');
    var debugDiv   = root.querySelector('#sim_ep0304_debug');
    var gridOrig   = root.querySelector('#sim_ep0304_grid_orig');
    var gridEq     = root.querySelector('#sim_ep0304_grid_eq');
    var lutTable   = root.querySelector('#sim_ep0304_lut_table');
    var histOrig   = root.querySelector('#sim_ep0304_hist_orig');
    var histEq     = root.querySelector('#sim_ep0304_hist_eq');

    var btnNew     = root.querySelector('#sim_ep0304_btnNew');
    var btnReset   = root.querySelector('#sim_ep0304_btnReset');

    var ROWS = 4, COLS = 4;
    var pixels = [];

    function generatePixels() {
      var b = parseInt(slBits.value) || 3;
      var levels = Math.pow(2, b);
      var lo = Math.floor(levels * 0.2);
      var hi = Math.floor(levels * 0.5);
      pixels = Array.from({ length: ROWS * COLS }, function(){
        return lo + Math.floor(Math.random() * (hi - lo + 1));
      });
    }

    function computeEqualization(pixArr, b) {
      var levels = Math.pow(2, b);
      var N = pixArr.length;
      var h = new Array(levels).fill(0);
      pixArr.forEach(function(p){ h[p]++; });

      var cdf = new Array(levels).fill(0);
      cdf[0] = h[0] / N;
      for (var k = 1; k < levels; k++) {
        cdf[k] = cdf[k - 1] + h[k] / N;
      }

      var lut = cdf.map(function(c){ return Math.round(c * (levels - 1)); });
      var result = pixArr.map(function(p){ return lut[p]; });
      return { h: h, cdf: cdf, lut: lut, result: result };
    }

    function drawHistogram(canvas, counts, levels, color) {
      var ctx = canvas.getContext('2d');
      var W = canvas.width, H = canvas.height;
      ctx.clearRect(0, 0, W, H);

      var maxVal = Math.max.apply(null, counts.concat([1]));
      var barW = W / levels;

      counts.forEach(function(c, i){
        var barH = (c / maxVal) * (H - 6);
        ctx.fillStyle = color;
        ctx.fillRect(i * barW + 1, H - barH, barW - 2, barH);
      });
    }

    function toGray(val, levels) {
      return Math.round((val / (levels - 1)) * 255);
    }

    function render() {
      var b = parseInt(slBits.value) || 3;
      var levels = Math.pow(2, b);
      vlBits.textContent = b + ' bit' + (b > 1 ? 's' : '') + ' → ' + levels + ' níveis';

      var eqData = computeEqualization(pixels, b);
      var hOrig = eqData.h;
      var lut = eqData.lut;
      var result = eqData.result;

      var hEq = new Array(levels).fill(0);
      result.forEach(function(p){ hEq[p]++; });

      lutTable.innerHTML = lut.map(function(v, k){
        return '<span style="display:inline-block;margin-right:10px;color:#8a8371;">' + k + ' → <b style="color:#16a085;">' + v + '</b></span>';
      }).join('');

      debugDiv.innerHTML = '<b>lut[k] = round(cdf[k] · ' + (levels - 1) + ')</b> &nbsp;|&nbsp; B = ' + b + ', níveis = ' + levels;

      gridOrig.innerHTML = '';
      gridEq.innerHTML = '';

      pixels.forEach(function(p, i) {
        var grayO = toGray(p, levels);
        var fgO = grayO > 128 ? '#000000' : '#ffffff';
        var cellO = document.createElement('div');
        cellO.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + grayO + ',' + grayO + ',' + grayO + ');color:' + fgO + ';box-sizing:border-box;';
        cellO.textContent = p;
        gridOrig.appendChild(cellO);

        var r = result[i];
        var grayR = toGray(r, levels);
        var fgR = grayR > 128 ? '#000000' : '#ffffff';
        var cellR = document.createElement('div');
        cellR.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;transition:all 0.15s ease;background:rgb(' + grayR + ',' + grayR + ',' + grayR + ');color:' + fgR + ';box-sizing:border-box;';
        cellR.textContent = r;
        gridEq.appendChild(cellR);
      });

      drawHistogram(histOrig, hOrig, levels, '#95a5a6');
      drawHistogram(histEq, hEq, levels, '#16a085');
    }

    slBits.addEventListener('input', function(){
      generatePixels();
      render();
    });

    btnNew.addEventListener('click', function(){
      generatePixels();
      render();
    });

    btnReset.addEventListener('click', function(){
      generatePixels();
      render();
    });

    generatePixels();
    render();
  }

  function tryInitSimEP0304(){
    var root = document.getElementById('sim-ep0304-equalizacao');
    if (root) initSimEP0304(root); else setTimeout(tryInitSimEP0304, 200);
  }
  tryInitSimEP0304();
})();
</script>
</div>
""")

**Figure 3.29:** Simulator EP03_04: Histogram Equalization (Levels L = 2^B)


<figure id="fig-03-sim-ep0304-equalizacao">
  <img src="imagens/fig-03-sim-ep0304-equalizacao.png" alt=" Simulator EP03_04: Histogram Equalization (Levels L = 2^B) " style="max-width:80%" />
  <figcaption><strong>Figure 3.29:</strong>  Simulator EP03_04: Histogram Equalization (Levels L = 2^B) </figcaption>
</figure>

In [94]:
%%writefile EP03_04.cpp
// your solution

Overwriting EP03_04.cpp


In [95]:
TestSuite("EP03_04.cpp").run()

### 3.0.5 EP03_05 🔲 Binary AND Mask Application

In industrial computer vision inspection systems, it is necessary to isolate regions of interest (ROI) in part images to verify manufacturing defects. The **bitwise AND operation with a binary mask** is the fundamental mechanism for precisely cropping the inspection area, zeroing all pixels outside it.

See [Figure 3.30](#fig-03-sim-ep0305-mascara) for a simulation of this exercise.

#### 3.0.5.1 📋 Implementation Guidelines

1. **Dimensions:** Read the integers $L$ (rows) and $C$ (columns).
2. **Data:** Read the pixel matrix $f$ (values $\in [0, 255]$).
3. **Mask:** Read the binary matrix $m$ (values: only 0 or 255).
4. **Mapping:** For each pixel $(i,j)$, apply the bitwise AND:

$$
g(i,j) = f(i,j) \;\text{AND}\; m(i,j)
$$

where $255 =$ `11111111` and $0 =$ `00000000` in binary.

5. **Output:** Display the resulting $L \times C$ matrix.

#### 3.0.5.2 📌 Computational Constraints

* **AND with 255:** $p \; \text{AND} \; 255 = p$ (all bits preserved).
* **AND with 0:** $p \; \text{AND} \; 0 = 0$ (all bits zeroed).
* **Mask:** The only possible values in the mask are 0 and 255.
* **Implementation:** In Python, bitwise AND between integers uses the `&` operator.

#### 3.0.5.3 🧠 Theoretical Foundation

| Pixel $f$ | Mask $m$ | Result $f$ AND $m$ |
|:---------:|:-----------:|:---------------------:|
| any $v$ | 255 (`11111111`) | $v$ (preserved) |
| any $v$ | 0 (`00000000`) | 0 (zeroed) |

#### 3.0.5.4 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $L$.
* Line 2: Integer $C$.
* Following lines: Elements of $f$ ($L$ rows).
* Following lines: Elements of $m$ ($L$ rows with values 0 or 255).

**Output:**

* Resulting $L \times C$ matrix.

#### 3.0.5.5 📌 Examples

| Input | Output | Observation |
|---------|-------|------------|
| 2<br>3<br>100 150 200<br>50 80 120<br>255 255 0<br>0 255 255 | 100 150 0<br>0 80 120 | Mask selects region |
| 1<br>4<br>10 20 30 40<br>255 0 255 0 | 10 0 30 0 | Alternating preserved/zeroed |

In [96]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0305-mascara" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">⬛ Simulator EP03_05: Binary AND Mask</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = f AND m</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Click on the cells of the <b>Mask m</b> to toggle between pass-through (255) and blocking (0), applying the logical operation pixel by pixel.</p>

    <!-- Três Colunas Principais: Imagem f, Máscara m, Resultado g -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(200px, 1fr));gap:16px;align-items:start;margin-bottom:16px;">
      
      <!-- Imagem f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Image f (0–255)</span>
        <div id="sim_ep0305_grid_f" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0305_btnNew" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 New Image</button>
      </div>

      <!-- Máscara m -->
      <div style="background:#fafaf7;border:2px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Mask m (Click to Toggle)</span>
        <div id="sim_ep0305_grid_mask" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <button id="sim_ep0305_btnReset" style="margin-top:12px;padding:6px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:9px;white-space:nowrap;transition:all 0.15s ease;">↺ Reset Mask</button>
      </div>

      <!-- Resultado g -->
      <div style="background:#fafaf7;border:2px solid #27ae60;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:10px;">Result g = f AND m</span>
        <div id="sim_ep0305_grid_result" style="display:grid;grid-template-columns:repeat(4, 42px);gap:4px;justify-content:center;user-select:none;"></div>
        <div style="margin-top:12px;height:26px;display:flex;align-items:center;justify-content:center;">
          <span id="sim_ep0305_pct" style="font-size:11px;font-weight:700;color:#26241d;font-family:monospace;">—</span>
        </div>
      </div>

    </div>

    <!-- Estatísticas de Preservação -->
    <div style="display:grid;grid-template-columns:repeat(3, minmax(0, 1fr));gap:10px;margin-bottom:14px;text-align:center;">
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:8px;padding:8px 10px;font-size:11px;color:#8a8371;">
        <b id="sim_ep0305_stat_preserved" style="font-size:14px;display:block;color:#26241d;font-family:monospace;">—</b>preserved
      </div>
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:8px;padding:8px 10px;font-size:11px;color:#8a8371;">
        <b id="sim_ep0305_stat_zeroed" style="font-size:14px;display:block;color:#26241d;font-family:monospace;">—</b>zeroed
      </div>
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:8px;padding:8px 10px;font-size:11px;color:#8a8371;">
        <b id="sim_ep0305_stat_ratio" style="font-size:14px;display:block;color:#26241d;font-family:monospace;">—</b>visible
      </div>
    </div>

    <!-- Legenda e Debug -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 14px;display:flex;gap:16px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:12px;">
      <span style="font-size:11px;font-weight:700;color:#5e5a4a;">Legend:</span>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:24px;height:24px;background:#ebf4fd;border:1.5px solid #2980b9;border-radius:4px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:700;color:#2980b9;">255</div>
        <span style="font-size:10.5px;color:#5e5a4a;">Pass-through (preserved)</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:24px;height:24px;background:#26241d;border:1.5px solid #8a8371;border-radius:4px;display:flex;align-items:center;justify-content:center;font-size:9px;font-weight:700;color:#7ee7c6;">0</div>
        <span style="font-size:10.5px;color:#5e5a4a;">Blocking (zeroed)</span>
      </div>
    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0305_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:8px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;">
      g(i,j) = f(i,j) &amp; m(i,j)
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0305(root){
    if (!root || root.dataset.simEp0305Init) return;
    root.dataset.simEp0305Init = "1";

    var gridF      = root.querySelector('#sim_ep0305_grid_f');
    var gridMask   = root.querySelector('#sim_ep0305_grid_mask');
    var gridResult = root.querySelector('#sim_ep0305_grid_result');
    var debugDiv   = root.querySelector('#sim_ep0305_debug');
    var pctSpan    = root.querySelector('#sim_ep0305_pct');
    var statPres   = root.querySelector('#sim_ep0305_stat_preserved');
    var statZero   = root.querySelector('#sim_ep0305_stat_zeroed');
    var statRatio  = root.querySelector('#sim_ep0305_stat_ratio');

    var btnNew     = root.querySelector('#sim_ep0305_btnNew');
    var btnReset   = root.querySelector('#sim_ep0305_btnReset');

    var N = 16;
    var pixels = [];
    var mask = [];

    function generatePixels() {
      pixels = Array.from({ length: N }, function(){ return Math.floor(Math.random() * 256); });
    }

    function resetMask() {
      mask = Array.from({ length: N }, function(_, i) {
        var r = Math.floor(i / 4), c = i % 4;
        return (r + c) % 2 === 0 ? 255 : 0;
      });
    }

    function render() {
      gridF.innerHTML = '';
      gridMask.innerHTML = '';
      gridResult.innerHTML = '';

      var preserved = 0, zeroed = 0;

      for (var i = 0; i < N; i++) {
        var p = pixels[i];
        var m = mask[i];
        var res = p & m;

        // Célula F
        var cellF = document.createElement('div');
        var fgF = p > 128 ? '#000000' : '#ffffff';
        cellF.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + fgF + ';box-sizing:border-box;';
        cellF.textContent = p;
        gridF.appendChild(cellF);

        // Célula Máscara M (interativa)
        var cellM = document.createElement('div');
        cellM.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;cursor:pointer;user-select:none;box-sizing:border-box;transition:all 0.15s ease;';
        if (m === 255) {
          cellM.style.background = '#ebf4fd';
          cellM.style.border = '2px solid #2980b9';
          cellM.style.color = '#2980b9';
        } else {
          cellM.style.background = '#26241d';
          cellM.style.border = '2px solid #8a8371';
          cellM.style.color = '#7ee7c6';
        }
        cellM.textContent = m;
        cellM.title = m === 255 ? 'Clique para bloquear (0)' : 'Clique para passar (255)';

        (function(idx){
          cellM.addEventListener('click', function(){
            mask[idx] = mask[idx] === 255 ? 0 : 255;
            render();
          });
        })(i);

        gridMask.appendChild(cellM);

        // Célula Resultado G
        var cellR = document.createElement('div');
        var fgR = res > 128 ? '#000000' : '#ffffff';
        var borderStyle = m === 255 ? '2px solid #27ae60' : '1px solid #e4dcc8';
        cellR.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;user-select:none;background:rgb(' + res + ',' + res + ',' + res + ');color:' + fgR + ';border:' + borderStyle + ';box-sizing:border-box;';
        cellR.textContent = res;
        gridResult.appendChild(cellR);

        if (m === 255) preserved++; else zeroed++;
      }

      var ratioPct = Math.round((preserved / N) * 100);
      statPres.textContent = preserved;
      statZero.textContent = zeroed;
      statRatio.textContent = ratioPct + '%';
      pctSpan.textContent = ratioPct + '% visível';
      debugDiv.innerHTML = 'g(i,j) = f(i,j) &amp; m(i,j) &nbsp;|&nbsp; <b>' + preserved + '</b> preservados · <b>' + zeroed + '</b> zerados';
    }

    btnNew.addEventListener('click', function(){
      generatePixels();
      render();
    });

    btnReset.addEventListener('click', function(){
      resetMask();
      render();
    });

    generatePixels();
    resetMask();
    render();
  }

  function tryInitSimEP0305(){
    var root = document.getElementById('sim-ep0305-mascara');
    if (root) initSimEP0305(root); else setTimeout(tryInitSimEP0305, 200);
  }
  tryInitSimEP0305();
})();
</script>
</div>
""")

**Figure 3.30:** EP03_05 Simulator: Binary AND Mask Application


<figure id="fig-03-sim-ep0305-mascara">
  <img src="imagens/fig-03-sim-ep0305-mascara.png" alt=" EP03_05 Simulator: Binary AND Mask Application " style="max-width:80%" />
  <figcaption><strong>Figure 3.30:</strong>  EP03_05 Simulator: Binary AND Mask Application </figcaption>
</figure>

In [97]:
%%writefile EP03_05.cpp
// your solution

Overwriting EP03_05.cpp


In [98]:
TestSuite("EP03_05.cpp").run()

### 3.0.6 EP03_06 🌫️ N×N Mean Filter *Kernel*

In autonomous vehicle cameras, images captured under rain or fog exhibit Gaussian noise. The **mean filter** is widely used for real-time noise reduction, being implemented directly in the **ISP** (*Image Signal Processor*) of **CMOS** (*Complementary Metal-Oxide-Semiconductor*) sensors.

CMOS sensors are the image sensors used in most modern cameras (*smartphones*, *webcams*, automotive cameras, etc.). They convert light into electrical signals, and the ISP processes these signals in real time — applying operations such as noise reduction, white balance, and other image adjustments.

See the simulation of this EP in [Figure 3.31](#fig-03-sim-ep0306-media).

#### 3.0.6.1 📋 Implementation Guidelines

1. **Dimensions:** Read the integers $L$ (rows), $C$ (columns), and $N$ (kernel size, always odd).
2. **Data:** Read the pixel matrix $f$.
3. **Mean Filter:** For each **internal** pixel $(i,j)$ (without borders), compute:

$$g(i,j) = \text{round}\left(\frac{1}{N^2} \sum_{s=-(r)}^{r} \sum_{t=-(r)}^{r} f(i+s,\, j+t)\right), \quad r = \lfloor N/2 \rfloor$$

4. **Border Handling:** Pixels at the border (where the $N \times N$ window exceeds the limits) must be **copied directly** from the original without modification.
5. **Output:** Display the resulting $L \times C$ matrix.

#### 3.0.6.2 📌 Computational Constraints

* **Radius:** $r = \lfloor N/2 \rfloor$ (half of the kernel, integer).
* **Internal pixels:** $(i,j)$ with $r \le i < L-r$ and $r \le j < C-r$.
* **Rounding:** Use mathematical rounding before converting to integer.
* **No clipping:** The average of values $\in [0,255]$ remains in $[0,255]$.

#### 3.0.6.3 🧠 Theoretical Background

| Size $N$ | Coefficient | Pixels in the window | Effect |
|:-----------:|:-----------:|:----------------:|:------:|
| 3 | $1/9 \approx 0.111$ | 9 | Smooth |
| 5 | $1/25 = 0.04$ | 25 | Medium |
| 7 | $1/49 \approx 0.020$ | 49 | Strong |

#### 3.0.6.4 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $L$.
* Line 2: Integer $C$.
* Line 3: Integer $N$ (odd, $N \ge 3$).
* Following lines: Elements of the original matrix.

**Output:**

* Filtered $L \times C$ matrix.

#### 3.0.6.5 📌 Examples
| Input | Output | Observation |
|---------|-------|------------|
| 3<br>3<br>3<br>10 20 30<br>40 50 60<br>70 80 90 | 10 20 30<br>40 50 60<br>70 80 90 | Only border (3×3 = all border) |
| 5<br>5<br>3<br>0 0 0 0 0<br>0 0 0 0 0<br>0 0 100 0 0<br>0 0 0 0 0<br>0 0 0 0 0 | 0 0 0 0 0<br>0 11 11 11 0<br>0 11 11 11 0<br>0 11 11 11 0<br>0 0 0 0 0 | Isolated pixel: all 9 internal pixels whose 3×3 window includes the value 100 receive round(100/9)=11 |

In [99]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0306-media" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🔲 Simulator EP03_06: Mean Filter with N×N Kernel</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = Mean(Neighbors)</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Select the kernel size and hover over the result pixels to inspect the neighborhood and the arithmetic mean calculation.</p>

    <!-- Barra de Controles / Seleção de Kernel -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;display:flex;align-items:center;gap:10px;flex-wrap:wrap;justify-content:space-between;">
      <div style="display:flex;align-items:center;gap:8px;flex-wrap:wrap;">
        <span style="font-size:11px;font-weight:700;color:#26241d;">Kernel size:</span>
        <button id="sim_ep0306_btn_k3" style="padding:5px 12px;font-size:11px;font-weight:700;border:1px solid #2980b9;background:#ebf4fd;color:#2980b9;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">3 × 3 (9 neighbors)</button>
        <button id="sim_ep0306_btn_k5" style="padding:5px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">5 × 5 (25 neighbors)</button>
      </div>
      <button id="sim_ep0306_btnNew" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 New Image</button>
    </div>

    <!-- Comparativo Lado a Lado: Original vs Resultado -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Imagem Original (7x7) -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Original Image f (7×7)</span>
        <span style="font-size:10px;color:#8a8371;display:block;margin-bottom:10px;">With salt-and-pepper noise</span>
        <div id="sim_ep0306_grid_f" style="display:grid;gap:4px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Resultado Suavizado -->
      <div style="background:#fafaf7;border:2px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Result g (Smoothed Filter)</span>
        <span style="font-size:10px;color:#2980b9;display:block;margin-bottom:10px;">Hover to inspect</span>
        <div id="sim_ep0306_grid_result" style="display:grid;gap:4px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Legenda -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 14px;display:flex;gap:16px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:12px;">
      <span style="font-size:11px;font-weight:700;color:#5e5a4a;">Legend:</span>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#fef5e7;border:1.5px dashed #b9770e;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Kernel Window</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#eafaf1;border:1.5px solid #27ae60;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Border (Copied)</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#ebf4fd;border:1.5px solid #2980b9;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Inspected Pixel</span>
      </div>
    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0306_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:10px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;min-height:36px;line-height:1.5;">
      Hover over an inner pixel of the result to see the mean calculation.
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0306(root){
    if (!root || root.dataset.simEp0306Init) return;
    root.dataset.simEp0306Init = "1";

    var ROWS = 7, COLS = 7, N = 49;
    var pixels = [], result = [], kSize = 3, radius = 1;

    var gridF   = root.querySelector('#sim_ep0306_grid_f');
    var gridRes = root.querySelector('#sim_ep0306_grid_result');
    var debug   = root.querySelector('#sim_ep0306_debug');
    var btnK3   = root.querySelector('#sim_ep0306_btn_k3');
    var btnK5   = root.querySelector('#sim_ep0306_btn_k5');
    var btnNew  = root.querySelector('#sim_ep0306_btnNew');

    function generatePixels() {
      pixels = Array.from({ length: N }, function(){
        var v = 60 + Math.floor(Math.random() * 60);
        if (Math.random() > 0.82) v = Math.random() > 0.5 ? 255 : 0;
        return v;
      });
    }

    function calculateFilter() {
      result = pixels.slice();
      radius = Math.floor(kSize / 2);
      for (var r = 0; r < ROWS; r++) {
        for (var c = 0; c < COLS; c++) {
          if (r >= radius && r < ROWS - radius && c >= radius && c < COLS - radius) {
            var sum = 0, cnt = 0;
            for (var s = -radius; s <= radius; s++) {
              for (var t = -radius; t <= radius; t++) {
                sum += pixels[(r + s) * COLS + (c + t)];
                cnt++;
              }
            }
            result[r * COLS + c] = Math.round(sum / cnt);
          }
        }
      }
    }

    function textColor(g){ return g > 140 ? '#000000' : '#ffffff'; }

    function highlightKernel(tr, tc, on) {
      var cells = gridF.children;
      for (var r = 0; r < ROWS; r++) {
        for (var c = 0; c < COLS; c++) {
          var idx = r * COLS + c;
          if (!cells[idx]) continue;
          var p = pixels[idx];
          if (on && Math.abs(r - tr) <= radius && Math.abs(c - tc) <= radius) {
            cells[idx].style.background = '#fef5e7';
            cells[idx].style.color = '#b9770e';
            cells[idx].style.boxShadow = '0 0 0 2px #b9770e inset';
          } else {
            cells[idx].style.background = 'rgb(' + p + ',' + p + ',' + p + ')';
            cells[idx].style.color = textColor(p);
            cells[idx].style.boxShadow = 'none';
          }
        }
      }
    }

    function render() {
      var cols = 'repeat(' + COLS + ', 42px)';
      gridF.style.gridTemplateColumns = cols;
      gridRes.style.gridTemplateColumns = cols;
      gridF.innerHTML = '';
      gridRes.innerHTML = '';

      for (var i = 0; i < N; i++) {
        var r = Math.floor(i / COLS), c = i % COLS;
        var p = pixels[i], res = result[i];
        var isBorder = r < radius || r >= ROWS - radius || c < radius || c >= COLS - radius;

        // Célula Original f
        var cf = document.createElement('div');
        cf.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + textColor(p) + ';box-sizing:border-box;';
        cf.textContent = p;
        gridF.appendChild(cf);

        // Célula Resultado g
        var cr = document.createElement('div');
        cr.style.cssText = 'width:42px;height:42px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;font-family:monospace;border-radius:6px;user-select:none;box-sizing:border-box;transition:all 0.15s ease;';

        if (isBorder) {
          cr.style.background = '#eafaf1';
          cr.style.border = '2px solid #27ae60';
          cr.style.color = '#27ae60';
          cr.textContent = res;

          (function(row, col, val){
            cr.addEventListener('mouseenter', function(){
              debug.innerHTML = 'Pixel (' + row + ',' + col + ') é <b>borda</b>: valor herdado do original sem cálculo &nbsp;→&nbsp; <b>' + val + '</b>';
            });
            cr.addEventListener('mouseleave', function(){ resetDebug(); });
          })(r, c, res);
        } else {
          cr.style.background = 'rgb(' + res + ',' + res + ',' + res + ')';
          cr.style.color = textColor(res);
          cr.style.border = '1px solid #e4dcc8';
          cr.style.cursor = 'pointer';
          cr.textContent = res;

          (function(row, col, val){
            cr.addEventListener('mouseenter', function(){
              highlightKernel(row, col, true);
              cr.style.transform = 'scale(1.12)';
              cr.style.boxShadow = '0 0 0 2px #2980b9 inset';
              cr.style.background = '#ebf4fd';
              cr.style.color = '#2980b9';

              var neighbors = [];
              for (var s = -radius; s <= radius; s++) {
                for (var t = -radius; t <= radius; t++) {
                  neighbors.push(pixels[(row + s) * COLS + (col + t)]);
                }
              }
              var kTotal = kSize * kSize;
              debug.innerHTML = 'Pixel (' + row + ',' + col + '): round( (' + neighbors.join(' + ') + ') / ' + kTotal + ' ) &nbsp;=&nbsp; <b>' + val + '</b>';
            });

            cr.addEventListener('mouseleave', function(){
              highlightKernel(row, col, false);
              cr.style.transform = 'scale(1)';
              cr.style.boxShadow = 'none';
              cr.style.background = 'rgb(' + val + ',' + val + ',' + val + ')';
              cr.style.color = textColor(val);
              resetDebug();
            });
          })(r, c, res);
        }
        gridRes.appendChild(cr);
      }
    }

    function resetDebug() {
      debug.textContent = 'Passe o mouse sobre um pixel interno do resultado para ver o cálculo da média.';
    }

    function setKernel(k) {
      kSize = k;
      if (k === 3) {
        btnK3.style.background = '#ebf4fd';
        btnK3.style.borderColor = '#2980b9';
        btnK3.style.color = '#2980b9';
        btnK3.style.fontWeight = '700';

        btnK5.style.background = '#f1ead7';
        btnK5.style.borderColor = '#e4dcc8';
        btnK5.style.color = '#5e5a4a';
        btnK5.style.fontWeight = '600';
      } else {
        btnK5.style.background = '#ebf4fd';
        btnK5.style.borderColor = '#2980b9';
        btnK5.style.color = '#2980b9';
        btnK5.style.fontWeight = '700';

        btnK3.style.background = '#f1ead7';
        btnK3.style.borderColor = '#e4dcc8';
        btnK3.style.color = '#5e5a4a';
        btnK3.style.fontWeight = '600';
      }
      calculateFilter();
      render();
    }

    btnK3.addEventListener('click', function(){ setKernel(3); });
    btnK5.addEventListener('click', function(){ setKernel(5); });

    btnNew.addEventListener('click', function(){
      generatePixels();
      calculateFilter();
      render();
    });

    generatePixels();
    calculateFilter();
    render();
  }

  function tryInitSimEP0306(){
    var root = document.getElementById('sim-ep0306-media');
    if (root) initSimEP0306(root); else setTimeout(tryInitSimEP0306, 200);
  }
  tryInitSimEP0306();
})();
</script>
</div>
""")

**Figure 3.31:** Simulator EP03_06: Average Filter with N×N Kernel


<figure id="fig-03-sim-ep0306-media">
  <img src="imagens/fig-03-sim-ep0306-media.png" alt=" Simulator EP03_06: Average Filter with N×N Kernel " style="max-width:80%" />
  <figcaption><strong>Figure 3.31:</strong>  Simulator EP03_06: Average Filter with N×N Kernel </figcaption>
</figure>

In [100]:
%%writefile EP03_06.cpp
// your solution

Overwriting EP03_06.cpp


In [101]:
TestSuite("EP03_06.cpp").run()

### 3.0.7 EP03_07 🔍 Laplacian Operator (w4) for Edge Enhancement

In high-resolution tomography, the sharpness of edges between tissues is critical for diagnosis. The **Laplacian operator** is widely used in medical image preprocessing *pipelines* to automatically enhance anatomical contours before segmentation, avoiding manual intervention by the radiologist.

See [Figure 3.32](#fig-03-sim-ep0307-laplaciano) for a simulation of this EP.


#### 3.0.7.1 📋 Implementation Guidelines

1. **Dimensions:** Read the integers $L$ (rows) and $C$ (columns).
2. **Data:** Read the pixel matrix $f$.
3. **Laplacian (w4):** For each **internal** pixel $(i,j)$ with $1 \le i < L-1$, $1 \le j < C-1$, compute:

$$\nabla^2 f(i,j) = f(i-1,j) + f(i+1,j) + f(i,j-1) + f(i,j+1) - 4 \cdot f(i,j)$$

4. **Enhancement:** Compute the enhanced image:

$$g(i,j) = \text{clip}(f(i,j) - \nabla^2 f(i,j))$$

5. **Border:** Border pixels are copied directly: $g(i,j) = f(i,j)$.
6. **Output:** Display the enhanced matrix $L \times C$.

#### 3.0.7.2 📌 Computational Constraints

* ***Kernel* w4:** $\begin{bmatrix} 0 & 1 & 0 \\ 1 & -4 & 1 \\ 0 & 1 & 0 \end{bmatrix}$ — only 4-neighbors.
* **Saturation:** $\text{clip}(x) = \max(0, \min(255, x))$ applied to the enhancement result.
* **No rounding:** The Laplacian uses only integer additions/subtractions.

#### 3.0.7.3 🧠 Theoretical Background

| Region | $\nabla^2 f$ | Enhancement Effect |
|:------:|:------------:|:------------------:|
| Uniform | $\approx 0$ | No change |
| Rising edge | $< 0$ | Pixel lightened |
| Falling edge | $> 0$ | Pixel darkened |

#### 3.0.7.4 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $L$.
* Line 2: Integer $C$.
* Following lines: Elements of the original matrix.

**Output:**

* Enhanced matrix $L \times C$.

#### 3.0.7.5 📌 Examples

| Input | Output | Observation |
|-------|--------|-------------|
| 3<br>3<br>0 0 0<br>0 100 0<br>0 0 0 | 0 0 0<br>0 255 0<br>0 0 0 | Isolated peak: lap=−400, g=100−(−400)=500 → clip=255 |
| 3<br>3<br>50 50 50<br>50 50 50<br>50 50 50 | 50 50 50<br>50 50 50<br>50 50 50 | Uniform region: Laplacian=0, no change |

In [102]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0307-laplaciano" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">📐 Simulator EP03_07: Laplacian Operator (w4)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = f ∓ ∇²f</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Select the enhancement variant and hover over the inner pixels of the result to inspect the 4-point neighborhood and the Laplacian equation.</p>

    <!-- Barra de Controles / Seleção de Variante -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;display:flex;align-items:center;gap:10px;flex-wrap:wrap;justify-content:space-between;">
      <div style="display:flex;align-items:center;gap:8px;flex-wrap:wrap;">
        <span style="font-size:11px;font-weight:700;color:#26241d;">Variant:</span>
        <button id="sim_ep0307_btn_v1" style="padding:5px 12px;font-size:11px;font-weight:700;border:1px solid #b9770e;background:#fef5e7;color:#b9770e;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">g = f − ∇²f (Standard Enhancement)</button>
        <button id="sim_ep0307_btn_v2" style="padding:5px 12px;font-size:11px;font-weight:600;border:1px solid #e4dcc8;background:#f1ead7;color:#5e5a4a;cursor:pointer;border-radius:8px;transition:all 0.15s ease;">g = f + ∇²f (Inverts Sign)</button>
      </div>
      <button id="sim_ep0307_btnNew" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 New Step</button>
    </div>

    <!-- Grid Principal de Comparação (2 colunas + setas) -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Imagem Original f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">① Original Image f</span>
        <span style="font-size:10px;color:#8a8371;display:block;margin-bottom:10px;">Step with light noise</span>
        <div id="sim_ep0307_grid_f" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Laplaciano ∇²f -->
      <div style="background:#fafaf7;border:2px solid #b9770e;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#b9770e;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">② Laplacian ∇²f</span>
        <span style="font-size:10px;color:#b9770e;display:block;margin-bottom:10px;">Detected edges (±128 shift)</span>
        <div id="sim_ep0307_grid_l" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Linha de Resultado g e Kernel w4 -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Resultado g -->
      <div style="background:#fafaf7;border:2px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;" id="sim_ep0307_flabel">③ Result g = f − ∇²f</span>
        <span style="font-size:10px;color:#2980b9;display:block;margin-bottom:10px;">Hover to inspect</span>
        <div id="sim_ep0307_grid_res" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Estrutura do Kernel w4 -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#5e5a4a;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:6px;font-family:monospace;">Kernel w4 (4-Neighbor)</span>
        <div style="display:inline-grid;grid-template-columns:repeat(3, 30px);gap:2px;margin-bottom:6px;justify-content:center;">
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;border:1px solid #e4dcc8;border-radius:4px;color:#8a8371;">0</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fef5e7;border:1px solid #f8c471;border-radius:4px;color:#b9770e;">+1</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;border:1px solid #e4dcc8;border-radius:4px;color:#8a8371;">0</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fef5e7;border:1px solid #f8c471;border-radius:4px;color:#b9770e;">+1</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#b9770e;border:1px solid #b9770e;border-radius:4px;color:#ffffff;">−4</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fef5e7;border:1px solid #f8c471;border-radius:4px;color:#b9770e;">+1</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;border:1px solid #e4dcc8;border-radius:4px;color:#8a8371;">0</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#fef5e7;border:1px solid #f8c471;border-radius:4px;color:#b9770e;">+1</div>
          <div style="width:30px;height:30px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;border:1px solid #e4dcc8;border-radius:4px;color:#8a8371;">0</div>
        </div>
        <span style="font-size:10px;color:#8a8371;display:block;font-family:monospace;">∇²f = T + B + L + R − 4·f</span>
      </div>

    </div>

    <!-- Legenda -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 14px;display:flex;gap:16px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:12px;">
      <span style="font-size:11px;font-weight:700;color:#5e5a4a;">Legend:</span>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#fef5e7;border:1.5px dashed #b9770e;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Kernel 4-Neighbors</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#faece7;border:1.5px solid #c0392b;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Central Pixel</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#eafaf1;border:1.5px solid #27ae60;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Border (Copied)</span>
      </div>
    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0307_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:10px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;min-height:36px;line-height:1.5;">
      Hover over an inner pixel of the result to detail the equation.
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0307(root){
    if (!root || root.dataset.simEp0307Init) return;
    root.dataset.simEp0307Init = "1";

    var ROWS = 5, COLS = 5, N = 25;
    var pixels = [], lapV = [], lapA = [], result = [];
    var variant = 'subtract';

    var gridF   = root.querySelector('#sim_ep0307_grid_f');
    var gridL   = root.querySelector('#sim_ep0307_grid_l');
    var gridRes = root.querySelector('#sim_ep0307_grid_res');
    var debug   = root.querySelector('#sim_ep0307_debug');
    var fLabel  = root.querySelector('#sim_ep0307_flabel');
    var btnV1   = root.querySelector('#sim_ep0307_btn_v1');
    var btnV2   = root.querySelector('#sim_ep0307_btn_v2');
    var btnNew  = root.querySelector('#sim_ep0307_btnNew');

    function generate() {
      var sc = 2 + Math.floor(Math.random() * 2);
      var dark = 40 + Math.floor(Math.random() * 30);
      var light = 160 + Math.floor(Math.random() * 40);
      pixels = Array.from({ length: N }, function(_, i) {
        var c = i % COLS;
        var v = c < sc ? dark : light;
        v += Math.floor(Math.random() * 14) - 7;
        return Math.max(0, Math.min(255, v));
      });
      calculate();
    }

    function calculate() {
      lapV = new Array(N).fill(0);
      lapA = new Array(N).fill(0);
      result = pixels.slice();

      for (var r = 0; r < ROWS; r++) {
        for (var c = 0; c < COLS; c++) {
          var i = r * COLS + c;
          if (r >= 1 && r < ROWS - 1 && c >= 1 && c < COLS - 1) {
            var t  = pixels[(r - 1) * COLS + c];
            var b  = pixels[(r + 1) * COLS + c];
            var l  = pixels[r * COLS + (c - 1)];
            var ri = pixels[r * COLS + (c + 1)];
            var f  = pixels[i];
            var lap = t + b + l + ri - 4 * f;

            lapV[i] = lap;
            lapA[i] = Math.max(0, Math.min(255, lap + 128));
            result[i] = Math.max(0, Math.min(255, variant === 'subtract' ? f - lap : f + lap));
          }
        }
      }
    }

    function textColor(g){ return g > 140 ? '#000000' : '#ffffff'; }

    function highlightCross(tr, tc, on) {
      var cells = gridF.children;
      for (var i = 0; i < N; i++) {
        if (!cells[i]) continue;
        var p = pixels[i];
        cells[i].style.background = 'rgb(' + p + ',' + p + ',' + p + ')';
        cells[i].style.color = textColor(p);
        cells[i].style.boxShadow = 'none';
      }
      if (on) {
        var ci = tr * COLS + tc;
        if (cells[ci]) {
          cells[ci].style.background = '#faece7';
          cells[ci].style.color = '#c0392b';
          cells[ci].style.boxShadow = '0 0 0 2px #c0392b inset';
        }
        var neighbors = [[tr - 1, tc], [tr + 1, tc], [tr, tc - 1], [tr, tc + 1]];
        neighbors.forEach(function(n) {
          var nr = n[0], nc = n[1];
          if (nr >= 0 && nr < ROWS && nc >= 0 && nc < COLS) {
            var ni = nr * COLS + nc;
            if (cells[ni]) {
              cells[ni].style.background = '#fef5e7';
              cells[ni].style.color = '#b9770e';
              cells[ni].style.boxShadow = '0 0 0 2px #b9770e inset';
            }
          }
        });
      }
    }

    function resetDebug() {
      debug.textContent = 'Passe o mouse sobre um pixel interno do resultado para detalhar a equação.';
    }

    function render() {
      gridF.innerHTML = '';
      gridL.innerHTML = '';
      gridRes.innerHTML = '';
      fLabel.textContent = variant === 'subtract' ? '③ Resultado g = f − ∇²f' : '③ Resultado g = f + ∇²f';

      for (var i = 0; i < N; i++) {
        var r = Math.floor(i / COLS), c = i % COLS;
        var p = pixels[i], lv = lapV[i], la = lapA[i], res = result[i];
        var isBorder = (r === 0 || r === ROWS - 1 || c === 0 || c === COLS - 1);

        // Célula Original f
        var cf = document.createElement('div');
        cf.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + textColor(p) + ';box-sizing:border-box;';
        cf.textContent = p;
        gridF.appendChild(cf);

        // Célula Laplaciano l
        var cl = document.createElement('div');
        cl.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;';
        if (isBorder) {
          cl.style.background = '#fafaf7';
          cl.style.color = '#8a8371';
          cl.style.border = '1px solid #e4dcc8';
          cl.textContent = '—';
        } else {
          cl.style.background = 'rgb(' + la + ',' + la + ',' + la + ')';
          cl.style.color = textColor(la);
          cl.style.border = '1px solid #e4dcc8';
          cl.textContent = lv;
        }
        gridL.appendChild(cl);

        // Célula Resultado g
        var cr = document.createElement('div');
        cr.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;transition:all 0.15s ease;';

        if (isBorder) {
          cr.style.background = '#eafaf1';
          cr.style.border = '1.5px solid #27ae60';
          cr.style.color = '#27ae60';
          cr.textContent = res;

          (function(row, col, val){
            cr.addEventListener('mouseenter', function(){
              debug.innerHTML = 'Pixel borda (' + row + ',' + col + '): valor herdado sem cálculo &nbsp;→&nbsp; <b>' + val + '</b>';
            });
            cr.addEventListener('mouseleave', function(){ resetDebug(); });
          })(r, c, res);
        } else {
          cr.style.background = 'rgb(' + res + ',' + res + ',' + res + ')';
          cr.style.color = textColor(res);
          cr.style.border = '1px solid #e4dcc8';
          cr.style.cursor = 'pointer';
          cr.textContent = res;

          (function(row, col, val, lapVal, origVal){
            var tVal = pixels[(row - 1) * COLS + col];
            var bVal = pixels[(row + 1) * COLS + col];
            var lVal = pixels[row * COLS + (col - 1)];
            var rVal = pixels[row * COLS + (col + 1)];

            cr.addEventListener('mouseenter', function(){
              highlightCross(row, col, true);
              cr.style.transform = 'scale(1.12)';
              cr.style.boxShadow = '0 0 0 2px #2980b9 inset';
              cr.style.background = '#ebf4fd';
              cr.style.color = '#2980b9';

              var sign = variant === 'subtract' ? '−' : '+';
              debug.innerHTML = '∇²f = (' + tVal + ' + ' + bVal + ' + ' + lVal + ' + ' + rVal + ') − 4·' + origVal + ' = <b>' + lapVal + '</b> &nbsp;|&nbsp; g = clip(' + origVal + ' ' + sign + ' ' + lapVal + ') = <b>' + val + '</b>';
            });

            cr.addEventListener('mouseleave', function(){
              highlightCross(row, col, false);
              cr.style.transform = 'scale(1)';
              cr.style.boxShadow = 'none';
              cr.style.background = 'rgb(' + val + ',' + val + ',' + val + ')';
              cr.style.color = textColor(val);
              resetDebug();
            });
          })(r, c, res, lv, p);
        }
        gridRes.appendChild(cr);
      }
    }

    function setVariant(v) {
      variant = v;
      if (v === 'subtract') {
        btnV1.style.background = '#fef5e7';
        btnV1.style.borderColor = '#b9770e';
        btnV1.style.color = '#b9770e';
        btnV1.style.fontWeight = '700';

        btnV2.style.background = '#f1ead7';
        btnV2.style.borderColor = '#e4dcc8';
        btnV2.style.color = '#5e5a4a';
        btnV2.style.fontWeight = '600';
      } else {
        btnV2.style.background = '#fef5e7';
        btnV2.style.borderColor = '#b9770e';
        btnV2.style.color = '#b9770e';
        btnV2.style.fontWeight = '700';

        btnV1.style.background = '#f1ead7';
        btnV1.style.borderColor = '#e4dcc8';
        btnV1.style.color = '#5e5a4a';
        btnV1.style.fontWeight = '600';
      }
      calculate();
      render();
    }

    btnV1.addEventListener('click', function(){ setVariant('subtract'); });
    btnV2.addEventListener('click', function(){ setVariant('add'); });

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0307(){
    var root = document.getElementById('sim-ep0307-laplaciano');
    if (root) initSimEP0307(root); else setTimeout(tryInitSimEP0307, 200);
  }
  tryInitSimEP0307();
})();
</script>
</div>
""")

**Figure 3.32:** Simulator EP03_07: Laplacian Operator (w4) for Edge Enhancement


<figure id="fig-03-sim-ep0307-laplaciano">
  <img src="imagens/fig-03-sim-ep0307-laplaciano.png" alt=" Simulator EP03_07: Laplacian Operator (w4) for Edge Enhancement " style="max-width:80%" />
  <figcaption><strong>Figure 3.32:</strong>  Simulator EP03_07: Laplacian Operator (w4) for Edge Enhancement </figcaption>
</figure>

In [103]:
%%writefile EP03_07.cpp
// your solution

Overwriting EP03_07.cpp


In [104]:
TestSuite("EP03_07.cpp").run()

### 3.0.8 EP03_08 🧭 Sobel Gradient: Gx and Gy

In Mars exploration rovers (such as Perseverance), obstacle detection is performed in real time by stereoscopic cameras. The **Sobel operator** computes the directional gradient of the scene and is used in the edge detection algorithm to identify rocks, cracks, and terrain unevenness that could compromise navigation.

See [Figure 3.33](#fig-03-sim-ep0308-sobel) for a simulation of this EP.


#### 3.0.8.1 📋 Implementation Guidelines

1. **Dimensions:** Read the integers $L$ (rows) and $C$ (columns).
2. **Data:** Read the matrix $f$.
3. **Gx and Gy:** For each **internal** pixel $(i,j)$ with $1 \le i < L-1$, $1 \le j < C-1$:

$$G_x(i,j) = [f(i-1,j+1) + 2f(i,j+1) + f(i+1,j+1)] - [f(i-1,j-1) + 2f(i,j-1) + f(i+1,j-1)]$$

$$G_y(i,j) = [f(i+1,j-1) + 2f(i+1,j) + f(i+1,j+1)] - [f(i-1,j-1) + 2f(i-1,j) + f(i-1,j+1)]$$

4. **Magnitude:** $|\nabla f(i,j)| = \text{clip}(\text{round}(\sqrt{G_x^2 + G_y^2}))$.
5. **Edge:** Edge pixels receive magnitude 0.
6. **Output:** Display the $L \times C$ magnitude.

#### 3.0.8.2 📌 Computational Constraints

* **Rounding:** Apply `round` before converting to integer.
* **Saturation:** $\text{clip}(x) = \max(0, \min(255, x))$.
* **Square root:** Use $\sqrt{G_x^2 + G_y^2}$ (not the approximation $|G_x| + |G_y|$).

#### 3.0.8.3 🧠 Theoretical Foundation

| Operator | Detects | Diagonal coefficients |
|:--------:|:-------:|:----------------------:|
| $G_x$ | Vertical edges | $\pm 1$ |
| $G_y$ | Horizontal edges | $\pm 1$ |
| $|\nabla f|$ | All edges | Combined |

#### 3.0.8.4 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $L$.
* Line 2: Integer $C$.
* Following lines: Matrix elements.

**Output:**

* Gradient magnitude, $L \times C$ matrix.

#### 3.0.8.5 📌 Examples

| Input | Output | Observation |
|---------|-------|------------|
| 3<br>3<br>0 0 0<br>0 0 0<br>0 0 0 | 0 0 0<br>0 0 0<br>0 0 0 | Null image: zero gradient |
| 3<br>3<br>0 0 255<br>0 0 255<br>0 0 255 | 0 0 0<br>0 255 0<br>0 0 0 | Central vertical edge: high Gx |

In [105]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0308-sobel" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">🧭 Simulator EP03_08: Sobel Gradient (Gx and Gy)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">|∇f| = √(Gx² + Gy²)</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Analyze the horizontal (Gx) and vertical (Gy) decomposition of the Sobel operator and hover over the magnitude pixels to inspect the 3×3 neighborhood.</p>

    <!-- Barra de Controles / Kernels Explicativos -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;display:flex;align-items:center;gap:12px;flex-wrap:wrap;justify-content:space-between;">
      
      <div style="display:flex;align-items:center;gap:12px;flex-wrap:wrap;">
        <button id="sim_ep0308_btnNew" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 New Scene</button>
        <span style="font-size:11px;font-weight:700;color:#5e5a4a;">Sobel Kernels:</span>
      </div>

      <!-- Representação Visual dos Kernels Gx e Gy -->
      <div style="display:flex;align-items:center;gap:16px;flex-wrap:wrap;">
        
        <!-- Kernel Gx -->
        <div style="display:flex;align-items:center;gap:6px;">
          <div style="display:inline-grid;grid-template-columns:repeat(3, 26px);gap:2px;background:#f1ead7;border-radius:6px;padding:4px;">
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#0c447c;color:#b5d4f4;border-radius:3px;">−1</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;color:#5e5a4a;border-radius:3px;">0</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#185fa5;color:#e6f1fb;border-radius:3px;">+1</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#0c447c;color:#b5d4f4;border-radius:3px;">−2</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;color:#5e5a4a;border-radius:3px;">0</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#185fa5;color:#e6f1fb;border-radius:3px;">+2</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#0c447c;color:#b5d4f4;border-radius:3px;">−1</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;color:#5e5a4a;border-radius:3px;">0</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#185fa5;color:#e6f1fb;border-radius:3px;">+1</div>
          </div>
          <span style="font-size:11px;font-weight:700;color:#2980b9;font-family:monospace;">Gx</span>
        </div>

        <!-- Kernel Gy -->
        <div style="display:flex;align-items:center;gap:6px;">
          <div style="display:inline-grid;grid-template-columns:repeat(3, 26px);gap:2px;background:#f1ead7;border-radius:6px;padding:4px;">
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#633806;color:#fac775;border-radius:3px;">−1</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#854f0b;color:#fac775;border-radius:3px;">−2</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#633806;color:#fac775;border-radius:3px;">−1</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;color:#5e5a4a;border-radius:3px;">0</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;color:#5e5a4a;border-radius:3px;">0</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ffffff;color:#5e5a4a;border-radius:3px;">0</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ba7517;color:#faeeda;border-radius:3px;">+1</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ef9f27;color:#412402;border-radius:3px;">+2</div>
            <div style="width:26px;height:26px;display:flex;align-items:center;justify-content:center;font-size:10px;font-weight:700;background:#ba7517;color:#faeeda;border-radius:3px;">+1</div>
          </div>
          <span style="font-size:11px;font-weight:700;color:#b9770e;font-family:monospace;">Gy</span>
        </div>

      </div>

    </div>

    <!-- Comparativo em 2 Linhas (Original, Magnitude, Gx, Gy) -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Imagem Original f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Original Image f</span>
        <span style="font-size:10px;color:#8a8371;display:block;margin-bottom:10px;">5×5 pixel matrix</span>
        <div id="sim_ep0308_grid_f" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Magnitude do Gradiente |∇f| -->
      <div style="background:#fafaf7;border:2px solid #27ae60;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Magnitude |∇f|</span>
        <span style="font-size:10px;color:#27ae60;display:block;margin-bottom:10px;">√(Gx² + Gy²)</span>
        <div id="sim_ep0308_grid_m" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Gradiente Horizontal Gx -->
      <div style="background:#fafaf7;border:2px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Gx — Horizontal Gradient</span>
        <span style="font-size:10px;color:#2980b9;display:block;margin-bottom:10px;">Blue = Negative · White = Zero · Bright Blue = Positive</span>
        <div id="sim_ep0308_grid_gx" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Gradiente Vertical Gy -->
      <div style="background:#fafaf7;border:2px solid #b9770e;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#b9770e;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Gy — Vertical Gradient</span>
        <span style="font-size:10px;color:#b9770e;display:block;margin-bottom:10px;">Amber = Negative · White = Zero · Bright Amber = Positive</span>
        <div id="sim_ep0308_grid_gy" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Legenda -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 14px;display:flex;gap:16px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:12px;">
      <span style="font-size:11px;font-weight:700;color:#5e5a4a;">Legend:</span>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#fef5e7;border:1.5px dashed #b9770e;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Inspected 3×3 Neighborhood</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#ebf4fd;border:1.5px solid #2980b9;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Central Pixel</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#26241d;border:1.5px solid #8a8371;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Edge (Forced to 0)</span>
      </div>
    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0308_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:10px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;min-height:36px;line-height:1.5;">
      Hover over an inner magnitude pixel to see the Gx and Gy decomposition.
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0308(root){
    if (!root || root.dataset.simEp0308Init) return;
    root.dataset.simEp0308Init = "1";

    var ROWS = 5, COLS = 5, N = 25;
    var pixels = [], gxV = [], gyV = [], mgV = [];

    var gridF  = root.querySelector('#sim_ep0308_grid_f');
    var gridGx = root.querySelector('#sim_ep0308_grid_gx');
    var gridGy = root.querySelector('#sim_ep0308_grid_gy');
    var gridM  = root.querySelector('#sim_ep0308_grid_m');
    var debug  = root.querySelector('#sim_ep0308_debug');
    var btnNew = root.querySelector('#sim_ep0308_btnNew');

    function generate() {
      var block = Math.random() > 0.3;
      var bg = 30 + Math.floor(Math.random() * 30);
      var obj = 180 + Math.floor(Math.random() * 50);

      pixels = Array.from({ length: N }, function(_, i) {
        var r = Math.floor(i / COLS), c = i % COLS;
        var v = bg;
        if (block && r >= 2 && r <= 3 && c >= 2 && c <= 3) v = obj;
        else if (!block && r + c >= 4) v = obj - 40;
        v += Math.floor(Math.random() * 10) - 5;
        return Math.max(0, Math.min(255, v));
      });
      calculate();
    }

    function calculate() {
      gxV = new Array(N).fill(0);
      gyV = new Array(N).fill(0);
      mgV = new Array(N).fill(0);

      for (var r = 0; r < ROWS; r++) {
        for (var c = 0; c < COLS; c++) {
          var i = r * COLS + c;
          if (r >= 1 && r < ROWS - 1 && c >= 1 && c < COLS - 1) {
            var p = [
              pixels[(r - 1) * COLS + (c - 1)], pixels[(r - 1) * COLS + c], pixels[(r - 1) * COLS + (c + 1)],
              pixels[r * COLS + (c - 1)],       pixels[r * COLS + c],       pixels[r * COLS + (c + 1)],
              pixels[(r + 1) * COLS + (c - 1)], pixels[(r + 1) * COLS + c], pixels[(r + 1) * COLS + (c + 1)]
            ];
            var gx = (p[2] + 2 * p[5] + p[8]) - (p[0] + 2 * p[3] + p[6]);
            var gy = (p[6] + 2 * p[7] + p[8]) - (p[0] + 2 * p[1] + p[2]);

            gxV[i] = gx;
            gyV[i] = gy;
            mgV[i] = Math.min(255, Math.round(Math.sqrt(gx * gx + gy * gy)));
          }
        }
      }
    }

    function textColor(g){ return g > 150 ? '#000000' : '#ffffff'; }

    function colorGxBetter(v) {
      var n = Math.max(-400, Math.min(400, v));
      if (Math.abs(n) < 15) return '#fafaf7';
      if (n > 0) {
        var t = Math.min(1, n / 350);
        var r = Math.round(4 + t * 20);
        var g = Math.round(44 + t * 71);
        var b = Math.round(83 + t * 89);
        return 'rgb(' + r + ',' + g + ',' + b + ')';
      } else {
        var t = Math.min(1, -n / 350);
        return 'rgb(' + Math.round(12 + t * 0) + ',' + Math.round(68 - t * 24) + ',' + Math.round(165 - t * 82) + ')';
      }
    }

    function colorGyBetter(v) {
      var n = Math.max(-400, Math.min(400, v));
      if (Math.abs(n) < 15) return '#fafaf7';
      if (n > 0) {
        var t = Math.min(1, n / 350);
        return 'rgb(' + Math.round(186 + t * 63) + ',' + Math.round(117 + t * 70) + ',' + Math.round(23 - t * 18) + ')';
      } else {
        var t = Math.min(1, -n / 350);
        return 'rgb(' + Math.round(99 + t * 0) + ',' + Math.round(56 - t * 20) + ',' + Math.round(11 - t * 5) + ')';
      }
    }

    function textSignedColor(bg) {
      if (bg === '#fafaf7') return '#26241d';
      var m = bg.match(/rgb\((\d+),(\d+),(\d+)\)/);
      if (!m) return '#ffffff';
      var lum = 0.299 * m[1] + 0.587 * m[2] + 0.114 * m[3];
      return lum > 140 ? '#000000' : '#ffffff';
    }

    function highlight(tr, tc, on) {
      var cells = gridF.children;
      for (var i = 0; i < N; i++) {
        if (!cells[i]) continue;
        var p = pixels[i], r = Math.floor(i / COLS), c = i % COLS;
        if (on && Math.abs(r - tr) <= 1 && Math.abs(c - tc) <= 1) {
          if (r === tr && c === tc) {
            cells[i].style.background = '#ebf4fd';
            cells[i].style.color = '#2980b9';
            cells[i].style.boxShadow = '0 0 0 2px #2980b9 inset';
          } else {
            cells[i].style.background = '#fef5e7';
            cells[i].style.color = '#b9770e';
            cells[i].style.boxShadow = '0 0 0 2px #b9770e inset';
          }
        } else {
          cells[i].style.background = 'rgb(' + p + ',' + p + ',' + p + ')';
          cells[i].style.color = textColor(p);
          cells[i].style.boxShadow = 'none';
        }
      }
    }

    function resetDebug() {
      debug.textContent = 'Passe o mouse sobre um pixel interno da magnitude para ver a decomposição Gx e Gy.';
    }

    function render() {
      gridF.innerHTML = '';
      gridGx.innerHTML = '';
      gridGy.innerHTML = '';
      gridM.innerHTML = '';

      for (var i = 0; i < N; i++) {
        var r = Math.floor(i / COLS), c = i % COLS;
        var p = pixels[i], gx = gxV[i], gy = gyV[i], mag = mgV[i];
        var isBorder = (r === 0 || r === ROWS - 1 || c === 0 || c === COLS - 1);

        // Célula Original f
        var cf = document.createElement('div');
        cf.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + textColor(p) + ';box-sizing:border-box;';
        cf.textContent = p;
        gridF.appendChild(cf);

        // Célula Gx
        var cgx = document.createElement('div');
        cgx.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;';
        if (isBorder) {
          cgx.style.background = '#fafaf7';
          cgx.style.color = '#8a8371';
          cgx.style.border = '1px solid #e4dcc8';
          cgx.textContent = '—';
        } else {
          var bgGx = colorGxBetter(gx);
          cgx.style.background = bgGx;
          cgx.style.color = textSignedColor(bgGx);
          cgx.style.border = '1px solid #e4dcc8';
          cgx.textContent = (gx > 0 ? '+' : '') + gx;
        }
        gridGx.appendChild(cgx);

        // Célula Gy
        var cgy = document.createElement('div');
        cgy.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;';
        if (isBorder) {
          cgy.style.background = '#fafaf7';
          cgy.style.color = '#8a8371';
          cgy.style.border = '1px solid #e4dcc8';
          cgy.textContent = '—';
        } else {
          var bgGy = colorGyBetter(gy);
          cgy.style.background = bgGy;
          cgy.style.color = textSignedColor(bgGy);
          cgy.style.border = '1px solid #e4dcc8';
          cgy.textContent = (gy > 0 ? '+' : '') + gy;
        }
        gridGy.appendChild(cgy);

        // Célula Magnitude m
        var cm = document.createElement('div');
        cm.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;transition:all 0.15s ease;';
        if (isBorder) {
          cm.style.background = '#26241d';
          cm.style.border = '1.5px solid #8a8371';
          cm.style.color = '#7ee7c6';
          cm.textContent = '0';

          (function(row, col){
            cm.addEventListener('mouseenter', function(){
              debug.innerHTML = 'Pixel borda (' + row + ',' + col + '): sem vizinhança completa → forçado para <b>0</b>';
            });
            cm.addEventListener('mouseleave', function(){ resetDebug(); });
          })(r, c);
        } else {
          cm.style.background = 'rgb(' + mag + ',' + mag + ',' + mag + ')';
          cm.style.color = textColor(mag);
          cm.style.border = '1px solid #e4dcc8';
          cm.style.cursor = 'pointer';
          cm.textContent = mag;

          (function(row, col, gxv, gyv, mgv){
            cm.addEventListener('mouseenter', function(){
              highlight(row, col, true);
              cm.style.transform = 'scale(1.12)';
              cm.style.boxShadow = '0 0 0 2px #27ae60 inset';
              cm.style.background = '#eafaf1';
              cm.style.color = '#27ae60';

              debug.innerHTML = 'Pixel (' + row + ',' + col + '): Gx = <b>' + gxv + '</b> &nbsp;|&nbsp; Gy = <b>' + gyv + '</b> &nbsp;|&nbsp; |∇f| = round(√(' + gxv + '² + ' + gyv + '²)) = <b>' + mgv + '</b>';
            });

            cm.addEventListener('mouseleave', function(){
              highlight(row, col, false);
              cm.style.transform = 'scale(1)';
              cm.style.boxShadow = 'none';
              cm.style.background = 'rgb(' + mgv + ',' + mgv + ',' + mgv + ')';
              cm.style.color = textColor(mgv);
              resetDebug();
            });
          })(r, c, gx, gy, mag);
        }
        gridM.appendChild(cm);
      }
    }

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0308(){
    var root = document.getElementById('sim-ep0308-sobel');
    if (root) initSimEP0308(root); else setTimeout(tryInitSimEP0308, 200);
  }
  tryInitSimEP0308();
})();
</script>
</div>
""")

**Figure 3.33:** EP03_08 Simulator: Sobel Gradient (Gx and Gy)


<figure id="fig-03-sim-ep0308-sobel">
  <img src="imagens/fig-03-sim-ep0308-sobel.png" alt=" EP03_08 Simulator: Sobel Gradient (Gx and Gy) " style="max-width:80%" />
  <figcaption><strong>Figure 3.33:</strong>  EP03_08 Simulator: Sobel Gradient (Gx and Gy) </figcaption>
</figure>

In [106]:
%%writefile EP03_08.cpp
// your solution

Overwriting EP03_08.cpp


In [107]:
TestSuite("EP03_08.cpp").run()

### 3.0.9 EP03_09 📡 3×3 Median Filter

Synthetic aperture radar (SAR) images used in environmental and military monitoring suffer from a specific type of noise called *speckle*, which has characteristics similar to salt-and-pepper noise. The **median filter** is the standard method for removing this noise because it preserves the edges of structures while eliminating spurious points.

See [Figure 3.34](#fig-03-sim-ep0309-mediana) for a simulation of this exercise.

#### 3.0.9.1 📋 Implementation Guidelines

1. **Dimensions:** Read the integers $L$ (rows) and $C$ (columns).
2. **Data:** Read the pixel matrix $f$.
3. **3×3 Median Filter:** For each **internal** pixel $(i,j)$ with $1 \le i < L-1$, $1 \le j < C-1$:
   - Collect the 9 pixels from the $3 \times 3$ neighborhood: $\{f(i+s, j+t) : s,t \in \{-1,0,1\}\}$.
   - Sort the 9 values in ascending order.
   - Assign $g(i,j)$ to the central value (index position 4, considering index 0).

$$g(i,j) = \text{median}\{f(i+s, j+t) : s,t \in \{-1,0,1\}\}$$

4. **Border:** Copy directly: $g(i,j) = f(i,j)$.
5. **Output:** Display the filtered $L \times C$ matrix.

#### 3.0.9.2 📌 Computational Constraints

* **Window:** Always $3 \times 3 = 9$ elements.
* **Median:** The central element of the sorted sequence (index 4 from 0 to 8).
* **No clipping:** The median of values in $[0, 255]$ remains in $[0, 255]$.
* **Nonlinear:** The median filter cannot be expressed as a linear convolution.

#### 3.0.9.3 🧠 Theoretical Background

| Noise | Mean Filter | Median Filter |
|:-----:|:-----------:|:-------------:|
| Salt and pepper (0 or 255) | Spreads the noise | Removes without distorting edges |
| Gaussian | Reduces effectively | Reduces partially |

#### 3.0.9.4 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $L$.
* Line 2: Integer $C$.
* Following lines: Matrix elements.

**Output:**

* Filtered $L \times C$ matrix.

#### 3.0.9.5 📌 Examples

| Input | Output | Observation |
|-------|--------|-------------|
| 3<br>3<br>100 100 100<br>100 0 100<br>100 100 100 | 100 100 100<br>100 100 100<br>100 100 100 | Black point removed: median of 8×100+1×0 = 100 |
| 3<br>3<br>50 50 50<br>50 255 50<br>50 50 50 | 50 50 50<br>50 50 50<br>50 50 50 | White point (salt) removed |

In [108]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0309-mediana" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">📉 Simulator EP03_09: 3×3 Median Filter</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = Median(Neighbors)</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Inject impulsive noise (salt and pepper) and hover over internal pixels of the result to inspect the sorting of the neighborhood vector and the noise removal.</p>

    <!-- Barra de Controles / Ação -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;display:flex;align-items:center;gap:12px;flex-wrap:wrap;justify-content:space-between;">
      <button id="sim_ep0309_btnNew" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 Inject Impulsive Noise</button>
      <span style="font-size:11px;color:#8a8371;">Salt (255) and pepper (0) noise — ~30% of internal pixels affected</span>
    </div>

    <!-- Comparativo Lado a Lado: Original com Ruído vs Resultado -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- Imagem f com Ruído -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Image f — With Noise</span>
        <span style="font-size:10px;color:#8a8371;display:block;margin-bottom:10px;">Salt (255) and pepper (0) visible</span>
        <div id="sim_ep0309_grid_f" style="display:grid;grid-template-columns:repeat(5, 38px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- Resultado g sem Ruído -->
      <div style="background:#fafaf7;border:2px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">Result g — Without Noise</span>
        <span style="font-size:10px;color:#2980b9;display:block;margin-bottom:10px;">Hover to inspect</span>
        <div id="sim_ep0309_grid_result" style="display:grid;grid-template-columns:repeat(5, 38px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Vetor Ordenado -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:14px;text-align:center;">
      <span style="font-size:10px;font-weight:700;color:#5e5a4a;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:3px;">3×3 Neighborhood Vector — Sorted</span>
      <span style="font-size:10.5px;color:#8a8371;display:block;margin-bottom:8px;">Hover over an internal pixel of the result to visualize</span>
      <div id="sim_ep0309_vector" style="display:flex;flex-wrap:wrap;justify-content:center;gap:3px;min-height:32px;padding:4px 0;align-items:center;">
        <span style="font-size:11px;color:#8a8371;font-style:italic;">—</span>
      </div>
    </div>

    <!-- Legenda -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 14px;display:flex;gap:16px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:12px;">
      <span style="font-size:11px;font-weight:700;color:#5e5a4a;">Legend:</span>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#fef5e7;border:1.5px dashed #b9770e;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">3×3 Window</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#ebf4fd;border:1.5px solid #2980b9;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Central Pixel</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#eafaf1;border:1.5px solid #27ae60;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Border (Copied)</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#eafaf1;border:1.5px solid #27ae60;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Median</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#26241d;border:1.5px solid #c0392b;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Noise (Removed)</span>
      </div>
    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0309_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:10px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;min-height:36px;line-height:1.5;">
      Hover over an internal pixel of the result to see the sorting process.
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0309(root){
    if (!root || root.dataset.simEp0309Init) return;
    root.dataset.simEp0309Init = "1";

    var ROWS = 5, COLS = 5, N = 25, RADIUS = 1;
    var pixels = [], result = [];

    var gridF  = root.querySelector('#sim_ep0309_grid_f');
    var gridRes= root.querySelector('#sim_ep0309_grid_result');
    var vector = root.querySelector('#sim_ep0309_vector');
    var debug  = root.querySelector('#sim_ep0309_debug');
    var btnNew = root.querySelector('#sim_ep0309_btnNew');

    function generate() {
      pixels = Array.from({ length: N }, function(){
        var v = 110 + Math.floor(Math.random() * 30);
        var rnd = Math.random();
        if (rnd > 0.82) v = 255;
        else if (rnd < 0.18) v = 0;
        return v;
      });
      calculate();
    }

    function calculate() {
      result = pixels.slice();
      for (var r = 0; r < ROWS; r++) {
        for (var c = 0; c < COLS; c++) {
          if (r >= RADIUS && r < ROWS - RADIUS && c >= RADIUS && c < COLS - RADIUS) {
            var win = [];
            for (var s = -RADIUS; s <= RADIUS; s++) {
              for (var t = -RADIUS; t <= RADIUS; t++) {
                win.push(pixels[(r + s) * COLS + (c + t)]);
              }
            }
            win.sort(function(a, b){ return a - b; });
            result[r * COLS + c] = win[4];
          }
        }
      }
    }

    function textColor(g){ return g > 140 ? '#000000' : '#ffffff'; }
    function isNoise(v){ return v === 0 || v === 255; }

    function highlight(tr, tc, on) {
      var cells = gridF.children;
      for (var i = 0; i < N; i++) {
        if (!cells[i]) continue;
        var p = pixels[i], r = Math.floor(i / COLS), c = i % COLS;
        if (on && Math.abs(r - tr) <= 1 && Math.abs(c - tc) <= 1) {
          if (r === tr && c === tc) {
            cells[i].style.background = '#ebf4fd';
            cells[i].style.color = '#2980b9';
            cells[i].style.boxShadow = '0 0 0 2px #2980b9 inset';
          } else {
            cells[i].style.background = '#fef5e7';
            cells[i].style.color = '#b9770e';
            cells[i].style.boxShadow = '0 0 0 2px #b9770e inset';
          }
        } else {
          if (isNoise(p)) {
            cells[i].style.background = p === 255 ? '#ffffff' : '#26241d';
            cells[i].style.color = p === 255 ? '#c0392b' : '#e74c3c';
            cells[i].style.border = '2px solid #c0392b';
          } else {
            cells[i].style.background = 'rgb(' + p + ',' + p + ',' + p + ')';
            cells[i].style.color = textColor(p);
            cells[i].style.border = '1px solid #e4dcc8';
          }
          cells[i].style.boxShadow = 'none';
        }
      }
    }

    function showVector(raw, sorted, median) {
      vector.innerHTML = '';

      var rawLabel = document.createElement('span');
      rawLabel.style.cssText = 'font-size:10px;color:#8a8371;margin-right:6px;font-family:monospace;';
      rawLabel.textContent = 'Bruto:';
      vector.appendChild(rawLabel);

      raw.forEach(function(v){
        var d = document.createElement('div');
        var noise = isNoise(v);
        d.style.cssText = 'display:inline-flex;align-items:center;justify-content:center;width:28px;height:24px;border-radius:4px;font-size:10px;font-weight:700;font-family:monospace;margin:1px;';
        d.style.background = noise ? '#26241d' : '#fafaf7';
        d.style.color = noise ? '#e74c3c' : '#26241d';
        d.style.border = noise ? '1.5px solid #c0392b' : '1px solid #e4dcc8';
        d.textContent = v;
        vector.appendChild(d);
      });

      var arr = document.createElement('span');
      arr.style.cssText = 'font-size:14px;margin:0 6px;color:#8a8371;font-weight:700;';
      arr.textContent = '→';
      vector.appendChild(arr);

      var sortLabel = document.createElement('span');
      sortLabel.style.cssText = 'font-size:10px;color:#8a8371;margin-right:6px;font-family:monospace;';
      sortLabel.textContent = 'Ordenado:';
      vector.appendChild(sortLabel);

      sorted.forEach(function(v, i){
        var d = document.createElement('div');
        var isMedian = (i === 4);
        var noise = isNoise(v);
        d.style.cssText = 'display:inline-flex;align-items:center;justify-content:center;width:28px;height:24px;border-radius:4px;font-size:10px;font-weight:700;font-family:monospace;margin:1px;';
        if (isMedian) {
          d.style.background = '#eafaf1';
          d.style.color = '#27ae60';
          d.style.border = '2px solid #27ae60';
        } else if (noise) {
          d.style.background = '#26241d';
          d.style.color = '#e74c3c';
          d.style.border = '1.5px solid #c0392b';
        } else {
          d.style.background = '#fafaf7';
          d.style.color = '#26241d';
          d.style.border = '1px solid #e4dcc8';
        }
        d.textContent = v;
        vector.appendChild(d);
      });

      var eq = document.createElement('span');
      eq.style.cssText = 'font-size:11px;margin-left:8px;font-family:monospace;color:#27ae60;font-weight:700;';
      eq.innerHTML = '→ mediana = <b>' + median + '</b>';
      vector.appendChild(eq);
    }

    function resetVector() {
      vector.innerHTML = '<span style="font-size:11px;color:#8a8371;font-style:italic;">—</span>';
    }

    function resetDebug() {
      debug.textContent = 'Passe o mouse sobre um píxel interno do resultado para ver o processo de ordenação.';
    }

    function render() {
      gridF.innerHTML = '';
      gridRes.innerHTML = '';

      for (var i = 0; i < N; i++) {
        var r = Math.floor(i / COLS), c = i % COLS;
        var p = pixels[i], res = result[i];
        var isBorder = (r < RADIUS || r >= ROWS - RADIUS || c < RADIUS || c >= COLS - RADIUS);
        var noise = isNoise(p);

        // Célula F com Ruído
        var cf = document.createElement('div');
        if (noise) {
          cf.style.cssText = 'width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;user-select:none;background:' + (p === 255 ? '#ffffff' : '#26241d') + ';color:' + (p === 255 ? '#c0392b' : '#e74c3c') + ';border:2px solid #c0392b;box-sizing:border-box;';
        } else {
          cf.style.cssText = 'width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + textColor(p) + ';box-sizing:border-box;';
        }
        cf.textContent = p;
        gridF.appendChild(cf);

        // Célula Resultado g
        var cr = document.createElement('div');
        cr.style.cssText = 'width:38px;height:38px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;transition:all 0.15s ease;';

        if (isBorder) {
          cr.style.background = '#eafaf1';
          cr.style.border = '1.5px solid #27ae60';
          cr.style.color = '#27ae60';
          cr.textContent = res;

          (function(row, col, val){
            cr.addEventListener('mouseenter', function(){
              debug.innerHTML = 'Pixel borda (' + row + ',' + col + '): copiado sem filtro → <b>' + val + '</b>';
              resetVector();
            });
            cr.addEventListener('mouseleave', function(){ resetDebug(); });
          })(r, c, res);
        } else {
          cr.style.background = 'rgb(' + res + ',' + res + ',' + res + ')';
          cr.style.color = textColor(res);
          cr.style.border = '1px solid #e4dcc8';
          cr.style.cursor = 'pointer';
          cr.textContent = res;

          (function(row, col, resVal){
            cr.addEventListener('mouseenter', function(){
              highlight(row, col, true);
              cr.style.transform = 'scale(1.12)';
              cr.style.boxShadow = '0 0 0 2px #2980b9 inset';
              cr.style.background = '#ebf4fd';
              cr.style.color = '#2980b9';

              var raw = [];
              for (var s = -RADIUS; s <= RADIUS; s++) {
                for (var t = -RADIUS; t <= RADIUS; t++) {
                  raw.push(pixels[(row + s) * COLS + (col + t)]);
                }
              }
              var sorted = raw.slice().sort(function(a, b){ return a - b; });
              showVector(raw, sorted, resVal);

              var noiseCount = raw.filter(isNoise).length;
              debug.innerHTML = 'Pixel (' + row + ',' + col + '): ' + noiseCount + ' vizinho(s) com ruído na janela &nbsp;→&nbsp; mediana = posição [4] = <b>' + resVal + '</b>';
            });

            cr.addEventListener('mouseleave', function(){
              highlight(row, col, false);
              cr.style.transform = 'scale(1)';
              cr.style.boxShadow = 'none';
              cr.style.background = 'rgb(' + resVal + ',' + resVal + ',' + resVal + ')';
              cr.style.color = textColor(resVal);
              resetDebug();
              resetVector();
            });
          })(r, c, res);
        }
        gridRes.appendChild(cr);
      }
    }

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0309(){
    var root = document.getElementById('sim-ep0309-mediana');
    if (root) initSimEP0309(root); else setTimeout(tryInitSimEP0309, 200);
  }
  tryInitSimEP0309();
})();
</script>
</div>
""")

**Figure 3.34:** EP03_09 Simulator: 3×3 Median Filter


<figure id="fig-03-sim-ep0309-mediana">
  <img src="imagens/fig-03-sim-ep0309-mediana.png" alt=" EP03_09 Simulator: 3×3 Median Filter " style="max-width:80%" />
  <figcaption><strong>Figure 3.34:</strong>  EP03_09 Simulator: 3×3 Median Filter </figcaption>
</figure>

In [109]:
%%writefile EP03_09.cpp
// your solution

Overwriting EP03_09.cpp


In [110]:
TestSuite("EP03_09.cpp").run()

### 3.0.10 EP03_10 ✨ *Unsharp Masking* (USM)

In digitization systems for historical documents and works of art, image sharpness is essential for reading handwritten texts and ornamental details. ***Unsharp Masking* (USM)** is the standard sharpening enhancement algorithm used in professional scanners and software such as Adobe Photoshop, controlled by the parameter $k$ that determines the enhancement intensity.

See [Figure 3.35](#fig-03-sim-ep0310-unsharp) for a simulation of this EP.

#### 3.0.10.1 📋 Implementation Guidelines

1. **Dimensions:** Read the integers $L$ (rows) and $C$ (columns).
2. **Parameter:** Read the real value $k$ (enhancement intensity, $k \ge 0$).
3. **Data:** Read the pixel matrix $f$.
4. **Smoothing:** Compute $\bar{f}$ with a $3\times3$ averaging filter (internal pixels only; borders preserved):

$$\bar{f}(i,j) = \frac{1}{9} \sum_{s=-1}^{1} \sum_{t=-1}^{1} f(i+s, j+t)$$

5. **High-frequency mask:** $m(i,j) = f(i,j) - \bar{f}(i,j)$.
6. **USM Enhancement:** For each internal pixel:

$$g(i,j) = \text{clip}\left(\text{round}\left(f(i,j) + k \cdot m(i,j)\right)\right)$$

7. **Border:** $g(i,j) = f(i,j)$ (direct copy).
8. **Output:** Display the enhanced matrix $L \times C$.

#### 3.0.10.2 📌 Computational Constraints

* **Rounding:** Apply `round` before clipping.
* **Saturation:** $\text{clip}(x) = \max(0, \min(255, x))$.
* **Float operations:** Compute $\bar{f}$ and $m$ in floating point before rounding the final result.
* **$k = 0$:** No enhancement — the output is identical to the input (except for borders).

#### 3.0.10.3 🧠 Theoretical Foundation

| Step | Operation | Description |
|:-----:|:---------|:----------|
| 1 | $\bar{f} = f * \frac{1}{9}\mathbf{1}_{3\times3}$ | Smoothing (low frequencies) |
| 2 | $m = f - \bar{f}$ | Mask (high frequencies) |
| 3 | $g = \text{clip}(\text{round}(f + k \cdot m))$ | Weighted enhancement |

#### 3.0.10.4 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $L$.
* Line 2: Integer $C$.
* Line 3: Real $k$.
* Subsequent lines: Elements of the original matrix.

**Output:**

* Enhanced matrix $L \times C$.

#### 3.0.10.5 📌 Examples

| Input | Output | Observation |
|---------|-------|------------|
| 3<br>3<br>0.0<br>100 100 100<br>100 100 100<br>100 100 100 | 100 100 100<br>100 100 100<br>100 100 100 | k=0: no enhancement |
| 3<br>3<br>1.0<br>50 50 50<br>50 200 50<br>50 50 50 | 50 50 50<br>50 255 50<br>50 50 50 | k=1: center pixel enhanced and saturated |

In [111]:
# @title { display-mode: "form" }
from IPython.display import HTML
HTML("""
<div id="sim-ep0310-unsharp" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

  <!-- Cabeçalho -->
  <div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
    <span style="font-weight:600;color:#26241d;">✨ Simulator EP03_10: Unsharp Masking (USM)</span>
    <span style="background:#26241d;color:#7ee7c6;border-radius:40px;padding:3px 12px;font-weight:600;font-size:10px;font-family:monospace;">g = f + k · m</span>
  </div>

  <div style="padding:16px;background:#ffffff;">
    <p style="font-size:11px;color:#777;margin-bottom:12px;text-align:center;">Adjust the gain factor k, observe the complete enhancement pipeline (blurring, high-frequency mask) and hover over the result.</p>

    <!-- Barra de Controles / Slider de Ganho k -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;margin-bottom:16px;display:flex;align-items:center;gap:12px;flex-wrap:wrap;justify-content:space-between;">
      
      <div style="display:flex;align-items:center;gap:10px;flex:1;min-width:240px;">
        <span style="font-size:11px;font-weight:700;color:#26241d;">Gain factor k:</span>
        <input id="sim_ep0310_k" type="range" min="0.0" max="3.0" step="0.5" value="1.0" style="flex:1;cursor:pointer;accent-color:#2980b9;">
        <span id="sim_ep0310_klabel" style="font-family:monospace;font-size:11px;font-weight:700;background:#26241d;color:#7ee7c6;padding:3px 8px;border-radius:6px;">k = 1.0</span>
      </div>

      <button id="sim_ep0310_btnNew" style="padding:6px 12px;font-size:11px;font-weight:700;border:1px solid #26241d;background:#26241d;color:#7ee7c6;cursor:pointer;border-radius:9px;white-space:nowrap;font-family:monospace;transition:all 0.15s ease;">🎲 New Image</button>
    </div>

    <!-- Comparativo do Pipeline USM (Grid de 4 Cards) -->
    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- ① Imagem Original f -->
      <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#27ae60;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">① Original Image f</span>
        <span style="font-size:10px;color:#8a8371;display:block;margin-bottom:10px;">5×5 pixel matrix</span>
        <div id="sim_ep0310_grid_f" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- ② Desfocado f_barra -->
      <div style="background:#fafaf7;border:2px solid #b9770e;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#b9770e;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">② Blurred f̄</span>
        <span style="font-size:10px;color:#b9770e;display:block;margin-bottom:10px;">3 × 3 average</span>
        <div id="sim_ep0310_grid_b" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(220px, 1fr));gap:16px;align-items:start;margin-bottom:14px;">
      
      <!-- ③ Máscara m -->
      <div style="background:#fafaf7;border:2px solid #c0392b;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#c0392b;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;">③ Mask m</span>
        <span style="font-size:10px;color:#c0392b;display:block;margin-bottom:10px;">m = f − f̄ (High Frequencies)</span>
        <div id="sim_ep0310_grid_m" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

      <!-- ④ Resultado g -->
      <div style="background:#fafaf7;border:2px solid #2980b9;border-radius:12px;padding:12px;text-align:center;">
        <span style="font-size:10px;font-weight:700;color:#2980b9;letter-spacing:.3px;text-transform:uppercase;display:block;margin-bottom:4px;" id="sim_ep0310_flabel">④ Result g = f + 1.0·m</span>
        <span style="font-size:10px;color:#2980b9;display:block;margin-bottom:10px;">Hover to inspect</span>
        <div id="sim_ep0310_grid_res" style="display:grid;grid-template-columns:repeat(5, 36px);gap:3px;justify-content:center;user-select:none;"></div>
      </div>

    </div>

    <!-- Legenda -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;border-radius:10px;padding:10px 14px;display:flex;gap:16px;align-items:center;flex-wrap:wrap;justify-content:center;margin-bottom:12px;">
      <span style="font-size:11px;font-weight:700;color:#5e5a4a;">Legend:</span>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#fef5e7;border:1.5px dashed #b9770e;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">3×3 neighborhood</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#ebf4fd;border:1.5px solid #2980b9;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Central Pixel</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#eafaf1;border:1.5px solid #27ae60;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Border (Copied)</span>
      </div>
      <div style="display:flex;align-items:center;gap:6px;">
        <div style="width:18px;height:18px;background:#fbeaf0;border:1.5px solid #c0392b;border-radius:3px;"></div>
        <span style="font-size:10.5px;color:#5e5a4a;">Positive/Negative Mask</span>
      </div>
    </div>

    <!-- Painel Explicativo Dinâmico -->
    <div id="sim_ep0310_debug" style="background:#f1ead7;border:1px solid #e4dcc8;border-radius:8px;padding:10px 12px;text-align:center;font-size:11px;font-family:monospace;color:#26241d;min-height:36px;line-height:1.5;">
      Hover over an inner pixel of the result to trace the full pipeline.
    </div>

  </div>
</div>

<script>
(function(){
  function initSimEP0310(root){
    if (!root || root.dataset.simEp0310Init) return;
    root.dataset.simEp0310Init = "1";

    var ROWS = 5, COLS = 5, N = 25, RADIUS = 1;
    var pixels = [], blurred = [], mask = [], result = [];
    var gainK = 1.0;

    var gridF  = root.querySelector('#sim_ep0310_grid_f');
    var gridB  = root.querySelector('#sim_ep0310_grid_b');
    var gridM  = root.querySelector('#sim_ep0310_grid_m');
    var gridRes= root.querySelector('#sim_ep0310_grid_res');
    var debug  = root.querySelector('#sim_ep0310_debug');
    var fLabel = root.querySelector('#sim_ep0310_flabel');
    var sliderK= root.querySelector('#sim_ep0310_k');
    var kLabel = root.querySelector('#sim_ep0310_klabel');
    var btnNew = root.querySelector('#sim_ep0310_btnNew');

    function textColor(g){ return g > 140 ? '#000000' : '#ffffff'; }

    function colorMask(v) {
      if (Math.abs(v) < 2) return '#fafaf7';
      if (v > 0) {
        var t = Math.min(1, v / 80);
        return 'rgb(' + Math.round(230 + t * 20) + ',' + Math.round(210 - t * 60) + ',' + Math.round(220 - t * 60) + ')';
      }
      var t = Math.min(1, -v / 80);
      return 'rgb(' + Math.round(210 - t * 60) + ',' + Math.round(220 - t * 40) + ',' + Math.round(230 + t * 20) + ')';
    }

    function generate() {
      pixels = Array.from({ length: N }, function(_, i) {
        var r = Math.floor(i / COLS), c = i % COLS;
        var v = (r >= 2 && c >= 2) ? 170 : 70;
        v += Math.floor(Math.random() * 16) - 8;
        return Math.max(0, Math.min(255, v));
      });
      calculate();
    }

    function calculate() {
      blurred = new Array(N).fill(0);
      mask = new Array(N).fill(0);
      result = new Array(N).fill(0);

      for (var r = 0; r < ROWS; r++) {
        for (var c = 0; c < COLS; c++) {
          var i = r * COLS + c;
          if (r >= RADIUS && r < ROWS - RADIUS && c >= RADIUS && c < COLS - RADIUS) {
            var sum = 0;
            for (var dr = -RADIUS; dr <= RADIUS; dr++) {
              for (var dc = -RADIUS; dc <= RADIUS; dc++) {
                sum += pixels[(r + dr) * COLS + (c + dc)];
              }
            }
            var b = sum / 9;
            var m = pixels[i] - b;
            blurred[i] = b;
            mask[i] = m;
            result[i] = Math.max(0, Math.min(255, Math.round(pixels[i] + gainK * m)));
          } else {
            blurred[i] = pixels[i];
            mask[i] = 0;
            result[i] = pixels[i];
          }
        }
      }
    }

    function highlight(tr, tc, on) {
      var cells = gridF.children;
      for (var i = 0; i < N; i++) {
        if (!cells[i]) continue;
        var p = pixels[i], r = Math.floor(i / COLS), c = i % COLS;
        if (on && Math.abs(r - tr) <= 1 && Math.abs(c - tc) <= 1) {
          if (r === tr && c === tc) {
            cells[i].style.background = '#ebf4fd';
            cells[i].style.color = '#2980b9';
            cells[i].style.boxShadow = '0 0 0 2px #2980b9 inset';
          } else {
            cells[i].style.background = '#fef5e7';
            cells[i].style.color = '#b9770e';
            cells[i].style.boxShadow = '0 0 0 2px #b9770e inset';
          }
        } else {
          cells[i].style.background = 'rgb(' + p + ',' + p + ',' + p + ')';
          cells[i].style.color = textColor(p);
          cells[i].style.boxShadow = 'none';
        }
      }
    }

    function resetDebug() {
      debug.textContent = 'Passe o mouse sobre um píxel interno do resultado para rastrear o pipeline completo.';
    }

    function render() {
      gridF.innerHTML = '';
      gridB.innerHTML = '';
      gridM.innerHTML = '';
      gridRes.innerHTML = '';
      fLabel.textContent = '④ Result g = f +' + gainK.toFixed(1) + '·m';

      for (var i = 0; i < N; i++) {
        var r = Math.floor(i / COLS), c = i % COLS;
        var p = pixels[i], b = blurred[i], m = mask[i], res = result[i];
        var isBorder = (r < RADIUS || r >= ROWS - RADIUS || c < RADIUS || c >= COLS - RADIUS);

        // Célula F
        var cf = document.createElement('div');
        cf.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;border:1px solid #e4dcc8;user-select:none;background:rgb(' + p + ',' + p + ',' + p + ');color:' + textColor(p) + ';box-sizing:border-box;';
        cf.textContent = p;
        gridF.appendChild(cf);

        // Célula Desfocado B
        var cb = document.createElement('div');
        cb.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;';
        if (isBorder) {
          cb.style.background = '#fafaf7';
          cb.style.color = '#8a8371';
          cb.style.border = '1px solid #e4dcc8';
          cb.textContent = '—';
        } else {
          var bv = Math.round(b);
          cb.style.background = 'rgb(' + bv + ',' + bv + ',' + bv + ')';
          cb.style.color = textColor(bv);
          cb.style.border = '1px solid #e4dcc8';
          cb.textContent = b.toFixed(1);
        }
        gridB.appendChild(cb);

        // Célula Máscara M
        var cm = document.createElement('div');
        cm.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;';
        if (isBorder) {
          cm.style.background = '#fafaf7';
          cm.style.color = '#8a8371';
          cm.style.border = '1px solid #e4dcc8';
          cm.textContent = '0';
        } else {
          var sg = m >= 0 ? '+' : '';
          cm.style.background = colorMask(m);
          cm.style.color = '#26241d';
          cm.style.border = '1px solid #e4dcc8';
          cm.textContent = sg + m.toFixed(1);
        }
        gridM.appendChild(cm);

        // Célula Resultado g
        var cr = document.createElement('div');
        cr.style.cssText = 'width:36px;height:36px;display:flex;align-items:center;justify-content:center;font-size:9.5px;font-weight:700;font-family:monospace;border-radius:6px;box-sizing:border-box;transition:all 0.15s ease;';

        if (isBorder) {
          cr.style.background = '#eafaf1';
          cr.style.border = '1.5px solid #27ae60';
          cr.style.color = '#27ae60';
          cr.textContent = res;

          (function(row, col, val){
            cr.addEventListener('mouseenter', function(){
              debug.innerHTML = 'Pixel borda (' + row + ',' + col + '): copiado sem alteração &nbsp;→&nbsp; <b>' + val + '</b>';
            });
            cr.addEventListener('mouseleave', function(){ resetDebug(); });
          })(r, c, res);
        } else {
          cr.style.background = 'rgb(' + res + ',' + res + ',' + res + ')';
          cr.style.color = textColor(res);
          cr.style.border = '1px solid #e4dcc8';
          cr.style.cursor = 'pointer';
          cr.textContent = res;

          (function(row, col, pVal, bVal, mVal, resVal){
            cr.addEventListener('mouseenter', function(){
              highlight(row, col, true);
              cr.style.transform = 'scale(1.12)';
              cr.style.boxShadow = '0 0 0 2px #2980b9 inset';
              cr.style.background = '#ebf4fd';
              cr.style.color = '#2980b9';

              var sg = mVal >= 0 ? '+' : '';
              var raw = pVal + gainK * mVal;
              debug.innerHTML = '① f=' + pVal + ' &nbsp;→&nbsp; ② f̄=' + bVal.toFixed(1) + ' &nbsp;→&nbsp; ③ m=f−f̄=' + sg + mVal.toFixed(1) + ' &nbsp;→&nbsp; ④ g = clip(' + pVal + ' + ' + gainK.toFixed(1) + '·(' + sg + mVal.toFixed(1) + ')) = clip(' + raw.toFixed(1) + ') = <b>' + resVal + '</b>';
            });

            cr.addEventListener('mouseleave', function(){
              highlight(row, col, false);
              cr.style.transform = 'scale(1)';
              cr.style.boxShadow = 'none';
              cr.style.background = 'rgb(' + resVal + ',' + resVal + ',' + resVal + ')';
              cr.style.color = textColor(resVal);
              resetDebug();
            });
          })(r, c, p, b, m, res);
        }
        gridRes.appendChild(cr);
      }
    }

    sliderK.addEventListener('input', function(e){
      gainK = parseFloat(e.target.value) || 0;
      kLabel.textContent = 'k = ' + gainK.toFixed(1);
      calculate();
      render();
    });

    btnNew.addEventListener('click', function(){
      generate();
      render();
    });

    generate();
    render();
  }

  function tryInitSimEP0310(){
    var root = document.getElementById('sim-ep0310-unsharp');
    if (root) initSimEP0310(root); else setTimeout(tryInitSimEP0310, 200);
  }
  tryInitSimEP0310();
})();
</script>
</div>
""")

**Figure 3.35:** EP03_10 Simulator: *Unsharp Masking* (USM)


<figure id="fig-03-sim-ep0310-unsharp">
  <img src="imagens/fig-03-sim-ep0310-unsharp.png" alt=" EP03_10 Simulator: *Unsharp Masking* (USM) " style="max-width:80%" />
  <figcaption><strong>Figure 3.35:</strong>  EP03_10 Simulator: *Unsharp Masking* (USM) </figcaption>
</figure>

In [112]:
%%writefile EP03_10.cpp
// your solution

Overwriting EP03_10.cpp


In [113]:
TestSuite("EP03_10.cpp").run()

## Chapter References


BRADSKI, Gary; KAEHLER, Adrian. **Learning OpenCV: Computer vision with the OpenCV library**. " O'Reilly Media, Inc.", 2008.

GONZALEZ, R. C.; WOODS, R. E. **Digital Image Processing**. New York, Pearson, 2018.

SZELISKI, Richard. **Computer Vision: Algorithms and Applications**. Springer, 2022.